In [ ]:

######################################################### Endothelial产图的code   ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
import scvelo as scv
import cellrank as cr
from anndata import AnnData
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors
import stream as st


In [ ]:
Fibro_meta_check = pd.read_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta_check_updated.csv',sep=',')
Fibro_meta_check


In [ ]:
# adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')
adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation.h5ad')
adata.shape


In [ ]:
adata.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation.h5ad')


In [ ]:
adata.obs['Age_group'].value_counts()


In [ ]:
(adata.obs.columns)


In [ ]:
sc.pl.umap(adata, color=['tissue','leiden','Celltype_general','Celltype','Celltype_sub','Age_group'],  show=True, ncols =3, wspace=0.5,hspace=0.2 )  # wspace=0.3,legend_loc ='on data'



In [ ]:
[0.2,0.5,1,1.5,2,5]

In [ ]:
# len(adata.obs['Celltype'].unique())
Nat_color_list = ['#E64B35',  # Nature红
        '#4DBBD5',  # Nature蓝
        '#00A087',  # Nature绿
        '#3C5488',  # Nature深蓝
        '#F39B7F',  # 珊瑚色
        '#8491B4',  # 淡紫色
        '#91D1C2',  # 薄荷绿
        '#DC0000',  # 正红
        '#7E6148',  # 棕色
        '#B09C85',  # 米色
        '#F4649D',  # 粉红
        '#7C6A9C',  # 紫罗兰
        '#D9A428',  # 金黄色
        '#59A14F',  # 翠绿
        '#AF7AC4',  # 浅紫
        '#FFBB78',  # 杏色
        '#98DF8A',  # 嫩绿
        '#F7B6D2',  # 浅粉
        '#C5B0D5',  # 淡紫
        '#FF9896',  # 浅红
        '#9467BD',  # 深紫
        '#8C564B',  # 赭石
        '#E377C2',  # 玫红
        '#7F7F7F',  # 中灰
        '#BCBD22',  # 黄绿
        '#17BECF',  # 青绿
        '#AEC7E8',  # 天蓝
        '#C49C94',  # 肉色
        '#DBDB8D',  # 淡黄
        '#9EDAE5',  # 浅青
        '#F5B3A5',  # 浅珊瑚
        '#B3B3B3',  # 浅灰
        '#C2B0B0',  # 灰粉
        '#8C8C8C',  # 深灰
        '#6B8E23',  # 橄榄绿
        '#FF8C00',  # 暗橙
        '#4682B4'   # 钢蓝
    ]


In [ ]:
# sorted(dir())


In [ ]:
adata


In [ ]:
len(adata.obs['Celltype'].unique())


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general']].drop_duplicates().sort_values('Celltype_general')


In [ ]:
adata.obs[['leiden','Celltype','Celltype_C','Celltype_general']].drop_duplicates().sort_values('leiden') 
adata.obs['Celltype_C'] = adata.obs['Celltype_C'].astype('category')


In [ ]:
adata.obs['Celltype_C'].unique()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd


# 1. 基础检查
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要列: {col}")

# 确保 leiden 是字符串（Scanpy 推荐）
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)

# 根据adata.obs['leiden'] 产生 leiden color list, 参考Nature系列的配色
Nat_color_list = ['#E64B35',  # Nature红
        '#4DBBD5',  # Nature蓝
        '#00A087',  # Nature绿
        '#3C5488',  # Nature深蓝
        '#F39B7F',  # 珊瑚色
        '#8491B4',  # 淡紫色
        '#91D1C2',  # 薄荷绿
        '#DC0000',  # 正红
        '#7E6148',  # 棕色
        '#B09C85',  # 米色
        '#F4649D',  # 粉红
        '#7C6A9C',  # 紫罗兰
        '#D9A428',  # 金黄色
        '#59A14F',  # 翠绿
        '#AF7AC4',  # 浅紫
        '#FFBB78',  # 杏色
        '#98DF8A',  # 嫩绿
        '#F7B6D2',  # 浅粉
        '#C5B0D5',  # 淡紫
        '#FF9896',  # 浅红
        '#9467BD',  # 深紫
        '#8C564B',  # 赭石
        '#E377C2',  # 玫红
        '#7F7F7F',  # 中灰
        '#BCBD22',  # 黄绿
        '#17BECF',  # 青绿
        '#AEC7E8',  # 天蓝
        '#C49C94',  # 肉色
        '#DBDB8D',  # 淡黄
        '#9EDAE5',  # 浅青
        '#F5B3A5',  # 浅珊瑚
        '#B3B3B3',  # 浅灰
        '#C2B0B0',  # 灰粉
        '#8C8C8C',  # 深灰
        '#6B8E23',  # 橄榄绿
        '#FF8C00',  # 暗橙
        '#4682B4'   # 钢蓝
    ]

# ================================
# 2. 构建 Leiden → 颜色映射
# ================================
leiden_categories = sorted(
    adata.obs['leiden'].unique(),
    key=lambda x: int(x)
)
if len(leiden_categories) != len(Nat_color_list):
    raise ValueError("Nat_color_list 数量必须等于 Leiden cluster 数量")

leiden_color_dict = dict(zip(leiden_categories, Nat_color_list))
# 告诉 scanpy 使用这套颜色
adata.uns['leiden_colors'] = [
    leiden_color_dict[c] for c in leiden_categories
]

# ================================
# 3. 构建 Celltype_C 列
# ================================

adata.obs['Celltype_C'] = (
    adata.obs['Celltype'] +
    "_C" +
    adata.obs['leiden']
)

# ================================
# 4. 构建 Celltype_C → 颜色映射（来自 Leiden）
# ================================
celltypeC_color_dict = {}
for leiden_id in leiden_categories:
    mask = adata.obs['leiden'] == leiden_id
    celltypeC_values = adata.obs.loc[mask, 'Celltype_C'].unique()
    
    for val in celltypeC_values:
        celltypeC_color_dict[val] = leiden_color_dict[leiden_id]

# ================================
# 5. 开始绘图
# ================================
plt.rcParams['figure.figsize'] = (13, 6)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)

ax.set_title('Fibroblast Atlas', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
# ================================

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

# ----------------------
# 5. 添加带箭头的轴标签
# ----------------------
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%

# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='left',  # 文本对齐方式
    va='center'
)

# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# ================================
# 8. 构建分组 legend（按 Celltype_general）
# ================================

general_groups = ['qFibro', 'Myfibro', 'iFibro', 'eFibro']
col1_generals = ['qFibro', 'Myfibro', 'iFibro']
col2_generals = ['eFibro']
def build_legend_handles(generals):
    handles = []
    labels = []
    
    for g in generals:
        subset = adata.obs[adata.obs['Celltype_general'] == g]
        unique_celltypeC = sorted(subset['Celltype_C'].unique())
        # 添加 general 标题（黑色）
        handles.append(mpatches.Patch(color='white', label=g))
        labels.append(g)
        # 添加每个 Celltype_C
        for ct in unique_celltypeC:
            handles.append(
                mpatches.Patch(
                    color=celltypeC_color_dict[ct],
                    label=ct
                )
            )
            labels.append("  " + ct)  
    return handles, labels


# 左 legend
handles1, labels1 = build_legend_handles(col1_generals)

legend1 = ax.legend(
    handles1,
    labels1,
    bbox_to_anchor=(1.02, 0.95),
    loc='upper left',
    frameon=False,
    fontsize=9
)

ax.add_artist(legend1)

# 右 legend
handles2, labels2 = build_legend_handles(col2_generals)

legend2 = ax.legend(
    handles2,
    labels2,
    bbox_to_anchor=(1.32, 0.95),
    loc='upper left',
    frameon=False,
    fontsize=9
)


# ----------------------
# 8. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height
        }
    return bounds

# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 9. 为每个Celltype_general添加弯曲虚线圆圈和文本标签
general_colors = {
    'qFibro': '#ff7f0e',    # 橙色
    'Myfibro': '#d62728',   # 红色
    'iFibro': '#2ca02c' ,   # 绿色
    'eFibro': '#1f77b4'    # 蓝色
}

# ----------------------
for general_type, color in general_colors.items():
    if general_type not in cluster_bounds:
        continue
    # 获取该分组的边界信息
    bounds = cluster_bounds[general_type]
    center_x, center_y = bounds['center']
    width = bounds['width']
    height = bounds['height']
    # 创建弯曲的椭圆虚线框（模拟弯曲圈）
    ellipse = mpatches.Ellipse(
        (center_x, center_y),
        width=width,
        height=height,
        edgecolor=color,
        facecolor='none',
        linestyle='--',  # 虚线
        linewidth=2,     # 线宽
        alpha=0.8        # 透明度
    )
    ax.add_patch(ellipse)
    # 添加文本标签（在圈的右侧/上方，避免遮挡）
    # 计算标签位置（圈的右侧偏移一点）
    label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
    label_y = center_y
    
    # 添加文本
    ax.text(
        label_x, label_y,
        general_type,
        fontsize=12,
        fontweight='bold',
        color=color,
        ha='left',
        va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
    )


# ================================
# 10. 保存
# ================================

plt.tight_layout()
fig_dir='./TabulaStromal_figures/Fibroblasts/'
plt.savefig(
    fig_dir+"Fibroblast_umap_leiden_Celltype.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
plt.show()


In [ ]:
adata.obs['leiden'].unique()


In [ ]:
tissue_list = list(adata.obs['tissue'].unique())
del_tissue = ['embryo','heart','peripheral nerve','soft tissue','prostate','none','bladder','tonsil','airway','organoid','carcinoma']
include_tissue = [item for item in tissue_list if item not in set(del_tissue)]
adata = adata[adata.obs['tissue'].isin(include_tissue)]


In [ ]:
import pandas as pd

# 先获取tissue的计数Series
tissue_counts = adata.obs['tissue'].value_counts()

# 初始化空列表
tissue_count_list = []

# 遍历每个tissue和对应的细胞数
for tissue, count in tissue_counts.items():
    # 转换为千为单位，保留一位小数
    count_k = count / 1000
    count_formatted = f"{count_k:.1f}"
    # 拼接字符串
    tissue_str = f"{tissue}: {count_formatted}k"
    # 添加到列表
    tissue_count_list.append(tissue_str)

# 查看结果
print(tissue_count_list)


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general']].drop_duplicates().sort_values('Celltype_general')

In [ ]:
# 定义要展示的基因集
genesets = ['ECM','Immune','AP', 'leiden']
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=['ECM','Immune','AP', 'leiden'],
                  frameon=False,
                  ncols=2,
                  size=3,
                  vmin=global_min,  # 统一最小值(6Lineage那里算了)
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)
for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'
    
plt.gcf().set_size_inches(11, 9)
fig = plt.gcf()
plt.tight_layout()
fig_path = './TabulaStromal_figures/Fibroblasts/umap/'
plt.savefig(
    fig_path+"Fib_all_umap.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
plt.savefig(fig_path+"Fib_all_umap.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 定义要展示的基因集
genesets = ['T','B','Fibro', 'Endo','MoMac', 'Epi']
global_min = min([adata.obs[gs].min() for gs in genesets if gs in adata.obs])
global_max = max([adata.obs[gs].max() for gs in genesets if gs in adata.obs])
print(f"统一颜色范围: [{global_min:.3f}, {global_max:.3f}]")
# 绘制UMAP图，使用统一的vmin和vmax
axes = sc.pl.umap(adata, 
                  color=genesets,
                  color_map='viridis',
                  title=['T','B','Fibro', 'Endo','MoMac', 'Epi'],
                  frameon=False,
                  ncols=3,
                  size=3,
                  vmin=global_min,  # 统一最小值
                  vmax=global_max,  # 统一最大值
                  show=False,
                  legend_fontsize=18,
                  legend_fontoutline=2,
                  wspace=0.1)

for i, ax in enumerate(axes):
    ax.set_title(f'{ax.get_title()}', fontsize=20)  #   fontweight='bold'
    
# 调整图形大小
plt.gcf().set_size_inches(18, 9)
fig = plt.gcf()
# 创建一个用于colorbar的mappable对象
import matplotlib.cm as cm
from matplotlib.colors import Normalize

norm = Normalize(vmin=global_min, vmax=global_max)
sm = cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
plt.tight_layout()
fig_path = './TabulaStromal_figures/Fibroblasts/umap/'
plt.savefig(
    fig_path+"6Lineage_umap.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
plt.savefig(fig_path+"6Lineage_umap.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# sc.pl.umap(adata, color=['T','B','Epi','Endo',"Fibro",'MoMac','age_group','leiden2','tissue'],  show=True, ncols =3, wspace=0.2,hspace=0.2 )  # wspace=0.3,legend_loc ='on data'


In [ ]:
############## 做downsampling，用来做 stream 的结果

In [ ]:
np.random.seed(42)

# 方法1: 使用 groupby + sample
def sample_cells(group):
    n_sample = min(20, len(group))
    return group.sample(n=n_sample, random_state=42)

# 获取抽样后的索引
sampled_indices = adata.obs.groupby('Celltype_general', group_keys=False).apply(sample_cells).index


In [ ]:
adata_downsample = adata[sampled_indices, :]
adata_downsample.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_ds_80_ctgeneral.h5ad')


In [ ]:
leiden = list(sorted(adata_sub.obs['leiden'].unique()))
celltype = list(sorted(adata_sub.obs['Celltype_general'].unique()))
import random
DefaulfColorPalette = list(set([
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628",
    "#0096FF", "#F3D4F4", "#FDCDAC", "#548235", "#9271CB", "#917F1D",
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#BCBD22",
    "#B279A2", "#EECA3B", "#17BECF", "#FF9DA6", "#778AAE", "#1B9E77",
    "#A6761D", "#526A83", "#B82E2E", "#80B1D3", "#68855C", "#D95F02",
    "#BEBADA", "#AF6458", "#D9AF6B", "#9C9C5E", "#625377", "#8C785D",
    "#88CCEE", "#E73F74", "#FFFFB3", "#CCEBC5", "#332288", "#A65628",
    "#0096FF", "#F3D4F4", "#FDCDAC", "#548235", "#9271CB", "#917F1D"
]))

df2 = pd.DataFrame({
    'Celltype': celltype,
    'color': random.sample(DefaulfColorPalette, len(celltype))
})


In [ ]:
df2.to_csv('./data/scalex/fibro/human/stream_result/celltype_general_color.tsv',sep='\t',index=False,header=False)
cell = pd.DataFrame({'Cell':adata_sub.obs['Celltype_general'].to_list()})
cell.to_csv('./data/scalex/fibro/human/stream_result/celltype_general_ds.tsv',index=False, sep='\t',header=False)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def create_simple_subset(adata, min_sample_cells=50, cells_per_celltype=2000):
    """
    简化的平衡子集创建函数
    """
    # 1. 筛选合格样本
    sample_counts = adata.obs['sample'].value_counts()
    valid_samples = sample_counts[sample_counts >= min_sample_cells].index.tolist()
    adata_filtered = adata[adata.obs['sample'].isin(valid_samples)].copy()
    print(f"筛选后细胞数: {adata_filtered.n_obs}")
    print(f"合格样本数: {len(valid_samples)}")
    # 2. 为每个Celltype抽样
    selected_indices = []
    for celltype in adata_filtered.obs['Celltype'].unique():
        # 获取该Celltype的所有细胞
        celltype_mask = adata_filtered.obs['Celltype'] == celltype
        celltype_cells = adata_filtered.obs[celltype_mask]
        # 如果细胞数不足，使用所有细胞
        if len(celltype_cells) <= cells_per_celltype:
            selected = celltype_cells.index.tolist()
        else:
            # 分层抽样：按样本比例
            celltype_by_sample = celltype_cells.groupby('sample').size()
            sample_weights = celltype_by_sample / celltype_by_sample.sum()
            sample_targets = (sample_weights * cells_per_celltype).round().astype(int)
            
            # 调整总数
            total = sample_targets.sum()
            if total != cells_per_celltype:
                # 简单调整：从最大的样本增减
                largest_sample = celltype_by_sample.idxmax()
                sample_targets[largest_sample] += cells_per_celltype - total
            
            # 从每个样本抽样
            celltype_selected = []
            for sample, n in sample_targets.items():
                if n > 0:
                    sample_cells = celltype_cells[celltype_cells['sample'] == sample]
                    if len(sample_cells) <= n:
                        celltype_selected.extend(sample_cells.index.tolist())
                    else:
                        celltype_selected.extend(
                            np.random.choice(sample_cells.index.tolist(), n, replace=False)
                        )
            
            # 确保总数正确
            if len(celltype_selected) != cells_per_celltype:
                if len(celltype_selected) < cells_per_celltype:
                    # 补充
                    remaining = list(set(celltype_cells.index) - set(celltype_selected))
                    additional = np.random.choice(
                        remaining, 
                        cells_per_celltype - len(celltype_selected), 
                        replace=False
                    )
                    celltype_selected.extend(additional)
                else:
                    # 随机减少
                    celltype_selected = np.random.choice(
                        celltype_selected, 
                        cells_per_celltype, 
                        replace=False
                    ).tolist()
            
            selected = celltype_selected
        
        selected_indices.extend(selected)
        print(f"{celltype}: 从 {len(celltype_cells)} 个细胞中选择了 {len(selected)} 个")
    
    # 3. 创建子集
    adata_sub = adata_filtered[selected_indices].copy()
    
    print(f"\n最终子集:")
    print(f"  细胞数: {adata_sub.n_obs}")
    print(f"  样本数: {adata_sub.obs['sample'].nunique()}")
    print(f"  Celltype数: {adata_sub.obs['Celltype'].nunique()}")    
    return adata_sub

# 使用简化版本
adata_sub_simple = create_simple_subset(adata, min_sample_cells=50, cells_per_celltype=2000)

# 保存
adata_sub_simple.write('adata_fibroblast_subset.h5ad', compression='gzip')
print("子集已保存为 'adata_fibroblast_subset.h5ad'")


In [ ]:
def isstr(x):
    try: 
        float(x)
        return False
    except ValueError as e:
        return True
        

In [ ]:
adata.obs['Celltype_general'].value_counts()



In [ ]:
adata  = adata[~np.array(list(map(lambda x: isstr(x),adata.obs['age'])))]
adata


In [ ]:
adata.obs['age'] = adata.obs['age'].astype(float) 
def assign_age_group(age):
    # age = str(age).lower()
    if 'Young adult' in age:
        return 'B'
    elif 'nfant' in age:
        return 'A'
    elif 'E' in age:
        return 'A'
    elif age == 'Aged':
        return 'H'
    elif 'month' in age:
        return 'A'
    try:
        age = float(age)  # 转换为浮动类型
        if 0<=age<20:
            return 'A'
        elif 20 <= age < 30:
            return 'B'
        elif 30 <= age < 40:
            return 'C'
        elif 40 <= age < 50:
            return 'D'
        elif 50 <= age < 60:
            return 'E'
        elif 60 <= age < 70:
            return 'F'
        elif 70 <= age < 90:
            return 'G'
        elif age >= 90:
            return 'H'
    except ValueError:
        return 'Unknown'  # 对age 信息重新进行分群 



In [ ]:
adata.obs['Age_group']  = adata.obs['age'].apply(assign_age_group)
adata.obs['Age'] = adata.obs['age']


In [ ]:
sc.pl.umap(adata, color=['tissue','leiden','Celltype_general','Celltype_sub','Celltype'],  show=True, ncols =2, wspace=0.5,hspace=0.2 )  # wspace=0.3,legend_loc ='on data'
# import matplotlib.pyplot as plt


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
expanded_iFibro_markers = {
    'iFibro_extended': [
        'CXCL12', 'CXCL14', 'CCL2', 'CCL11', 'CCL19',  # 趋化因子
        'IL6', 'IL33', 'LIF',                         # 白细胞介素
        'ICAM1', 'VCAM1',                             # 粘附分子
        'PDCD1LG2', 'CD274',                          # 免疫检查点配体 (PD-L1/2)
        'TNFSF13B', 'TNFSF14',                        # TNF超家族
        'MMP3', 'MMP10',                              # 基质金属蛋白酶
        'PTGS2'                                       # 炎症介质 (COX-2)
    ]
}

sc.pl.umap(adata, color=sorted(list(set(adata.var_names) & set(expanded_iFibro_markers['iFibro_extended']))), 
          ncols=4, frameon=False, show=True)


In [ ]:
# adata.obs['Celltype_general'].value_counts()


In [ ]:
adata.obs['Celltype_general'].value_counts()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib as mpl

# 设置Nature/Science级别的出版级绘图参数
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'font.family': 'Arial',  # 学术期刊常用字体
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'figure.autolayout': False,
})

# 需要可视化的5列
score_columns = ['Myfibro_raw', 'apFibro_raw', 'eFibro_raw', 'iFibro_raw', 'qFibro_raw']
score_names = {
    'Myfibro_raw': 'Myfibro',
    'apFibro_raw': 'apFibro', 
    'eFibro_raw': 'eFibro',
    'iFibro_raw': 'iFibro',
    'qFibro_raw': 'qFibro'
}

# 检查这些列是否存在
missing_cols = [col for col in score_columns if col not in adata.obs.columns]
if missing_cols:
    print(f"Warning: Missing columns: {missing_cols}")
    score_columns = [col for col in score_columns if col in adata.obs.columns]

# 获取所有Leiden并排序
all_leiden = sorted(adata.obs['leiden'].unique().astype(str))
leiden_count = len(all_leiden)
print(f"Total {leiden_count} Leiden clusters identified")

# 将Leiden分成4组（可根据需要调整）
n_groups = 4
leiden_per_group = (leiden_count + n_groups - 1) // n_groups
leiden_groups = [all_leiden[i:i+leiden_per_group] for i in range(0, leiden_count, leiden_per_group)]

# Science/Nature级别的高对比度配色方案
# 使用cet_CET_C5色系（科学可视化的高质量配色）
colors = [
    '#1f77b4',  # 蓝色 - Myofibro
    '#ff7f0e',  # 橙色 - apFibro  
    '#2ca02c',  # 绿色 - eFibro
    '#d62728',  # 红色 - iFibro
    '#9467bd',  # 紫色 - qFibro
]

# 或者使用更科学的配色（Viridis变种）
scientific_colors = [
    '#440154',  # 深紫色
    '#3b528b',  # 深蓝色
    '#21918c',  # 青绿色
    '#5ec962',  # 绿色
    '#fde725',  # 黄色
]

colors = scientific_colors[:len(score_columns)]
colors_dict = {col: colors[i] for i, col in enumerate(score_columns)}

# 创建子图，调整宽高比
fig, axes = plt.subplots(n_groups, 1, figsize=(15, 4.5*n_groups), 
                        gridspec_kw={'hspace': 0.25, 'wspace': 0.05})

# 如果只有一个子图，将axes转换为列表
if n_groups == 1:
    axes = [axes]

# 对每组Leiden创建小提琴图
for group_idx, (ax, leiden_group) in enumerate(zip(axes, leiden_groups)):
    # 筛选出这组Leiden的细胞
    mask = adata.obs['leiden'].isin(leiden_group)
    subset_data = adata[mask].obs.copy()
    # 为了画图，将宽表转换为长表
    plot_data_list = []
    for leiden in leiden_group:
        leiden_mask = subset_data['leiden'] == leiden
        if leiden_mask.sum() == 0:
            continue  
        for score_col in score_columns:
            scores = subset_data.loc[leiden_mask, score_col].values
            for score_val in scores:
                plot_data_list.append({
                    'Leiden': leiden,
                    'Score_Type': score_col,
                    'Score_Value': score_val
                })
    if plot_data_list:
        plot_df = pd.DataFrame(plot_data_list)
        # 设置x轴位置 - 确保正确的小提琴图宽度
        n_leiden = len(leiden_group)
        positions = np.arange(n_leiden)
        # 为每个Score类型绘制小提琴图
        width = 0.7 / len(score_columns)  # 调整宽度使小提琴更粗
        for i, score_col in enumerate(score_columns):
            offset = (i - len(score_columns)/2 + 0.5) * width
            score_positions = positions + offset
            # 提取当前score的数据
            score_data = []
            for leiden in leiden_group:
                leiden_scores = plot_df[(plot_df['Leiden'] == leiden) & 
                                       (plot_df['Score_Type'] == score_col)]['Score_Value'].values
                if len(leiden_scores) > 0:
                    score_data.append(leiden_scores)
                else:
                    score_data.append(np.array([]))
            # 绘制更粗的小提琴图
            violin_parts = ax.violinplot(score_data, positions=score_positions, 
                                        widths=width*0.8,  # 控制小提琴宽度
                                        showmeans=False, 
                                        showmedians=True,
                                        showextrema=False)
            # 自定义小提琴外观
            violin_parts['bodies'][0].set_facecolor(colors_dict[score_col])
            violin_parts['bodies'][0].set_alpha(0.8)
            violin_parts['bodies'][0].set_edgecolor('black')
            violin_parts['bodies'][0].set_linewidth(0.5)
            # 添加中位数线
            violin_parts['cmedians'].set_color('white')
            violin_parts['cmedians'].set_linewidth(1.5)
        # 设置x轴标签（带细胞数量）
        cell_counts = subset_data['leiden'].value_counts().loc[leiden_group]
        tick_labels = [f'{leiden}\n(n={cell_counts.get(leiden, 0)})' for leiden in leiden_group]
        ax.set_xticks(positions)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)
        # 设置y轴
        ax.set_ylabel('Signature Score', fontsize=11, fontweight='bold', labelpad=10)
        ax.set_xlabel('Leiden Cluster', fontsize=11, fontweight='bold', labelpad=10)
        # 添加标题
        # ax.set_title(f'Fibroblast Subtype Signature Scores: Leiden {leiden_group[0]} - {leiden_group[-1]}', 
        #             fontsize=12, fontweight='bold', pad=15)
        # 添加网格线
        ax.yaxis.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
        ax.xaxis.grid(False)
        ax.set_facecolor('#f8f9fa')
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color('#495057')
    else:
        ax.text(0.5, 0.5, f'No data for Leiden {leiden_group[0]} to {leiden_group[-1]}', 
               ha='center', va='center', transform=ax.transAxes, fontsize=10, 
               style='italic', color='gray')
        ax.set_axis_off()


from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors_dict[col], 
                        edgecolor='black',
                        linewidth=0.5,
                        alpha=0.8,
                        label=score_names.get(col, col.replace('_raw', ''))) 
                  for col in score_columns]
fig.legend(handles=legend_elements,
          title='Fibroblast Subtypes',
          title_fontsize=11,
          fontsize=10,
          loc='upper right',
          bbox_to_anchor=(0.98, 0.98),
          frameon=True,
          fancybox=True,
          shadow=False,
          framealpha=0.9,
          edgecolor='#495057',
          borderpad=1)
fig.suptitle('Distribution of Fibroblast Subtype Signature Scores Across Leiden Clusters', 
            fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # 为标题留出空间
plt.savefig('fibroblast_subtype_scores.png', 
           dpi=300, 
           bbox_inches='tight',
           facecolor='white',
           edgecolor='none',
           pad_inches=0.1)
plt.show()


In [ ]:
adata.obs[['leiden','Celltype','Celltype_general']].drop_duplicates().sort_values('Celltype_general')



In [ ]:
sc.pl.umap(adata, color=[ 'Celltype'], show=True, ncols =2, wspace=0.5 ,title ='BBKNN',legend_loc=None,frameon=False, hspace=0.2)  # wspace=0.3,legend_loc ='on data'


In [ ]:
plt.rcdefaults()
sc.pl.umap(adata, color=['tissue','leiden','Celltype_general', 'Celltype_sub', 'Celltype'], show=True, ncols =2, wspace=0.5 , hspace=0.2)  # wspace=0.3,legend_loc ='on data'



In [ ]:
###### 
import json
json_file = './data/scalex/fibro/human/small_file/subFibro_cluster_annotation.json'
with open(json_file, 'r', encoding='utf-8') as file:
    cluster_annotations = json.load(file)
    
all_signature_score_df = pd.read_csv('./data/scalex/fibro/human/small_file/subFibro_cluster_annotation_Confidence.tsv',sep='\t')


In [ ]:
potential_ifibro


In [ ]:
myofibroblast_specific_markers = [
    'ACTA2',    # α-平滑肌肌动蛋白 - 最经典特异的myofibroblast标记
    'TAGLN',    # 转胶蛋白 - 平滑肌分化标记
    'MYH11',    # 肌球蛋白重链11 - 收缩功能核心蛋白
    'MYL9',     # 肌球蛋白轻链9 - 收缩装置成分
    'TPM2',     # 原肌球蛋白2 - 肌动蛋白结合蛋白
    'CNN1',     # 钙调理蛋白1 - 平滑肌特异性
    'LMOD1',    # 平滑肌动蛋白调节蛋白1
    'ACTG2',    # γ-平滑肌肌动蛋白
    'MYLK',     # 肌球蛋白轻链激酶
    'NEXN',     # 神经丝蛋白F-肌动蛋白结合蛋白
    'MCAM',     # 黑色素瘤细胞粘附分子
    'LRRC15',   # 富含亮氨酸重复序列蛋白15 - 肌成纤维细胞新标记
    'ADAM12'    # 去整合素样金属蛋白酶12
]



quiescent_fibroblast_markers = [
    'HIC1',       # 静息态成纤维细胞标记
    'Gli1',       # 静息态标记
    'PDGFRα',     # 泛成纤维细胞标记，在静息态中表达
    'COL1A1',     # I型胶原蛋白，成纤维细胞基质成分
    'DPT',        # 泛成纤维细胞标记
    'FBLN1',      # Fibulin-1，与细胞静息状态相关
    'FBLN2',      # Fibulin-2，细胞外基质蛋白
    'APOE',       # 载脂蛋白E，在静息态成纤维细胞中表达
    'CFD',        # 补体因子D（adipsin）
    'CLU'         # 簇集蛋白，与细胞稳态相关
]

In [ ]:
sc.pl.umap(
    adata,
    color='Celltype',
    frameon=False,      # 去掉边框
    show=False, # 不立即显示，以便自定义
    # legend_loc='on data'
)

plt.savefig('./TabulaStromal_figures/Fibroblasts/Fib_celltype_umap.pdf', bbox_inches='tight', transparent=True)


In [ ]:
myo_marker = ['ACTA2','LMOD1','MYL9','TPM2','MCAM','MYL11','TAGLN','PTN','COL6A2']


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 设置图像参数
# sc.settings.set_figure_params(dpi=80, facecolor='white', figsize=(10, 8))


# 或者同时显示多个基因（建议4-6个一组）
sc.pl.umap(adata, color=sorted(list(set(adata.var_names) & set(['VIM', 'PDGFRB']))),  # myo_marker
          ncols=4, frameon=False, show=True)


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt


sc.pl.umap(adata, color=sorted(list(set(adata.var_names) & set(fibroblast_markers['Myfibro']+['PTN']))), 
          ncols=4, frameon=False, show=True)


In [ ]:
# sorted(dir())
adata.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')


In [ ]:
adata.obs[['Age','Age_group','disease_global','disease_global_check']].drop_duplicates()


In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()


In [ ]:
# adata.obs['age'].value_counts()


In [ ]:
adata.obs['Age_group'].value_counts()


In [ ]:
adata_sub = adata[adata.obs['Age_group'].isin(['B', 'D', 'H', 'C', 'G', 'F', 'E'])]
adata_sub


In [ ]:
adata_sub.obs['Age_group'].value_counts()


In [ ]:
# sorted(dir())


In [ ]:
adata_sub.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delA_Unknown.h5ad')




In [ ]:
adata.obs['Celltype_sub'].value_counts()


In [ ]:
adata_sub.obs['Celltype_general'].value_counts()


In [ ]:
adata.obs[['age','Age_group']].drop_duplicates()


In [ ]:
Fibro_meta_check


In [ ]:
meta_data = adata.obs


In [ ]:

meta_data.to_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_meta.tsv',sep='\t',index=False )


In [ ]:
import pandas as pd

# 从adata.obs中提取关键列用于合并
obs_key_columns = ['GSE', 'GSM']

# 从Endo_meta_check中提取需要合并的列
check_columns = [ 'tissue_check', 'disease_global_check', 'sex_check']

# 基于GSE和GSM列进行左连接合并
merged_obs = adata.obs.merge(
    Fibro_meta_check[['GSE', 'GSM'] + check_columns],
    on=['GSE', 'GSM'],
    how='left'
)


merged_obs = merged_obs.drop_duplicates()
merged_obs.index = adata.obs.index




In [ ]:
merged_obs

In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()



In [ ]:
adata.obs.columns


In [ ]:
merged_obs.shape, adata.shape


In [ ]:
adata.obs.columns


In [ ]:
# help(sc.pl.umap)



In [ ]:
adata


In [ ]:
adata.obs['

In [ ]:
# adata.obs.columns


In [ ]:
merged_obs[['GSE','GSM']].drop_duplicates()


In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()   # 'GSE','GSM',


In [ ]:
intersection_sorted = sorted(list(set(sorted(list(adata.obs['GSM'].unique()))) & set(Fibro_meta_check['GSM'].unique())))
len(intersection_sorted),len(Fibro_meta_check['GSM'].unique()),len(adata.obs['GSM'].unique())


In [ ]:
adata.obs['disease_global_check'] = adata.obs['disease_global_check'].astype(str)
adata.obs['disease_global'] = adata.obs['disease_global'].astype(str)







In [ ]:
# adata.obs['disease_global_check'] =  adata.obs['disease_global_check'].fillna(adata.obs['disease_global'])
mask = (adata.obs['disease_global_check'] == 'nan')
adata.obs.loc[mask, 'disease_global_check'] = adata.obs.loc[mask, 'disease_global']


In [ ]:
adata.obs['disease_global_check'].unique()

In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()


In [ ]:
adata.obs.loc[adata.obs['GSM'] == 'GSM4504966', 'age']

In [ ]:
adata.obs.columns

In [ ]:
len(list(adata.obs['age'].unique()))


In [ ]:
adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal','Healthy']))]
adata_h.write('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_healthy.h5ad')


In [ ]:
adata_h_fib = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_healthy.h5ad')
adata_h_endo = sc.read_h5ad('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_delNoneAge_healthy.h5ad')
adata_h = sc.concat([adata_h_fib, adata_h_endo])
adata_h.shape



In [ ]:
adata_h_fib.shape, adata_h_endo.shape


In [ ]:
adata_h.obs['Age_group'].value_counts()



In [ ]:
adata_h_fib.obs['Celltype_general'].value_counts()


In [ ]:
adata_h.write('./data/scalex/fibro/human/age_prediction/Endo_Fibro_delNoneAge_healthy.h5ad')


In [ ]:
#################################### First try the random forest model 
# adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal', 'Healthy'])) & (adata.obs['Age']!=0)]
adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal','Healthy']))]  #  & (adata.obs['tissue']=='skin')
target_cell_types = list(set(adata_h.obs['Celltype']))


In [ ]:
df.set_index('sample', inplace=True)
df.head(10)


In [ ]:
df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')


In [ ]:
# adata_immune.shape, adata_can.shape, adata_Neu.shape, adata_rare.shape


In [ ]:

adata_immune = adata[adata.obs['disease_global_check'].isin(['Immune disease'])]  # ,'Infectious disease'
adata_can = adata[adata.obs['disease_global_check'] == 'Cancer']
adata_Neu = adata[adata.obs['disease_global_check'] == 'Neuronal disease']
adata_rare = adata[adata.obs['disease_global_check'].isin(['Cancer,Rare disease','Rare disease'])]


In [ ]:


# 过滤掉细胞类型太少的个体
cell_prop_data = df.iloc[:, 2:]
zero_counts = (cell_prop_data == 0).sum(axis=1)
max_zeros = 31  # 至少包含20种细胞类型
df = df[zero_counts < max_zeros]


In [ ]:
cell_prop_data = df.iloc[:, 2:]
zero_counts = (cell_prop_data == 0).sum(axis=1)


In [ ]:
adata_h = adata_h[adata_h.obs['Age']!=0]
target_cell_types = list(set(adata_h.obs['Celltype']))


In [ ]:
len(adata_h.obs['sample'].unique())


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict, KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def rf_model_cv(adata, target_cell_types, use_loocv=False, random_state=42):
    # 构建细胞类型比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    # df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)

    # 过滤掉细胞类型太少的个体
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少包含20种细胞类型
    # df = df[zero_counts < max_zeros]

    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0
    X = df[target_cell_types]
    y = df['Age']

    # 标准化
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    # 参数搜索空间
    param_dist = {
        'n_estimators': [300],
        'max_depth': [None,30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    }

    rf = RandomForestRegressor(random_state=random_state)
    cv = LeaveOneOut() if use_loocv else KFold(n_splits=5, shuffle=True, random_state=random_state)

    rf_cv = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=50,
                               scoring='r2', cv=cv, random_state=random_state, n_jobs=-1, verbose=0)
    rf_cv.fit(X_scaled, y)

    best_rf = rf_cv.best_estimator_
    best_params = rf_cv.best_params_

    # 交叉验证预测
    y_cv_pred = cross_val_predict(best_rf, X_scaled, y, cv=cv)
    mse = mean_squared_error(y, y_cv_pred)
    r2 = r2_score(y, y_cv_pred)

    # 拟合最终模型
    best_rf.fit(X_scaled, y)
    y_pred = best_rf.predict(X_scaled)

    print("Random Forest 最佳参数：", best_params)
    print(f"Random Forest CV MSE: {mse:.4f}")
    print(f"Random Forest CV R²: {r2:.4f}")

    # 可视化：特征重要性
    importances = pd.Series(best_rf.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(10)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
    plt.title("Top 10 Feature Importances (Random Forest)")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    return best_rf, y, y_pred, mse, r2, scaler
    

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

def plot_rf_prediction(y_true, y_pred, extra_points=None, extra_labels=None):
    """
    可视化 Random Forest 模型的预测效果
    :param y_true: 实际年龄（Series 或 array）
    :param y_pred: 预测年龄（array）
    :param extra_points: 可选，额外点的坐标 [(x1, y1), (x2, y2), ...]
    :param extra_labels: 可选，额外点的标签 [label1, label2, ...]
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    # 评价指标
    mae = mean_absolute_error(y_true, y_pred)
    mad = np.median(np.abs(y_true - y_pred))
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pcc, _ = pearsonr(y_true, y_pred)

    # 回归线
    reg = LinearRegression().fit(y_true.reshape(-1, 1), y_pred)
    slope, intercept = reg.coef_[0], reg.intercept_

    # 绘图
    plt.figure(figsize=(8, 6))
    x_min, x_max = -10, 130
    plt.xlim(x_min, x_max)
    plt.ylim(x_min, x_max)
    plt.scatter(y_true, y_pred, c='dodgerblue', alpha=0.7, edgecolor='k', label='Samples')
    plt.plot([x_min, x_max], [x_min, x_max], 'r--', lw=2, label='Ideal line (y = x)')

    # 回归线
    x_line = np.linspace(x_min, x_max, 100)
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, 'g-', lw=2, label='Regression line')

    # 评价指标框
    plt.text(
        0.05, 0.95,
        f"$R^2$: {r2:.2f}\nMAE: {mae:.2f}\nMAD:{mad:.2f}\nPCC: {pcc:.2f}",
        transform=plt.gca().transAxes,
        fontsize=16,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )

    # 添加额外点
    if extra_points:
        for i, (x, y) in enumerate(extra_points):
            label = extra_labels[i] if extra_labels else 'Extra Point'
            plt.scatter(x, y, color='red', edgecolor='black', s=100, marker='o', label=label)

    # 去重图例
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='lower right')

    plt.xlabel('True Age')
    plt.ylabel('Predicted Age')
    plt.title('Random Forest Model: Predicted vs True Age')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
rf_best, y_h, y_pred_h, mse, r2, scaler = rf_model_cv(adata_h, target_cell_types)


In [ ]:
# adata_h.obs['sample'].unique()


In [ ]:
plot_rf_prediction(y_h, y_pred_h, extra_points=None, extra_labels=None)


In [ ]:

##################################### 画 RF 的 age difference figure          ######################
def rf_age_diff(adata, rf_model, target_cell_types, scaler):
    import pandas as pd
    import numpy as np
    # 细胞比例计算
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少保留20种细胞类型
    df = df[zero_counts < max_zeros]
    print(f"剩余样本数: {len(df)}")
    # 保证目标细胞类型列存在
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0

    X = df[target_cell_types]
    y = df['Age']
    # 标准化
    X_scaled = scaler.transform(X)

    # 使用训练好的随机森林模型进行预测
    y_pred = rf_model.predict(X_scaled)
    y_pred = pd.Series(y_pred.flatten(), index=y.index)
    # 差值
    age_diff = y_pred - y
    return y, y_pred, age_diff


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_age_diff_boxplot(age_diff_dict):
    """
    绘制多组 age_diff 的箱线图，并使用 Mann-Whitney U 检验与第一组进行显著性标注。
    在计算和绘图前，去除每组数据中的异常值。

    参数:
    - age_diff_dict: 一个字典，key 为组名，value 为 age_diff 的 Series 或 array
    """
    def get_p_label(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'ns'
            
    def remove_outliers(data):
        """
        使用 IQR 方法去除数据中的异常值。
        处理输入为 pandas.Series 或 numpy.ndarray 的情况。
        """
        if isinstance(data, pd.Series):
            data = data.values  # 转换为 numpy 数组，方便进行 IQR 操作
        data = np.array(data)  # 确保 data 是 numpy 数组
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 返回去除异常值后的数据
        return data[(data >= lower_bound) & (data <= upper_bound)]

    # 整理数据为 DataFrame，并去除异常值
    plot_data = []
    for group, values in age_diff_dict.items():
        flat_vals = np.array(values).flatten()
        cleaned_vals = remove_outliers(flat_vals)  # 去除异常值
        plot_data.append(pd.DataFrame({'Age Difference': cleaned_vals, 'Group': group}))
    df_plot = pd.concat(plot_data, ignore_index=True)

    # 进行显著性检验：全部与第一个组（基准组）进行比较
    base_group_name = list(age_diff_dict.keys())[0]
    base_vals = np.array(remove_outliers(age_diff_dict[base_group_name])).flatten()  # 去除基准组异常值

    p_labels = []
    group_names = list(age_diff_dict.keys())
    for name in group_names[1:]:
        test_vals = np.array(remove_outliers(age_diff_dict[name])).flatten()  # 去除测试组异常值
        p = mannwhitneyu(base_vals, test_vals, alternative='two-sided').pvalue
        p_labels.append(get_p_label(p))

    # 绘图
    plt.figure(figsize=(0.9 * len(group_names), 5))
    ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2', showfliers=False)
    sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)

    y_max = df_plot['Age Difference'].max()
    y_min = df_plot['Age Difference'].min()
    y_range = y_max - y_min

    # 添加显著性标注
    def draw_sig_line(x1, x2, y, h, label):
        plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
        plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)

    for i, label in enumerate(p_labels):
        draw_sig_line(0, i+1, y_max + (i+1)*0.06*y_range, 0.02*y_range, label)
        
    plt.xlabel("Disease Type", fontsize=14)
    plt.ylabel("Age Difference", fontsize=14)
    plt.title('Predicted - True Age Difference(RF)', fontsize= 15)
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [ ]:
y_h, y_peed_h, age_diff_h = rf_age_diff(adata_h, rf_best, target_cell_types, scaler)
y_immune, y_pred_immune, age_diff_immune = rf_age_diff(adata_immune, rf_best, target_cell_types, scaler)
y_can, y_pred_can, age_diff_can = rf_age_diff(adata_can, rf_best, target_cell_types, scaler)
y_rare, y_pred_rare, age_diff_rare = rf_age_diff(adata_rare, rf_best, target_cell_types, scaler)
y_neu, y_pred_neu, age_diff_neu = rf_age_diff(adata_Neu, rf_best, target_cell_types, scaler)


In [ ]:
age_diff_dict = {
        'Healthy': age_diff_h,
    'Immune': age_diff_immune,
    'Cancer': age_diff_can,
        'Rare': age_diff_rare,
        'Neuron': age_diff_neu
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
# len(adata.obs['sample'].unique())


In [ ]:
adata

In [ ]:
coef.abs().sort_values(ascending=False)


In [ ]:
adata.obs['Age'].unique()

In [ ]:
################### Next try the Elastic Net model   ###################
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def en_model_cv(adata, target_cell_types,max_zeros=30, use_loocv=False):
    # 构建细胞类型比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = max_zeros  # 最少有20种细胞类型
    df = df[zero_counts < max_zeros]
    remaining_samples = df.index.str[:-2].unique()
    total_cells = cell_counts.loc[remaining_samples].sum().sum()
    print(f"剩余个体数: {df.shape[0]}")
    print(f"剩余细胞总数: {total_cells}")
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0
    X = df[target_cell_types]
    y = df['Age']
    # 标准化
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    # 交叉验证方式
    cv = LeaveOneOut() if use_loocv else KFold(n_splits=5, shuffle=True, random_state=42)
    # 构建ElasticNet模型（自动交叉验证选择alpha和l1_ratio）
    en_cv = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], 
                         alphas=np.logspace(-4, 2, 100), 
                         cv=cv, max_iter=10000)
    y_cv_pred = cross_val_predict(en_cv, X_scaled, y, cv=cv)
    mse = mean_squared_error(y, y_cv_pred)
    r2 = r2_score(y, y_cv_pred)
    # 拟合最终模型
    en_cv.fit(X_scaled, y)
    y_pred = en_cv.predict(X_scaled)
    print(f"ElasticNet 最佳 alpha: {en_cv.alpha_}")
    print(f"ElasticNet 最佳 l1_ratio: {en_cv.l1_ratio_}")
    print(f"ElasticNet CV MSE: {mse:.4f}")
    print(f"ElasticNet CV R²: {r2:.4f}")
    # 提取权重
    coef = pd.Series(en_cv.coef_, index=X.columns)
    top_features = coef.abs().sort_values(ascending=False).head(10)
    # 可视化：Top 权重细胞类型
    plt.figure(figsize=(8, 4))
    sns.barplot(x=top_features.values, y=top_features.index, palette='coolwarm')
    plt.title('Top 10 Celltype (Absolute Weight) in ElasticNet')
    plt.xlabel("Coefficient Weight")
    plt.tight_layout()
    plt.savefig(
        fig_path+lineage+'_'+type+ "_EN_Celltype_top10Features.pdf",
        dpi=300,
        bbox_inches='tight',
        format='pdf'
    )
    plt.savefig(fig_path+lineage+'_'+type+ "_EN_Celltype_top10Features.png", dpi=300, bbox_inches='tight')
    plt.show()
    return en_cv, y, y_pred, mse, r2, scaler


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

def plot_en_prediction(y_true, y_pred, extra_points=None, extra_labels=None):
    """
    可视化 ElasticNet 模型的预测效果
    :param y_true: 实际年龄（Series 或 array）
    :param y_pred: 预测年龄（array）
    :param extra_points: 可选，额外点的坐标 [(x1, y1), (x2, y2), ...]
    :param extra_labels: 可选，额外点的标签 [label1, label2, ...]
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    # 评价指标
    mad = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pcc, _ = pearsonr(y_true, y_pred)
    # 回归线
    reg = LinearRegression().fit(y_pred.reshape(-1, 1), y_true)
    slope, intercept = reg.coef_[0], reg.intercept_
    # 绘图
    plt.figure(figsize=(8, 6))
    x_min, x_max = -10, 130
    plt.xlim(x_min, x_max)
    plt.ylim(x_min, x_max)
    plt.scatter(y_true, y_pred, c='dodgerblue', alpha=0.7, edgecolor='k', label='Samples')
    plt.plot([x_min, x_max], [x_min, x_max], 'r--', lw=2, label='Ideal line (y = x)')
    # 回归线
    x_line = np.linspace(x_min, x_max, 100)
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, 'g-', lw=2, label='ElasticNet regression line')
    # 评价指标框
    plt.text(
        0.05, 0.95,
        f"$R^2$: {r2:.2f}\nMAD: {mad:.2f}\nPCC: {pcc:.2f}",
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )
    # 添加额外点
    if extra_points:
        for i, (x, y) in enumerate(extra_points):
            label = extra_labels[i] if extra_labels else 'Extra Point'
            plt.scatter(x, y, color='red', edgecolor='black', s=100, marker='o', label=label)
    # 去重图例
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='lower right')
    plt.xlabel('True Age')
    plt.ylabel('Predicted Age')
    plt.title('ElasticNet(Celltype): Predicted vs True Age')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
def en_age_diff(adata, en_model, target_cell_types, scaler):
    import pandas as pd
    import numpy as np
    # 细胞比例计算
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少保留20种细胞类型
    df = df[zero_counts < max_zeros]
    print(f"剩余样本数: {len(df)}")
    # 保证目标细胞类型列存在
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0    
    X = df[target_cell_types]
    y = df['Age']
    # 标准化
    X_scaled = scaler.transform(X)
    # 预测
    y_pred = en_model.predict(X_scaled)
    y_pred = pd.Series(y_pred.flatten(), index=y.index)
    # 差值
    age_diff = y_pred - y
    return y, y_pred, age_diff


In [ ]:
target_cell_types = list(adata.obs['Celltype'].unique())
len(target_cell_types)

In [ ]:
en_best, y_h, y_pred_h, best_n, mse_list, scaler = en_model_cv(adata, target_cell_types,max_zeros=29)


In [ ]:
adata.obs['Age_group'].value_counts()



In [ ]:
len(adata.obs['sample'].unique())  
### [max_zeros,R2,MAD,PCC], Fibro_delNone_A
### max_zeros=19就 20个样本了
# [[25,0.44,5.94,0.69],[20,0.74,3.77,0.87],[19,0.34,7.86,0.58],[21,0,9.58,nan],[22,0.52,6.13,0.76],[23,0.62,4.95,0.79],[24,0.46,5.87,0.69],[26,0.51,5.54,0.72],[27,0.63,4.74.0.79],[28,0,8.93,nan],[29,0,8.88,nan]]



In [ ]:
plot_en_prediction(y_h, y_pred_h, extra_points=None, extra_labels=None)


In [ ]:
plot_en_prediction(y_h, y_pred_h, extra_points=None, extra_labels=None)


In [ ]:
y_h, y_peed_h, age_diff_h = en_age_diff(adata_h, en_best, target_cell_types, scaler)
y_immune, y_pred_immune, age_diff_immune = en_age_diff(adata_immune, en_best, target_cell_types, scaler)
y_can, y_pred_can, age_diff_can = en_age_diff(adata_can, en_best, target_cell_types, scaler)
y_Neu, y_pred_Neu, age_diff_Neu = en_age_diff(adata_Neu, en_best, target_cell_types, scaler)
y_rare, y_pred_rare, age_diff_rare = en_age_diff(adata_rare, en_best, target_cell_types, scaler)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_age_diff_boxplot(age_diff_dict):
    """
    绘制多组 age_diff 的箱线图，并使用 Mann-Whitney U 检验与第一组进行显著性标注。
    在计算和绘图前，去除每组数据中的异常值。
    参数:
    - age_diff_dict: 一个字典，key 为组名，value 为 age_diff 的 Series 或 array
    """
    def get_p_label(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'ns'

    def remove_outliers(data):
        """
        使用 IQR 方法去除数据中的异常值。
        处理输入为 pandas.Series 或 numpy.ndarray 的情况。
        """
        if isinstance(data, pd.Series):
            data = data.values  # 转换为 numpy 数组，方便进行 IQR 操作
        data = np.array(data)  # 确保 data 是 numpy 数组
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 返回去除异常值后的数据
        return data[(data >= lower_bound) & (data <= upper_bound)]

    # 整理数据为 DataFrame，并去除异常值
    plot_data = []
    for group, values in age_diff_dict.items():
        flat_vals = np.array(values).flatten()
        cleaned_vals = remove_outliers(flat_vals)  # 去除异常值
        plot_data.append(pd.DataFrame({'Age Difference': cleaned_vals, 'Group': group}))
    df_plot = pd.concat(plot_data, ignore_index=True)
    # 进行显著性检验：全部与第一个组（基准组）进行比较
    base_group_name = list(age_diff_dict.keys())[0]
    base_vals = np.array(remove_outliers(age_diff_dict[base_group_name])).flatten()  # 去除基准组异常值
    p_labels = []
    group_names = list(age_diff_dict.keys())
    for name in group_names[1:]:
        test_vals = np.array(remove_outliers(age_diff_dict[name])).flatten()  # 去除测试组异常值
        p = mannwhitneyu(base_vals, test_vals, alternative='two-sided').pvalue
        p_labels.append(get_p_label(p))

    # 绘图
    plt.figure(figsize=( 0.9* len(group_names), 5))
    ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2', showfliers=False)
    sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)
    y_max = df_plot['Age Difference'].max()
    y_min = df_plot['Age Difference'].min()
    y_range = y_max - y_min
    # 添加显著性标注
    def draw_sig_line(x1, x2, y, h, label):
        plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
        plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)
    for i, label in enumerate(p_labels):
        draw_sig_line(0, i+1, y_max + (i+1)*0.06*y_range, 0.02*y_range, label)

    plt.xlabel("Disease Type", fontsize=14)
    plt.ylabel("Age Difference", fontsize=14)
    plt.title('Predicted - True Age Difference(EN)', fontsize= 15)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

In [ ]:
age_diff_dict = {
        'Healthy':age_diff_h,
        'Immune':age_diff_immune,
        'Cancer':age_diff_can ,    'Neuron':age_diff_Neu,
    'Rare':age_diff_rare
    }

plot_age_diff_boxplot(age_diff_dict)


In [ ]:
adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNone_A_Age_healthy.h5ad')
target_cell_types = list(adata.obs['Celltype'].unique())


In [ ]:
# len(target_cell_types)


In [ ]:
def plsr_model_cv(adata, target_cell_types, max_components=10,max_zeros=30, use_loocv=False):
    # 构建细胞类型比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = max_zeros   # 每个样本中至少有 len(target_cell_types) -max_zeros 种 Celltype
    df = df[zero_counts < max_zeros]
    remaining_individuals = df.index.str[:-2].unique()  # 取去掉后缀的原名字
    total_cells = cell_counts.loc[remaining_individuals].sum().sum()
    print(f"剩余个体数: {df.shape[0]}")
    print(f"剩余细胞总数: {total_cells}")
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0
    X = df[target_cell_types]
    y = df['Age']
    # 标准化
    # scaler = StandardScaler()
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    # 交叉验证
    cv = LeaveOneOut() if use_loocv else KFold(n_splits=5, shuffle=True, random_state=42)
    mse_list = []
    ## max_components控制的是使用的 PC(Celltype), X.shape[1] = len(target_cell_types), X.columns 也就是 target_cell_types
    for n in range(1, min(max_components, X.shape[1]) + 1):  
        pls = PLSRegression(n_components=n)
        y_cv_pred = cross_val_predict(pls, X_scaled, y, cv=cv)
        mse = mean_squared_error(y, y_cv_pred)
        mse_list.append(mse)
    best_n = np.argmin(mse_list) + 1
    # 用最优 n_components 再跑一次交叉验证
    pls_best_cv = PLSRegression(n_components=best_n)
    y_cv_pred = cross_val_predict(pls_best_cv, X_scaled, y, cv=cv)
    r2_cv = r2_score(y, y_cv_pred)
    pcc_cv, _ = pearsonr(y, y_cv_pred)
    mad_cv = mean_absolute_error(y, y_cv_pred)
    rmse_cv = np.sqrt(mean_squared_error(y, y_cv_pred))
    print(f"\n交叉验证结果 (n_components={best_n}):")
    print(f"R²: {r2_cv:.4f}")
    print(f"PCC: {pcc_cv:.4f}")
    print(f"MAD: {mad_cv:.2f}")
    print(f"RMSE: {rmse_cv:.2f}")
    # 训练最终模型
    pls_best = PLSRegression(n_components=best_n)
    pls_best.fit(X_scaled, y)
    y_pred = pls_best.predict(X_scaled)
    # 显示主成分
    x_weights = pd.DataFrame(pls_best.x_weights_, 
                         index=X.columns, 
                         columns=[f'Component {i+1}' for i in range(best_n)])
    # 查看每个主成分中贡献最大的细胞类型（绝对值最大的）
    top_features_per_component = x_weights.abs().apply(lambda col: col.sort_values(ascending=False).head(5))
    print("每个主成分中贡献最大的细胞类型（按权重绝对值）:")
    print(top_features_per_component)
    for i in range(best_n):
        component_name = f'Component {i+1}'
        top_weights = x_weights[component_name].sort_values(key=abs, ascending=False).head(10)
        colors = ['red' if w > 0 else 'blue' for w in top_weights.values]
        plt.figure(figsize=(8, 4))
        sns.barplot(x=top_weights.values, y=top_weights.index, palette=colors)
        plt.title(f"Top 10 Celltype contribute to {component_name}", fontsize=20)
        plt.xlabel("Weight", fontsize=18)
        plt.ylabel("Cell type", fontsize=18)
        plt.yticks(fontsize=16, fontweight='bold')
        plt.xticks(fontsize=14)
        plt.tight_layout()
        # plt.savefig(
        #     fig_path+lineage+'_'+type+ "_PLSR_Celltype_top10Features.pdf",
        #     dpi=300,
        #     bbox_inches='tight',
        #     format='pdf'
        # )
        # plt.savefig(fig_path+lineage+'_'+type+ "_PLSR_Celltype_top10Features.png", dpi=300, bbox_inches='tight')
        plt.show()
    # 可视化：主成分个数 vs 交叉验证 MSE
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(mse_list) + 1), mse_list, marker='o', linestyle='-', color='teal')
    plt.axvline(best_n, color='red', linestyle='--', label=f'Best n_components = {best_n}')
    plt.xlabel('Number of PLS Components')
    plt.ylabel('Cross-validated MSE')
    plt.title('PLSR Model Selection')
    plt.legend()
    plt.tight_layout()
    # plt.savefig(
    #     fig_path+lineage+'_'+type+ "_PLSR_Celltype_nPC_selection.pdf",
    #     dpi=300,
    #     bbox_inches='tight',
    #     format='pdf'
    # )
    # plt.savefig(fig_path+lineage+'_'+type+ "_PLSR_Celltype_nPC_selection.png", dpi=300, bbox_inches='tight')
    plt.show()
    return pls_best, y, y_pred.flatten(), best_n, mse_list, scaler


In [ ]:
target_cell_types = list(set(adata.obs['Celltype']))


In [ ]:
adata.shape

In [ ]:
pls_best, y_h, y_pred_h, best_n, mse_list, scaler = plsr_model_cv(adata, target_cell_types,max_zeros=40,max_components=30)


In [ ]:
################ 再试一下 PLSR 方法   ################ 
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score

def plsr_model(adata, target_cell_types, n_components):
    # Step 1: 构建每个样本中各个细胞类型的比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
    cell_props = cell_props.dropna(axis=0, how='any')
    # 重置索引
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    # 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
    df['sample'] = (
        df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    )
    df.set_index('sample', inplace=True)
    # Step 2: 提取感兴趣的细胞类型的比例作为X
    X = df[target_cell_types]
    # Step 3: 构建y（年龄）
    y = df['Age']
    # Step 4: 标准化特征
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    # Step 5: 构建PLSR模型（设定成1-3个主成分可以交叉验证选择）
    pls = PLSRegression(n_components=n_components)
    pls.fit(X_scaled, y)
    y_pred = pls.predict(X_scaled)
    return pls, y, y_pred



In [ ]:
# pls, y_h, y_pred_h = plsr_model(adata_h, target_cell_types, n_components=10)


import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import numpy as np

# 计算评价指标
mad = mean_absolute_error(y_h, y_pred_h)
mse = mean_squared_error(y_h, y_pred_h)
rmse = np.sqrt(mse)
r2 = r2_score(y_h, y_pred_h)
pcc, _ = pearsonr(y_h, y_pred_h)

# 打印指标
print(f"MAD（Mean Absolute Deviation）: {mad:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² score: {r2:.4f}")
print(f"PCC (Pearson Correlation Coefficient): {pcc:.4f}")

# 绘图
plt.figure(figsize=(6, 6))
plt.scatter(y_h, y_pred_h, alpha=0.8, edgecolor='k', s=60, c='skyblue', label='Samples')
plt.plot([y_h.min(), y_h.max()], [y_h.min(), y_h.max()], 'r--', lw=2, label='Ideal Fit')

# 美化图像
plt.xlabel("Chronological Age", fontsize=12)
plt.ylabel("Predicted Age", fontsize=12)
plt.title("PLSR Age Prediction", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# 添加文本：R²、MAD 和 PCC
plt.text(
    0.05, 0.95,
    f"$R^2$: {r2:.2f}\nMAD: {mad:.2f}\nPCC: {pcc:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
)

plt.legend(loc='lower right')
plt.tight_layout()
plt.show()



In [ ]:
# Cancer 

cell_counts = adata_can.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_can = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_can = pls.predict(X_scaled) # 
age_diff_can = y_pred_can -  y_can
age_diff_can


In [ ]:
# Immune

cell_counts = adata_immune.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_immune = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_immune = pls.predict(X_scaled) # 
age_diff_immune = y_pred_immune -  y_immune
age_diff_immune


In [ ]:
# Neuron

cell_counts = adata_Neu.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_neu = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_neu = pls.predict(X_scaled) # 
age_diff_neu = y_pred_neu -  y_neu
age_diff_neu


In [ ]:
# Rare

cell_counts = adata_rare.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_rare = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_rare = pls.predict(X_scaled) # 
age_diff_rare = y_pred_rare -  y_rare
age_diff_rare


In [ ]:
############################### PLSR 的 age difference
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# 转为一维数组
healthy_vals = age_diff_h.to_numpy().flatten()
can_vals = age_diff_can.to_numpy().flatten()
immune_vals = age_diff_immune.to_numpy().flatten()
neu_vals = age_diff_neu.to_numpy().flatten()
rare_vals = age_diff_rare.to_numpy().flatten()

# 整理绘图数据
df_plot = pd.DataFrame({
    'Age Difference': np.concatenate([healthy_vals, can_vals,immune_vals, neu_vals,rare_vals]),
    'Group': (['Healthy'] * len(healthy_vals) +
              ['Cancer'] * len(can_vals) +
              ['Immune'] * len(immune_vals) +
              ['Neuron'] * len(neu_vals) + ['Rare'] * len(rare_vals)) 
})

# Mann-Whitney U检验
def get_p_label(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

# 只和 Healthy 组比
p_hc = mannwhitneyu(healthy_vals, can_vals, alternative='two-sided').pvalue
p_hi = mannwhitneyu(healthy_vals, immune_vals, alternative='two-sided').pvalue
p_hn = mannwhitneyu(healthy_vals, neu_vals, alternative='two-sided').pvalue
p_hr = mannwhitneyu(healthy_vals, rare_vals, alternative='two-sided').pvalue

label_hc = get_p_label(p_hc)
label_hi = get_p_label(p_hi)
label_hn = get_p_label(p_hn)
label_hr = get_p_label(p_hr)

# 可视化
plt.figure(figsize=(5, 6))
ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2')
sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)

# 获取最大 y 值用于标注位置
y_max = df_plot['Age Difference'].max()
y_min = df_plot['Age Difference'].min()
y_range = y_max - y_min

# 添加显著性连线和星号
def draw_sig_line(x1, x2, y, h, label):
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)

draw_sig_line(0, 1, y_max + 0.02*y_range, 0.02*y_range, label_hc)  # Healthy vs COVID
draw_sig_line(0, 2, y_max + 0.08*y_range, 0.02*y_range, label_hi)  # Healthy vs AML
draw_sig_line(0, 3, y_max + 0.14*y_range, 0.02*y_range, label_hn)  # Healthy vs IAV
draw_sig_line(0, 4, y_max + 0.2*y_range, 0.02*y_range, label_hr)  # Healthy vs IAV

plt.xlabel("Disease Type", fontsize=14)
plt.ylabel("Age Difference", fontsize=14)
plt.title('Predicted - True Age Difference(PLSR)', fontsize= 15)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()


In [ ]:
sorted(adata.obs['Age_group'].unique())


In [ ]:
len(adata.obs['Age'].unique())


In [ ]:
###### Python 中如何将单细胞数据如何将

In [ ]:
################################### JinDong Han's paper      ##############################
import os,sys
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt



# -------------------------- 统计样本×亚型的细胞数量并计算比例 --------------------------
obs_age_filtered = adata.obs[
    (adata.obs["Age_group"].isin(['B','C','D','E','F','G','H']))].copy()

initial_cell_count = adata.obs.shape[0]
# 统计每个样本不同亚型地细胞数量（不分健康/疾病）
cell_count = obs_age_filtered.groupby(['sample', 'Celltype']).size().unstack(fill_value=0)
print(f"原始计数矩阵维度：{cell_count.shape[0]}个样本 × {cell_count.shape[1]}个亚型")

# 转换为相对比例（每行求和=1，确保每个样本的亚型占比总和为1，避免绝对数量偏差）
cell_prop_all = cell_count.div(cell_count.sum(axis=1), axis=0)
# 处理零值
cell_prop_all = cell_prop_all.dropna(how='all', axis=0)  # 删除所有亚型比例为NaN的样本
print(f"比例矩阵维度（过滤无效样本后）：{cell_prop_all.shape[0]}个样本 × {cell_prop_all.shape[1]}个亚型")


# -------------------------- 过滤低丰度亚型 --------------------------
min_sample_ratio = 0.05  # 保留标准：在≥5%的样本中出现过（即该亚型至少在5%样本中有非零比例）
# 计算每个亚型的“非零样本数”（统计该亚型在多少样本中存在）
subtype_non_zero_count = cell_prop_all.gt(0).sum(axis=0)  # gt(0)：比例>0视为“存在”，sum(axis=0)按亚型统计样本数
# 筛选出符合条件的亚型（非零样本数 ≥ 总样本数×5%）
valid_subtypes = subtype_non_zero_count[subtype_non_zero_count >= cell_prop_all.shape[0] * min_sample_ratio].index.tolist()
# 过滤低丰度亚型，更新比例矩阵
cell_prop_all = cell_prop_all[valid_subtypes]
print(f"\n过滤后保留的细胞亚型数：{len(valid_subtypes)}（原始{len(subtype_non_zero_count)}个）")
print(f"最终比例矩阵维度：{cell_prop_all.shape[0]}个样本 × {cell_prop_all.shape[1]}个亚型")


# -------------------------- 将meta的age列从category转为float（数值型） --------------------------
print(f"\n转换age列数据类型（从category→float）")
# 转换前的age列信息
print(f"转换前：age列类型 = {obs_age_filtered['age'].dtype}")

obs_age_filtered['age'] = obs_age_filtered['age'].astype(str).apply(
    lambda x: pd.to_numeric(x, errors='coerce')
)

# # 验证转换结果
# print(f"转换后：age列类型 = {obs_age_filtered['age'].dtype}（目标类型：float64）")
# print(f"转换后：age列数值范围 = {obs_age_filtered['age'].min():.1f}岁 ~ {obs_age_filtered['age'].max():.1f}岁")
# print(f"转换后：有效细胞数 = {obs_age_filtered.shape[0]}（原始{initial_cell_count}个，过滤{initial_cell_count - obs_age_filtered.shape[0]}个无效细胞）")
# print(f"转换后：age列前5个值 = {list(obs_age_filtered['age'].head().round(1))}（已转为数值型）")


# 获取所有健康样本（disease_global为Healthy或none）
healthy_sample_ids = obs_age_filtered[obs_age_filtered['disease_global'].isin(['Healthy', 'Normal'])]['sample'].unique()
print(f"\n健康样本数：{len(healthy_sample_ids)}")

# 筛选健康样本的比例数据
cell_prop_healthy = cell_prop_all.loc[cell_prop_all.index.isin(healthy_sample_ids)]

# 获取健康样本对应的实际年龄
sample_age_healthy = obs_age_filtered[obs_age_filtered['sample'].isin(healthy_sample_ids)]\
    .groupby('sample')['age']\
    .apply(
        lambda x: 
            # 转为数值（非数值→NaN）
            pd.to_numeric(x, errors='coerce')
            # 删除NaN
            .dropna()
            # 有有效数据则取第一个，无则返回NaN
            .iloc[0] if not pd.to_numeric(x, errors='coerce').dropna().empty else pd.NA
    )
sample_age_healthy = sample_age_healthy.dropna()
print(f"有效健康样本数（年龄可转换且非缺失）：{len(sample_age_healthy)}")
sample_age_healthy = sample_age_healthy.astype(np.float64)

# 对齐比例矩阵和年龄的样本顺序（删除无有效年龄的样本）
cell_prop_healthy = cell_prop_healthy.reindex(sample_age_healthy.index).dropna()
sample_age_healthy = sample_age_healthy.reindex(cell_prop_healthy.index)

# 最终建模数据（仅健康样本，且年龄为数值）
X = cell_prop_healthy.values  
y = sample_age_healthy.values  
subtype_names = cell_prop_healthy.columns.tolist()

# 验证年龄类型和数据
print(f"年龄数据类型：{sample_age_healthy.dtype}")  # 应输出 float64/int64
print(f"最终建模数据维度：{X.shape[0]}个健康样本 × {X.shape[1]}个亚型")
print(f"健康样本年龄范围：{y.min():.1f}岁 ~ {y.max():.1f}岁")

# -------------------------- PLSR模型（保留前3个主成分）--------------------------
# 划分训练集/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 训练PLSR模型（n_components=3，避免过拟合）
pls = PLSRegression(n_components=3)
pls.fit(X_train, y_train)

# 模型评估
y_pred = pls.predict(X_test)

# 生成y_test
y_test_arr = np.asarray(y_test).flatten()  
# 生成y_pred
y_pred_arr = y_pred.flatten() 

# 计算PCC
pcc = np.corrcoef(y_test_arr, y_pred_arr)[0, 1]
# 计算MAD
mad = mean_absolute_error(y_test_arr, y_pred_arr)

print("\n=== PLSR模型性能（健康样本）===")
print(f"皮尔逊相关系数（PCC）：{pcc:.3f}")
print(f"平均绝对偏差（MAD）：{mad:.2f}岁")


# -------------------------- 识别衰老相关亚型（文献用VIP值，VIP>1为核心亚型）--------------------------
def calculate_plsr_vip(pls_model):
    """按文献公式计算VIP值（变量重要性）"""
    t = pls_model.x_scores_  # 样本得分：n_samples × n_components
    w = pls_model.x_weights_  # 特征权重：n_subtypes × n_components
    q = pls_model.y_loadings_  # 因变量载荷：1 × n_components
    p, h = w.shape
    vip = np.zeros(p)
    
    for j in range(p):
        s = np.diag(t.T @ t @ q.T @ q).sum()
        vip[j] = np.sqrt(p * (w[j, :] ** 2 @ (t.T @ t @ q.T @ q)).sum() / s)
    return vip

# 计算VIP值并筛选核心亚型
vip_scores = calculate_plsr_vip(pls)
vip_df = pd.DataFrame({
    'Celltype': subtype_names,
    'VIP_score': vip_scores
}).sort_values('VIP_score', ascending=False)

# 衰老相关核心亚型（标准：VIP>1）
aging_subtypes = vip_df[vip_df['VIP_score'] > 1].reset_index(drop=True)
print(f"\n=== 衰老相关核心亚型（VIP>1）===")
print(aging_subtypes)

# -------------------------- EN模型训练（文献对比模型，参数与文献一致）--------------------------
en = ElasticNet(alpha=5e-4, l1_ratio=0.08, max_iter=10000, random_state=42)
en.fit(X_train, y_train)

# 模型评估
y_pred_en = en.predict(X_test)
pcc_en = np.corrcoef(y_test, y_pred_en)[0, 1]
mad_en = mean_absolute_error(y_test, y_pred_en)

print("\n=== EN模型性能（健康样本）===")
print(f"皮尔逊相关系数（PCC）：{pcc_en:.3f}")
print(f"平均绝对偏差（MAD）：{mad_en:.2f}岁")

# EN模型高贡献亚型（权重绝对值排序）
en_coef_df = pd.DataFrame({
    'Celltype': subtype_names,
    'Coefficient': en.coef_,
    'Abs_Coefficient': np.abs(en.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print(f"\n=== EN模型前10高贡献亚型===")
print(en_coef_df.head(10))



# -------------------------- 预测疾病样本年龄并计算年龄差 --------------------------
# 定义目标疾病类型
target_disease_types = ['Cancer', 'Infectious disease', 'Immune disease', 'Respiratory disease']

# 提取目标疾病类型的样本
disease_sample_ids = obs_age_filtered[obs_age_filtered['disease_global'].isin(target_disease_types)]['sample'].unique()
print(f"目标疾病类型样本总数：{len(disease_sample_ids)}")

# 筛选疾病样本的比例数据（保持与健康样本一致的亚型）
cell_prop_disease = cell_prop_all.loc[cell_prop_all.index.isin(disease_sample_ids)]
# 确保疾病样本的亚型与建模时一致（避免维度不匹配）
cell_prop_disease = cell_prop_disease.reindex(columns=subtype_names).fillna(0)  # 缺失亚型填充0
print(f"疾病样本比例矩阵维度：{cell_prop_disease.shape[0]}个样本 × {cell_prop_disease.shape[1]}个亚型")

# 处理疾病样本的实际年龄
sample_age_disease = obs_age_filtered[obs_age_filtered['sample'].isin(disease_sample_ids)]\
    .groupby('sample')['age']\
    .apply(
        lambda x: 
            pd.to_numeric(x, errors='coerce')
            .dropna()
            .iloc[0] if not pd.to_numeric(x, errors='coerce').dropna().empty else pd.NA
    )
sample_age_disease = sample_age_disease.dropna().astype(np.float64)
print(f"有效疾病样本数（年龄可转换且非缺失）：{len(sample_age_disease)}")

# 对齐比例矩阵和年龄数据（删除无有效年龄的样本）
cell_prop_disease = cell_prop_disease.reindex(sample_age_disease.index).dropna()
sample_age_disease = sample_age_disease.reindex(cell_prop_disease.index)

# 用训练好的PLSR模型预测疾病样本年龄
X_disease = cell_prop_disease.values
y_true_disease = sample_age_disease.values
y_pred_disease = pls.predict(X_disease).flatten()  # 展平为一维数组

# 计算年龄差（预测年龄 - 实际年龄）
age_diff_disease = y_pred_disease - y_true_disease

# 整理疾病样本的年龄差数据（包含疾病类型标签）
disease_age_diff_df = pd.DataFrame({
    'sample': cell_prop_disease.index,
    'disease_type': [obs_age_filtered[obs_age_filtered['sample'] == s]['disease_global'].iloc[0] for s in cell_prop_disease.index],
    'age_diff': age_diff_disease
})

# 整理健康样本的年龄差数据（用于对比）
# 重新计算健康样本的预测年龄（训练集+测试集）
y_pred_healthy = pls.predict(X).flatten()
age_diff_healthy = y_pred_healthy - y
healthy_age_diff_df = pd.DataFrame({
    'sample': cell_prop_healthy.index,
    'disease_type': 'Healthy',
    'age_diff': age_diff_healthy
})

# 合并所有数据（Healthy + 四种疾病类型）
all_age_diff_df = pd.concat([healthy_age_diff_df, disease_age_diff_df], ignore_index=True)
# 确保疾病类型顺序与目标列表一致（绘图时按指定顺序排列）
all_age_diff_df['disease_type'] = pd.Categorical(
    all_age_diff_df['disease_type'],
    categories=['Healthy'] + target_disease_types,
    ordered=True
)
print(f"\n合并后的数据量：{len(all_age_diff_df)}个样本")
print(f"各类型样本数：\n{all_age_diff_df['disease_type'].value_counts()}")



In [ ]:
# adata.obs[['Age','Celltype']].drop_duplicates()
adata.obs['Age_group'].unique()


In [ ]:
meta_data = adata.obs


In [ ]:
meta_data.columns


In [ ]:
meta_data[['sample','tissue','age','age_group']]


In [ ]:
tissue_stats['cell_count'].sum()


In [ ]:
colors = plt.cm.tab20(np.linspace(0, 1, tissue_stats['tissue'].nunique()))
tissue_color_map = dict(zip(tissue_stats['tissue'].unique(), colors))


In [ ]:
tissue_stats


In [ ]:
adata_h


In [ ]:
celltype_check = set(['Fibroblasts','Endothelial','Mono/Macro'])
tisch_dataset_use = []
for dataset in tisch_dataset:
    print(dataset)
    os.chdir(directory + dataset)
    files = os.listdir(directory+ dataset)
    json_file = [f for f in files if f.endswith(dataset+'_umap.json')]
    with open(json_file[0], 'r') as file:
        meta_tmp = json.load(file)
    meta_df = pd.DataFrame(meta_tmp)
    ct_unique = set(meta_df['Celltype_curated'].unique())
    if celltype_check.issubset(ct_unique):
        tisch_dataset_use.append(dataset)


In [ ]:
########### 更高质量的图 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# 假设你的数据如下：
# tissue_stats = pd.read_csv("tissue_stats.csv")

# 用更稳定的排序（避免文字重叠）
tissue_stats = tissue_stats.sort_values("tissue").reset_index(drop=True)

# 计算基本参数
N = len(tissue_stats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
width = 2 * np.pi / N * 0.35  # 控制柱宽

# 设置环半径
inner_radius = 8       # sampleN 的半径起点
gap = 1.5              # sampleN 和 cellN 两圈之间的间距

# 配色方案
color_map = cm.get_cmap('tab20', N)
colors = [color_map(i) for i in range(N)]

# 计算缩放比例，使不同量级数据在图中均衡
sample_scale = tissue_stats['sampleN'].max() / 3
cell_scale = tissue_stats['cellN'].max() / 3

# 建立画布
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': 'polar'})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_axis_off()

# 绘制内圈 sampleN
for i, (idx, row) in enumerate(tissue_stats.iterrows()):
    theta = angles[i]
    ax.bar(
        theta,
        row['sampleN'] / sample_scale,
        width=width,
        bottom=inner_radius,
        color=colors[i],
        alpha=0.9,
        edgecolor='black',
        linewidth=0.8
    )
    # 添加 sampleN 标签
    ax.text(
        theta, inner_radius + row['sampleN'] / sample_scale + 0.5,
        f"Sample: {int(row['sampleN'])}",
        rotation=np.degrees(theta - np.pi/2),
        rotation_mode='anchor',
        ha='center', va='bottom',
        fontsize=7
    )

# 绘制外圈 cellN
for i, (idx, row) in enumerate(tissue_stats.iterrows()):
    theta = angles[i]
    ax.bar(
        theta,
        row['cellN'] / cell_scale,
        width=width,
        bottom=inner_radius + gap + 2,
        color=colors[i],
        alpha=0.6,
        edgecolor='black',
        linewidth=0.8
    )
    # 添加 cellN 标签
    ax.text(
        theta, inner_radius + gap + 2 + row['cellN'] / cell_scale + 0.5,
        f"Cells: {int(row['cellN'])}",
        rotation=np.degrees(theta - np.pi/2),
        rotation_mode='anchor',
        ha='center', va='bottom',
        fontsize=7,
        color='dimgray'
    )

# 添加 tissue 标签
for i, (idx, row) in enumerate(tissue_stats.iterrows()):
    theta = angles[i]
    ax.text(
        theta,
        inner_radius - 1,
        row['tissue'],
        rotation=np.degrees(theta - np.pi/2),
        rotation_mode='anchor',
        ha='center', va='center',
        fontsize=8, fontweight='bold'
    )

# 添加标题
plt.title("Tissue-wise Distribution of Samples and Cells", fontsize=16, fontweight='bold', pad=40)

# 添加图例
sample_patch = plt.Line2D([0], [0], color='gray', lw=6, alpha=0.9, label='SampleN (Inner Ring)')
cell_patch = plt.Line2D([0], [0], color='gray', lw=6, alpha=0.6, label='CellN (Outer Ring)')
plt.legend(
    handles=[sample_patch, cell_patch],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    frameon=False,
    fontsize=10
)

plt.tight_layout()
plt.show()


In [ ]:
adata.obs['Celltype'].unique()


In [ ]:
# sorted(dir())


gene_keep = list(set([item.split('_')[1] for item in adata.obs['Celltype'].unique().tolist() ]))
gene_keep


In [ ]:
# ([item.split('_')[1] for item in adata.obs['Celltype'].unique().tolist() ])


In [ ]:
set([item.split('_')[0] for item in adata.obs['Celltype'].unique().tolist() ])


In [ ]:
# sorted(adata.obs['Celltype'].unique())


In [ ]:
fibroblast_markers = {
    'Myfibro': ["ACTA2", "TNS1", "CDH11",'MYL9','TPM2','TAGLN','ACTN2','ACTA1','GJA4','RGS5'],
    'apFibro': [
        'CD74', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DQA1',
        'CIITA', 'CD80', 'CD86', 'LYZ', 'C1QA', 'C1QB', 'C1QC'
    ],
    'eFibro': [
        'COL1A1', 'COL1A2', 'COL3A1', 'COL6A1', 'COL6A2',
        'FN1', 'LUM', 'DCN', 'BGN', 'VCAN', 'SPARC', 'FBN1', 'ELN'
    ],
    'iFibro': [
        'CXCL12', 'CXCL14', 'IL6', 'IL11', 'IL33','APOD','PI16',
        'CCL2', 'CCL19', 'CCL21', 'PDGFRA', 'GREM1', 'NT5E', 'CD34'
    ],
    'qFibro': [
        'CFD', 'PI16', 'CLU', 'APOE', 'CD36','LY6A', 'PDGFRA', 'ITGA8', 'CRYAB', 'SFRP2', 'SFRP4'
    ]
}

# 获取所有唯一的marker基因
fibro_markers = list(set([
    gene for sublist in fibroblast_markers.values() 
    for gene in sublist
]))


In [ ]:
adata.obs[['Celltype','Celltype_general','Celltype_sub']].drop_duplicates()


In [ ]:
gene_keep = [
  'APOE', 'CFD', 'CLU', # qFibro
  'CD74','B2M','S100A1', # apFibro
   'APOD', 'CILP','ADH1B', 'DCN','CXCL14', 'VIM', # eFibro
  "RPS19",'CDH12','PI16', # iFibro
  "ACTA2",'MYL9','TPM2','TAGLN' # Myfibro, 原来的   "ACTA2",'GJA4','RGS5'
    ]

In [ ]:
iFibro_extended_markers = ['CXCL12', 'IL6', 'CCL2', 'ICAM1', 'CXCL1', 'CXCL8', 'IL33', 'PTGS2']
confusion_markers = {
    'apFibro': ['CD74', 'HLA-DRA', 'CIITA'],
    'eFibro': ['COL1A1', 'COL3A1'],
    'Myfibro': ['ACTA2', 'TAGLN'],
    'qFibro': ['CLU', 'APOD']
}
all_markers = iFibro_extended_markers + [m for lst in confusion_markers.values() for m in lst]

fibroblast_subtype_markers = [
        'ACTA2', 'TAGLN', 'MYL9', 'PDGFRB',  # 最特异的肌成纤维细胞marker
        'CD74', 'HLA-DRA', 'HLA-DRB1', 'CIITA' , # 抗原呈递相关核心基因
        'COL1A1', 'COL3A1', 'LUM', 'DCN' , # ECM核心成分
        'CXCL12', 'IL6', 'CCL2', 'ICAM1',  # 免疫调节关键因子
        'CLU', 'APOD', 'CFD', 'PI16'  # 静息状态特征基因
    ]
# all_markers = list(set(DEG_inter +fibroblast_subtype_markers))
all_markers =list(set(['RGS5', 'ACTN2', 'C1QB', 'LYZ', 'C1QC', 'PDGFRA', 'TPM2', 'SPARC', 'HLA-DRB1', 'CRABP1', 'GREM1', 'CD34', 'SFRP4', 'CD40', 'FBLN1', 'DES', 'CD36', 'CXCL12', 'MYLK', 'CCL19', 'HLA-DRA', 'PI16', 'COL1A2', 'IL33', 'VCAN', 'MYL9', 'NFKB1', 'CCL21', 'HLA-DPA1', 'SPARCL1', 'CD80', 'LY6A', 'DCN', 'GSN', 'COL6A3', 'HLA-DQA1', 'TNC', 'COL3A1', 'APOD', 'MYH11', 'TNS1', 'NFKB2', 'BGN', 'FBLN2', 'CD86',
     'POSTN', 'CFD', 'ACTG2', 'GJA4', 'CXCL14', 'CIITA', 'LUM', 'SFRP2', 'FN1', 'CNN1', 'CCL2', 'ACTA2', 'COL6A2', 'IL6', 'ICAM1', 'C1QA', 'TPM1', 'ACTA1', 'ELN', 'TAGLN', 'COL6A1', 'MFAP4', 'ITGA8', 'HLA-DPB1', 'CDH11', 'APOE', 'VCAM1', 'MGP', 'PTGS2', 'NT5E', 'HLA-DQB1', 'CLU', 'PDGFRB', 'CD74', 'COL1A1', 'IL11', 'IGFBP6', 'FBN1', 'CRYAB', 'CIITA']+iFibro_extended_markers))


In [ ]:
all_markers = list(set(all_markers).union(set(gene_keep)))


In [ ]:
len(set(all_markers))


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.sparse import issparse

# 定义要检查的marker基因
genes_to_check = all_markers

def prepare_marker_data(adata, genes_list, celltype_col='Celltype', output_path='meta_data.csv'):
    """
    准备用于R可视化的marker基因数据
    
    参数:
    - adata: AnnData对象
    - genes_list: 要检查的基因列表
    - celltype_col: 细胞类型列名
    - output_path: 输出文件路径
    """
    
    # 检查基因是否存在
    available_genes = [gene for gene in genes_list if gene in adata.var_names]
    missing_genes = set(genes_list) - set(available_genes)
    
    if missing_genes:
        print(f"以下基因在数据中不存在: {missing_genes}")
    
    # 获取表达矩阵
    if issparse(adata.X):
        expr_matrix = adata[:, available_genes].X.toarray()
    else:
        expr_matrix = adata[:, available_genes].X
    
    # 创建数据框
    df = pd.DataFrame(expr_matrix, columns=available_genes)
    df[celltype_col] = adata.obs[celltype_col].values
    df.index = adata.obs_names
    
    # 计算平均表达值
    mean_expression = df.groupby(celltype_col)[available_genes].mean()
    
    # 计算平均表达百分比（表达该基因的细胞比例）
    # 对于单细胞数据，通常将表达量>0视为表达
    expr_binary = (df[available_genes] > 0).astype(int)
    expr_binary[celltype_col] = df[celltype_col]
    mean_percentage = expr_binary.groupby(celltype_col)[available_genes].mean() * 100
    
    # 重塑数据为长格式
    mean_exp_long = mean_expression.reset_index().melt(
        id_vars=celltype_col, 
        var_name='gene', 
        value_name='mean_expression'
    )
    
    mean_pct_long = mean_percentage.reset_index().melt(
        id_vars=celltype_col, 
        var_name='gene', 
        value_name='mean_percentage'
    )
    
    # 合并两个数据框
    result_df = pd.merge(mean_exp_long, mean_pct_long, on=[celltype_col, 'gene'])
    
    # 保存结果
    result_df.to_csv(output_path, index=False)
    
    print(f"数据已保存到: {output_path}")
    print(f"数据形状: {result_df.shape}")
    print(f"包含的细胞类型: {result_df[celltype_col].unique()}")
    print(f"包含的基因数量: {result_df['gene'].nunique()}")
    
    return result_df


In [ ]:
sorted(dir())


In [ ]:
# 使用函数处理数据
meta_data = prepare_marker_data(
    adata, 
    genes_to_check, 
    celltype_col='Celltype', 
    output_path='./data/scalex/fibro/human/celltype_gene_exp.csv'
)

# 可选：查看前几行数据
print(meta_data.head())


In [ ]:
# 使用函数处理数据
meta_data = prepare_marker_data(
    adata, 
    genes_to_check, 
    celltype_col='Celltype_general', 
    output_path='./data/scalex/fibro/human/Generalcelltype_gene_exp.csv'
)

# 可选：查看前几行数据
print(meta_data.head())



In [ ]:
# 读取或准备数据
# mydata <- your_data_frame

# 获取细胞类型和标注颜色
cell_types <- levels(factor(mydata$cell_type))
label_colors <- ifelse(grepl("_pan$", cell_types), "green", "red")

# 准备y轴标注数据
y_annotations <- data.frame(
  class = c("qFibro", "eFibro", "iFibro", "Myfibro"),
  y_pos = c(2.5, 7.5, 13.0, 17.5)
)

# 创建图形
p1 <- ggplot(mydata, aes(x = x_numeric, y = gene, size = mean_percentage, color = mean_expression)) + 
  geom_point() +
  
  # 添加垂直线
  geom_vline(xintercept = 7.5, linetype = "dashed", color = "grey40") + 
  geom_vline(xintercept = 24.5, linetype = "dashed", color = "grey40") +
  geom_vline(xintercept = 31.5, linetype = "dashed", color = "grey40") +  
  
  # 添加水平线
  geom_hline(yintercept = 4.5, linetype = "dashed", color = "grey40") +  
  geom_hline(yintercept = 9.5, linetype = "dashed", color = "grey40") + 
  geom_hline(yintercept = 15.5, linetype = "dashed", color = "grey40") +
  
  # 添加左侧标注（修改部分：文本旋转90度）
  geom_text(data = y_annotations,
            aes(x = 2, y = y_pos, label = class),
            color = "green", 
            fontface = "bold",
            size = 4,
            angle = 90,  # 旋转90度使文本竖直
            hjust = 0.5,  # 水平居中
            vjust = 1,    # 底部对齐
            inherit.aes = FALSE) +
  
  # 设置x轴刻度标签
  scale_x_continuous(
    breaks = 1:length(cell_types),
    labels = cell_types,
    expand = expansion(add = c(1, 0.5))  # 左侧增加更多空间给竖直标注
  ) +
  theme_classic() +
  scale_color_gradient(low = "#d6ecf0", high = "#f20c00") +
  
  # 设置主题
  theme(
    axis.text.x = element_text(
      angle = 45, 
      hjust = 1,
      color = label_colors,
      face = "bold",
      size = 10
    ),
    axis.text.y = element_text(
      face = "bold",
      size = 10
    ),
    axis.title.x = element_text(
      face = "bold",
      size = 12,
      margin = margin(t = 10)
    ),
    axis.title.y = element_text(
      face = "bold",
      size = 12,
      margin = margin(r = 10)
    )
    # , plot.margin = margin(10, 10, 10, 70)  # 进一步增加左侧边距，为竖直标注留出更多空间
  ) +
  labs(
    x = "Cell Type", 
    y = "Gene",
    size = "Mean exp(%)", 
    color = "Mean exp"
  )
print(p1)

In [ ]:
adata.obs[['GSE','GSM','sample','tissue']]


In [ ]:
(len(['with_0', 'with_0', 'with_0','with_0', 'without_0', 'without_0', 'without_0','without_0','with_0', 'with_0', 'with_0', 'with_0', 'without_0', 'without_0', 'without_0', 'without_0', 'with_0', 'with_0', 'with_0','with_0', 'without_0', 'without_0', 'without_0','without_0']), len(['Fibro', 'Fibro', 'Fibro', 'Fibro', 'Fibro', 'Fibro','Fibro', 'Fibro', 'Endo', 'Endo','Endo', 'Endo', 'Endo', 'Endo', 'Endo', 'Endo', 'Fibro+Endo', 'Fibro+Endo','Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo']), 

len(['RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)']))




In [ ]:
######################### Benchmark different age prediction model  ###################
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


model_bench = pd.DataFrame({
    'Model': ['RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)', 'RF', 'EN', 'PLSR','EN(CG)',],
    'factor': ['CT'] * 24,
    'Celltype': ['Fibro', 'Fibro', 'Fibro', 'Fibro', 'Fibro', 'Fibro','Fibro', 'Fibro', 'Endo', 'Endo','Endo', 'Endo', 'Endo', 'Endo', 'Endo', 'Endo', 'Fibro+Endo', 'Fibro+Endo','Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo', 'Fibro+Endo'],
    'Data': ['with_0', 'with_0', 'with_0','with_0', 'without_0', 'without_0', 'without_0','without_0','with_0', 'with_0', 'with_0', 'with_0', 'without_0', 'without_0', 'without_0', 'without_0', 'with_0', 'with_0', 'with_0','with_0', 'without_0', 'without_0', 'without_0','without_0'],
    'R2': [0.93, 0.7, 0.71, 0.85, 0.68, 0.69, 0.92, 0.51, 0.6, 0.76, 0.34, 0.49, 0.9, 0.7, 0.74, 0.93, 0.7, 0.75],
    'MAD': [2.58, 8.92, 8.7, 3.81, 8.73, 8.54, 2.76, 10.67, 9.45, 3.84, 10.09, 8.81, 3.96, 9.2, 8.41, 2.46, 9.2, 7.66],
    'PCC': [0.98, 0.84, 0.84, 0.94, 0.83, 0.83, 0.97, 0.72, 0.78, 0.92, 0.59, 0.7, 0.96, 0.84, 0.86, 0.97, 0.84, 0.86]
})



age_model_new = pd.DataFrame({
    'Model': ['EN', 'EN', 'EN'],
    'factor': ['CT+G', 'CT+G', 'CT+G'],
    'Celltype': ['Fibro', 'Endo', 'Fibro+Endo'],
    'Data': ['with_0', 'with_0', 'with_0'],
    'R2': [0.9, 0.98, 0.88],
    'MAD': [3.82, 1.28, 4.92],
    'PCC': [0.95, 0.99, 0.94]
})


# 1. 数据处理：将宽数据转为长数据
# 首先创建组合的x轴变量
model_bench['x_category'] = model_bench['Celltype'] + '|' + model_bench['Data']

# 定义x轴的顺序
x_order = ['Fibro|with_0', 'Fibro|without_0', 'Endo|with_0', 'Endo|without_0', 
           'Fibro+Endo|with_0', 'Fibro+Endo|without_0']

# 将指标列从宽格式转为长格式
metrics_long = pd.melt(
    model_bench, 
    id_vars=['Model', 'x_category'], 
    value_vars=['R2', 'MAD', 'PCC'],
    var_name='Metric', 
    value_name='Value'
)

# 设置分类变量的顺序
metrics_long['x_category'] = pd.Categorical(metrics_long['x_category'], categories=x_order, ordered=True)
metrics_long['Model'] = pd.Categorical(metrics_long['Model'], categories=['RF', 'EN', 'PLSR'], ordered=True)
metrics_long['Metric'] = pd.Categorical(metrics_long['Metric'], categories=['R2', 'MAD', 'PCC'], ordered=True)

print("处理后的数据前10行:")
print(metrics_long.head(10))


In [ ]:
metrics_long.shape ##### 


In [ ]:
# 方法2：使用matplotlib直接绘制（更精细控制）
def create_grouped_barplot(df, metric, ax, colors=['#2E86AB', '#A23B72', '#F18F01']):
    """创建分组柱状图"""
    
    # 设置柱状图参数
    n_models = len(df['Model'].unique())
    n_categories = len(df['x_category'].unique())
    bar_width = 0.25
    x_pos = np.arange(n_categories)
    
    # 为每个模型绘制柱状图
    for i, model in enumerate(['RF', 'EN', 'PLSR']):
        model_data = df[df['Model'] == model]
        values = [model_data[model_data['x_category'] == cat]['Value'].values[0] 
                 for cat in x_order]
        
        # 计算每个柱子的位置
        positions = x_pos + i * bar_width - (bar_width * (n_models - 1) / 2)
        
        # 绘制柱状图
        bars = ax.bar(positions, values, bar_width, 
                     label=model, color=colors[i], alpha=0.8,
                     edgecolor='black', linewidth=0.5)
        
        # 添加数值标签
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{value:.2f}', ha='center', va='bottom', fontsize=8)

    # 设置图表属性
    ax.set_xlabel('Celltype | Data', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric}', fontsize=13, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_order, rotation=45, ha='right')
    ax.legend(title='Model')
    ax.grid(axis='y', alpha=0.3)
    
    return ax

# 创建三个子图
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 为每个指标创建图表
for i, metric in enumerate(['R2', 'MAD', 'PCC']):
    metric_data = metrics_long[metrics_long['Metric'] == metric]
    create_grouped_barplot(metric_data, metric, axes[i])

plt.tight_layout()
plt.suptitle('Linear model benchmarking', 
             fontsize=16, fontweight='bold', y=1.02)
plt
plt.show()


In [ ]:
# 方法2：使用matplotlib直接绘制（更精细控制）
def create_grouped_barplot(df, metric, ax, colors=['#2E86AB', '#A23B72', '#F18F01']):
    """创建分组柱状图"""
    # 设置柱状图参数
    n_models = len(df['model'].unique())
    n_categories = len(df['x_category'].unique())
    bar_width = 0.25
    x_pos = np.arange(n_categories)
    # 为每个模型绘制柱状图
    for i, model in enumerate(['RF', 'EN', 'PLSR']):
        model_data = df[df['Model'] == model]
        values = [model_data[model_data['x_category'] == cat]['Value'].values[0] 
                 for cat in x_order]
        
        # 计算每个柱子的位置
        positions = x_pos + i * bar_width - (bar_width * (n_models - 1) / 2)
        
        # 绘制柱状图
        bars = ax.bar(positions, values, bar_width, 
                     label=model, color=colors[i], alpha=0.8,
                     edgecolor='black', linewidth=0.5)
        
        # 添加数值标签
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{value:.2f}', ha='center', va='bottom', fontsize=8)

    # 设置图表属性
    ax.set_xlabel('Celltype | Data', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric}', fontsize=13, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_order, rotation=45, ha='right')
    ax.legend(title='Model')
    ax.grid(axis='y', alpha=0.3)
    
    return ax

# 创建三个子图
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 为每个指标创建图表
for i, metric in enumerate(['R2', 'MAD', 'PCC']):
    metric_data = metrics_long[metrics_long['Metric'] == metric]
    create_grouped_barplot(metric_data, metric, axes[i])

plt.tight_layout()
plt.suptitle('Linear model benchmarking', 
             fontsize=16, fontweight='bold', y=1.02)
plt.show()


In [ ]:
# 方法4：使用plotly进行交互式可视化（可选）
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # 创建交互式图表
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['R2 Score', 'MAD Score', 'PCC Score'],
        shared_yaxes=False
    )
    
    colors = ['#2E86AB', '#A23B72', '#F18F01']
    
    for i, metric in enumerate(['R2', 'MAD', 'PCC'], 1):
        metric_data = metrics_long[metrics_long['Metric'] == metric]
        
        for j, model in enumerate(['RF', 'EN', 'PLSR']):
            model_metric_data = metric_data[metric_data['Model'] == model]
            
            fig.add_trace(
                go.Bar(
                    x=model_metric_data['x_category'],
                    y=model_metric_data['Value'],
                    name=model,
                    marker_color=colors[j],
                    legendgroup=model,
                    showlegend=(i == 1)  # 只在第一个子图显示图例
                ),
                row=1, col=i
            )
    
    fig.update_layout(
        title='Interactive Model Performance Comparison',
        height=500,
        showlegend=True,
        barmode='group'
    )
    
    fig.show()
    
except ImportError:
    print("Plotly未安装，跳过交互式可视化")
    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


age_model_new = pd.DataFrame({
    'Model': ['EN', 'EN', 'EN'],
    'factor': ['CT+G', 'CT+G', 'CT+G'],
    'Celltype': ['Fibro', 'Endo', 'Fibro+Endo'],
    'Data': ['with_0', 'with_0', 'with_0'],
    'R2': [0.9, 0.98, 0.88],
    'MAD': [3.82, 1.28, 4.92],
    'PCC': [0.95, 0.99, 0.94]
})

# 1. 数据预处理
print("age_model_new 数据:")
print(age_model_new)

# 将宽数据转为长数据
metrics_long = pd.melt(
    age_model_new, 
    id_vars=['Model', 'Celltype', 'Data'], 
    value_vars=['R2', 'MAD', 'PCC'],
    var_name='Metric', 
    value_name='Value'
)

# 设置分类顺序
metrics_long['Celltype'] = pd.Categorical(metrics_long['Celltype'], 
                                         categories=['Fibro', 'Endo', 'Fibro+Endo'], 
                                         ordered=True)
metrics_long['Metric'] = pd.Categorical(metrics_long['Metric'], 
                                       categories=['R2', 'MAD', 'PCC'], 
                                       ordered=True)

print("\n处理后的长数据:")
print(metrics_long)

In [ ]:
# 2. Seaborn 静态可视化
plt.figure(figsize=(6, 6))

# 设置颜色方案
celltype_colors = ['#2E86AB', '#A23B72', '#F18F01']  # Fibro, Endo, Fibro+Endo

# 创建分组柱状图
ax = sns.barplot(
    data=metrics_long,
    x='Metric',
    y='Value',
    hue='Celltype',
    palette=celltype_colors,
    saturation=0.8,
    edgecolor='black',
    linewidth=0.5
)

# 设置图表属性
plt.title('EN model(CT+Exp)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Performance Metrics', fontsize=12, fontweight='bold')
plt.ylabel('Score', fontsize=12, fontweight='bold')
plt.grid(axis='y', alpha=0.3, linestyle='--')

# 在每个柱子上添加数值标签
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10, fontweight='bold')

# 调整图例
plt.legend(title='CellType', title_fontsize=11, fontsize=10, 
           loc='upper right', frameon=True, fancybox=True, shadow=True)

# 调整布局
plt.tight_layout()
plt.show()

# 保存静态图片
# plt.savefig('en_model_celltype_performance.png', dpi=300, bbox_inches='tight')
# print("静态图表已保存为 'en_model_celltype_performance.png'")

In [ ]:
################################## EN model, Gene +CTProportion benchmarking and visualization ##################
FibEndo_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo/EN_ct_gene_FibEndo_with0_benchmarking_df.tsv',sep='\t')
FibEndo_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo/EN_ct_gene_FibEndo_without0_benchmarking_df.tsv',sep='\t')
Fibro_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/EN_ct_gene_Fibro_with0_benchmarking_df.tsv',sep='\t')
Fibro_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/EN_ct_gene_Fibro_without0_benchmarking_df.tsv',sep='\t')
Endo_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Endo/EN_ct_gene_Endo_with0_benchmarking_df.tsv',sep='\t')
Endo_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Endo/EN_ct_gene_Endo_without0_benchmarking_df.tsv',sep='\t')
evaluation_df_combined = pd.concat([FibEndo_with0_model_df, FibEndo_without0_model_df, Fibro_with0_model_df, Fibro_without0_model_df, Endo_with0_model_df, Endo_without0_model_df], ignore_index=True)




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 设置绘图风格和字体
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# 创建图形，调整长宽比（更长更窄）
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.suptitle('Model Performance Benchmarking by Celltype and Type', fontsize=16, fontweight='bold')

# 定义指标列表
metrics = ['R2', 'MAE', 'MAD', 'PCC']
metric_titles = ['R² Score', 'Mean Absolute Error (MAE)', 'Median Absolute Deviation (MAD)', 'Pearson Correlation Coefficient (PCC)']

# 定义nFeature的唯一值顺序
nFeature_order = [50, 100, 200, 500, 1000, 3000, 5000, 8000, 10000, 15000, 20000]

# 获取唯一的Celltype和Type
celltypes = evaluation_df_combined['Celltype'].unique()
type_values = evaluation_df_combined['Type'].unique()

# 使用更鲜明、对比度更好的颜色
colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFFF33', '#A65628', '#F781BF', '#999999']
# 为Type设置不同的透明度：with0更透明，without0更不透明
alphas = {'with0': 0.6, 'without0': 1.0}

# 绘制每个指标的子图
for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx//2, idx%2]
    
    # 设置x轴位置
    x_positions = np.arange(len(nFeature_order))
    bar_width = 0.8 / (len(celltypes) * len(type_values))
    
    # 存储图例元素
    celltype_handles = []
    celltype_labels = []
    type_handles = []
    type_labels = []
    
    # 为每个Celltype和Type组合绘制柱状图
    for i, celltype in enumerate(celltypes):
        for j, type_val in enumerate(type_values):
            # 筛选当前组合的数据
            subset = evaluation_df_combined[
                (evaluation_df_combined['Celltype'] == celltype) & 
                (evaluation_df_combined['Type'] == type_val)
            ]
            
            # 确保数据按照nFeature_order排序
            subset = subset.set_index('nFeature').reindex(nFeature_order).reset_index()
            
            # 计算柱状图位置
            pos = x_positions + (i * len(type_values) + j) * bar_width
            
            # 获取颜色和透明度
            color = colors[i % len(colors)]
            alpha = alphas[type_val]
            # 绘制柱状图
            bar = ax.bar(pos, subset[metric], width=bar_width, 
                        color=color, alpha=alpha, edgecolor='black', linewidth=0.5)
            
            # 只在第一个子图记录图例信息
            if idx == 0:
                # Celltype图例（只记录一次）
                if j == 0:
                    celltype_handles.append(plt.Rectangle((0,0),1,1, fc=color, alpha=1.0, edgecolor='black'))
                    celltype_labels.append(celltype)
                
                # Type图例（只记录一次）
                if i == 0:
                    type_handles.append(plt.Rectangle((0,0),1,1, fc='gray', alpha=alpha, edgecolor='black'))
                    type_labels.append(type_val)
    
    # 设置子图属性
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('nFeature', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    
    # 设置x轴刻度和标签
    ax.set_xticks(x_positions + bar_width * (len(celltypes) * len(type_values) - 1) / 2)
    ax.set_xticklabels([f'{x:,}' for x in nFeature_order], rotation=45, ha='right')
    
    # 添加网格
    ax.grid(True, alpha=0.3, axis='y')
    
    # 根据指标类型调整y轴范围
    if metric in ['R2', 'PCC']:
        ax.set_ylim(bottom=0)  # R2和PCC从0开始
    # 对于MAE和MAD，让matplotlib自动调整范围
    
    # 只在第一个子图添加图例
    if idx == 1:
        # 创建两个图例
        legend1 = ax.legend(celltype_handles, celltype_labels, 
                           title='Celltype', loc='upper right',
                           bbox_to_anchor=(0.8, 1), borderaxespad=0.)
        legend2 = ax.legend(type_handles, type_labels, 
                           title='Type', loc='upper right',
                           bbox_to_anchor=(0.8, 0.7), borderaxespad=0.)
        
        # 将两个图例都添加到图中
        ax.add_artist(legend1)
        ax.add_artist(legend2)

plt.tight_layout()
plt.subplots_adjust(top=0.92, right=0.85)  # 为图例留出空间

# 保存图像
#plt.savefig('Benchmark.png', dpi=300, bbox_inches='tight', facecolor='white')
#plt.savefig('Benchmark.pdf', bbox_inches='tight', facecolor='white')

plt.show()

# 打印颜色和透明度映射信息
print("Color mapping for Celltypes:")
for i, celltype in enumerate(celltypes):
    print(f"  {celltype}: {colors[i % len(colors)]}")

print("\nAlpha mapping for Types:")
for type_val, alpha in alphas.items():
    print(f"  {type_val}: alpha={alpha}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.suptitle('Model Performance Benchmarking by Celltype and Type', fontsize=16, fontweight='bold')

# 定义指标列表
metrics = ['R2', 'MAE', 'MAD', 'PCC']
metric_titles = ['R² Score', 'Mean Absolute Error (MAE)', 'Median Absolute Deviation (MAD)', 'Pearson Correlation Coefficient (PCC)']

# 定义nFeature的唯一值顺序
nFeature_order = [50, 100, 200, 500, 1000, 3000, 5000, 8000, 10000, 15000, 20000]

# 获取唯一的Celltype和Type
celltypes = evaluation_df_combined['Celltype'].unique()
type_values = evaluation_df_combined['Type'].unique()

# 使用鲜明的高对比度颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
alphas = {'with0': 0.6, 'without0': 1.0}

# 绘制每个指标的子图
for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx//2, idx%2]
    
    x_positions = np.arange(len(nFeature_order))
    bar_width = 0.8 / (len(celltypes) * len(type_values))
    
    for i, celltype in enumerate(celltypes):
        for j, type_val in enumerate(type_values):
            subset = evaluation_df_combined[
                (evaluation_df_combined['Celltype'] == celltype) & 
                (evaluation_df_combined['Type'] == type_val)
            ]
            subset = subset.set_index('nFeature').reindex(nFeature_order).reset_index()
            
            pos = x_positions + (i * len(type_values) + j) * bar_width
            color = colors[i % len(colors)]
            alpha = alphas[type_val]
            
            ax.bar(pos, subset[metric], width=bar_width, 
                  color=color, alpha=alpha, edgecolor='black', linewidth=0.5)
    
    # 设置子图属性
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('nFeature')
    ax.set_ylabel(metric)
    ax.set_xticks(x_positions + bar_width * (len(celltypes) * len(type_values) - 1) / 2)
    ax.set_xticklabels([f'{x:,}' for x in nFeature_order], rotation=45, ha='right')
    ax.grid(True, alpha=0.3, axis='y')

# 创建图例
celltype_handles = [plt.Rectangle((0,0),1,1, fc=colors[i], alpha=1.0, edgecolor='black') 
                   for i in range(len(celltypes))]
type_handles = [plt.Rectangle((0,0),1,1, fc='gray', alpha=alpha, edgecolor='black') 
               for alpha in alphas.values()]

# 在主图上添加图例
fig.legend(celltype_handles, celltypes, 
          title='Celltype', loc='upper right', 
          bbox_to_anchor=(0.98, 0.9))
fig.legend(type_handles, alphas.keys(), 
          title='Type', loc='upper right', 
          bbox_to_anchor=(0.98, 0.8))

plt.tight_layout()
plt.subplots_adjust(top=0.92)

# 保存图像
#plt.savefig('Benchmark.png', dpi=300, bbox_inches='tight', facecolor='white')
#plt.savefig('Benchmark.pdf', bbox_inches='tight', facecolor='white')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 设置绘图风格和字体大小
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12  # 增大基础字体大小

# 创建图形
fig, axes = plt.subplots(2, 2, figsize=(25, 12))  # 进一步调整尺寸
fig.suptitle('Model Performance Benchmarking by Celltype and Type', 
             fontsize=20, fontweight='bold', y=0.98)  # 增大标题字体

# 定义指标列表
metrics = ['R2', 'MAE', 'MAD', 'PCC']
metric_titles = ['R² Score', 'Mean Absolute Error (MAE)', 
                 'Median Absolute Deviation (MAD)', 'Pearson Correlation Coefficient (PCC)']

# 定义nFeature的唯一值顺序
nFeature_order = [50, 100, 200, 500, 1000, 3000, 5000, 8000, 10000, 15000, 20000]

# 获取唯一的Celltype和Type
celltypes = evaluation_df_combined['Celltype'].unique()
type_values = evaluation_df_combined['Type'].unique()

# 使用鲜明的高对比度颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
alphas = {'with0': 0.6, 'without0': 1.0}

# 绘制每个指标的子图
for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx//2, idx%2]
    
    x_positions = np.arange(len(nFeature_order))
    bar_width = 0.8 / (len(celltypes) * len(type_values))
    
    for i, celltype in enumerate(celltypes):
        for j, type_val in enumerate(type_values):
            # 筛选当前组合的数据
            subset = evaluation_df_combined[
                (evaluation_df_combined['Celltype'] == celltype) & 
                (evaluation_df_combined['Type'] == type_val)
            ]
            subset = subset.set_index('nFeature').reindex(nFeature_order).reset_index()
            
            pos = x_positions + (i * len(type_values) + j) * bar_width
            color = colors[i % len(colors)]
            alpha = alphas[type_val]
            
            # 绘制柱状图
            bars = ax.bar(pos, subset[metric], width=bar_width, 
                         color=color, alpha=alpha, edgecolor='black', linewidth=0.8)
            
            # 在柱子上添加数值标签
            for k, (x, y) in enumerate(zip(pos, subset[metric])):
                if not np.isnan(y):  # 只对非NaN值添加标签
                    # 根据数值大小调整标签位置和颜色
                    if metric in ['R2', 'PCC']:
                        # R2和PCC值较大，在柱子内部顶部显示
                        va = 'top'
                        label_y = y * 0.95
                        color_text = 'white' if y > 0.5 else 'black'
                    else:
                        # MAE和MAD值较小，在柱子外部顶部显示
                        va = 'bottom'
                        label_y = y + 0.02 * np.max(subset[metric])
                        color_text = 'black'
                    
                    ax.text(x, label_y, f'{y:.3f}', 
                           ha='center', va=va, fontsize=9, fontweight='bold',
                           color=color_text, rotation=90 if len(nFeature_order) > 8 else 0)
    
    # 设置子图属性 - 增大所有文字元素
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)  # 增大子图标题
    ax.set_xlabel('nFeature', fontsize=14, fontweight='bold', labelpad=10)  # 增大x轴标签
    ax.set_ylabel(metric, fontsize=14, fontweight='bold', labelpad=10)  # 增大y轴标签
    
    # 设置x轴刻度和标签 - 增大坐标轴文字
    ax.set_xticks(x_positions + bar_width * (len(celltypes) * len(type_values) - 1) / 2)
    ax.set_xticklabels([f'{x:,}' for x in nFeature_order], 
                      rotation=45, ha='right', fontsize=12)  # 增大x轴刻度标签
    
    # 设置y轴刻度标签大小
    ax.tick_params(axis='y', labelsize=12)
    
    # 添加网格
    ax.grid(True, alpha=0.3, axis='y')
    
    # 根据指标类型调整y轴范围，为数值标签留出空间
    if metric in ['R2', 'PCC']:
        ax.set_ylim(bottom=0, top=1.1)  # 为顶部标签留出空间
    else:
        # 对于MAE和MAD，自动调整上限
        y_max = ax.get_ylim()[1]
        ax.set_ylim(bottom=0, top=y_max * 1.15)  # 为顶部标签留出空间

# 创建图例 - 增大图例文字
celltype_handles = [plt.Rectangle((0,0),1,1, fc=colors[i], alpha=1.0, edgecolor='black') 
                   for i in range(len(celltypes))]
type_handles = [plt.Rectangle((0,0),1,1, fc='gray', alpha=alpha, edgecolor='black') 
               for alpha in alphas.values()]

# 在主图上添加图例 - 增大图例字体
legend1 = fig.legend(celltype_handles, celltypes, 
                   title='Celltype', loc='upper right', 
                   bbox_to_anchor=(0.98, 0.95),
                   fontsize=13, title_fontsize=14)  # 增大图例字体

legend2 = fig.legend(type_handles, alphas.keys(), 
                   title='Type', loc='upper right', 
                   bbox_to_anchor=(0.98, 0.78),
                   fontsize=13, title_fontsize=14)  # 增大图例字体

plt.tight_layout()
plt.subplots_adjust(top=0.92, right=0.88)  # 为图例留出更多空间

# 保存图像
#plt.savefig('Benchmark.png', dpi=300, bbox_inches='tight', facecolor='white')
#plt.savefig('Benchmark.pdf', bbox_inches='tight', facecolor='white')

plt.show()
print("可视化完成！图像已保存为 'Benchmark.png' 和 'Benchmark.pdf'")



In [ ]:
################# Check the cell number and  
adata_summary_df_fibonly_deg = pd.read_csv('/fse/home/wangyuting/Project/TabulaStromal/Fibroblasts/LLM_cellfiltering/All3Round_Fibro_human_FibOnlyDEG_count_notAge.tsv',sep='\t')
adata_summary_df_fiblike_deg = pd.read_csv('/fse/home/wangyuting/Project/TabulaStromal/Fibroblasts/LLM_cellfiltering/All3Round_Fibro_human_FibLikeDEG_count_notAge.tsv',sep='\t')



In [ ]:
adata_summary_df_fibonly_deg = adata_summary_df_fibonly_deg[adata_summary_df_fibonly_deg['cell_num']>=50]
adata_summary_df_fiblike_deg = adata_summary_df_fiblike_deg[adata_summary_df_fiblike_deg['cell_num']>=50]


In [ ]:
icb_keywords = {
    # 主要治疗类型
    "immune_checkpoint_blockade": [
        "immune checkpoint blockade", "ICB", "checkpoint blockade",
        "immune checkpoint inhibitor", "checkpoint inhibitor therapy"
    ],
    # PD-1/PD-L1轴
    "pd1_pdl1": [
        "PD-1", "programmed cell death protein 1", "CD279",
        "PD-L1", "programmed death ligand 1", "CD274", "B7-H1",
        "PD-L2", "programmed death ligand 2", "CD273", "B7-DC",
        "anti-PD-1", "anti-PD-L1", "PD-1 blockade", "PD-L1 blockade",
        "nivolumab", "Opdivo", "pembrolizumab", "Keytruda",
        "atezolizumab", "Tecentriq", "durvalumab", "Imfinzi",
        "avelumab", "Bavencio", "cemiplimab", "Libtayo"
    ],
    # CTLA-4轴
    "ctla4": [
        "CTLA-4", "cytotoxic T-lymphocyte-associated protein 4", "CD152",
        "anti-CTLA-4", "CTLA-4 blockade",
        "ipilimumab", "Yervoy", "tremelimumab", "Imjudo"
    ],
    # 其他检查点分子
    "other_checkpoints": [
        "LAG-3", "lymphocyte activation gene 3", "CD223",
        "TIM-3", "T cell immunoglobulin mucin 3", "HAVCR2",
        "TIGIT", "T cell immunoreceptor with Ig and ITIM domains",
        "VISTA", "V-domain Ig suppressor of T cell activation",
        "B7-H3", "CD276", "B7-H4", "VTCN1",
        "IDO1", "indoleamine 2,3-dioxygenase 1",
        "KIR", "killer cell immunoglobulin-like receptor",
        "CD96", "Tactile", "CD112R", "PVRIG"
    ],
    # 联合疗法
    "combination_therapy": [
        "combination immunotherapy", "dual checkpoint blockade",
        "PD-1/CTLA-4 combination", "nivolumab+ipilimumab",
        "pembrolizumab+ipilimumab", "ICB combination",
        "ICB plus chemotherapy", "ICB plus targeted therapy",
        "ICB plus radiotherapy"
    ],
    # 治疗响应相关
    "therapy_response": [
        "responder", "non-responder", "response to ICB",
        "durable clinical benefit", "DCB",
        "progressive disease", "PD", "stable disease", "SD",
        "partial response", "PR", "complete response", "CR",
        "objective response rate", "ORR",
        "progression-free survival", "PFS",
        "overall survival", "OS",
        "hyperprogression", "acquired resistance",
        "primary resistance", "adaptive resistance"
    ],
    # 生物标志物
    "biomarkers": [
        "tumor mutational burden", "TMB",
        "microsatellite instability", "MSI", "MSI-H",
        "PD-L1 expression", "tumor infiltrating lymphocytes", "TILs",
        "CD8+ T cells", "cytotoxic T cells",
        "T cell exhaustion", "exhausted T cells",
        "T cell dysfunction", "dysfunctional T cells",
        "immune signature", "immunophenotype",
        "immune score", "immunoscore"
    ]
}

# 创建扁平化的关键词列表便于搜索
all_icb_keywords = []
for category in icb_keywords.values():
    all_icb_keywords.extend(category)

# 可选：创建反向映射（关键词到类别）
keyword_to_category = {}
for category, keywords in icb_keywords.items():
    for keyword in keywords:
        keyword_to_category[keyword.lower()] = category

In [ ]:
fibroblast_subtype_markers = {
    'Myfibro': [
        'ACTA2', 'TAGLN', 'MYL9', 'PDGFRB'  # 最特异的肌成纤维细胞marker
    ],
  #  'apFibro': [
  #      'CD74', 'HLA-DRA', 'HLA-DRB1', 'CIITA'  # 抗原呈递相关核心基因
  #  ],
    'eFibro': [
        'COL1A1', 'COL3A1', 'LUM', 'DCN'  # ECM核心成分
    ],
    'iFibro': [
        'CXCL12','CD74', 'CCL2', 'ICAM1'  # 免疫调节关键因子
    ],
    'qFibro': [
        'CLU', 'APOD', 'CFD', 'PI16'  # 静息状态特征基因
    ]
}
fibroblast_subtype_markers_full = {
    'Myfibro1': [
        'ACTA2', 'TAGLN', 'MYL9', 'MYH11','PDGFRB',
        'CNN1', 'ACTG2','MYLK','DES','TPM1','TPM2'    # added
    ],
    'eFibro1': [
        'COL1A1','LUM', 'DCN', 'COL3A1', 
        'COL6A1', 'COL6A2', 'COL6A3','FN1',  'BGN', 'COL1A2', 'SPARC', 'POSTN', 'TNC', 'VCAN'     # added
    ],
    'iFibro1': [
        'CXCL12','CD74', 'CCL2', 'ICAM1'  ,  
        'IL6',   'HLA-DRA', 'HLA-DRB1','CIITA', 'CCL19', 'CCL21', 'CXCL14','IL11', 'PTGS2'  # added
    ],
    'qFibro1': [
        'CLU', 'APOD', 'CFD', 'PI16', 
        'CRABP1', 'FBLN1','FBLN2', 'MFAP4', 'GSN', 'SPARCL1', 'MGP', 'IGFBP6'   # added
    ]
}


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype', 'Myfibro1_4raw', 'iFibro1_4raw',
                        'eFibro1_4raw','qFibro1_4raw', 'Myfibro_4raw', 'iFibro_4raw', 
                        'eFibro_4raw', 'qFibro_4raw']].copy()
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)
# 定义leiden到Celltype的映射字典

# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['Myfibro1_4raw', 'iFibro1_4raw', 'eFibro1_4raw', 'qFibro1_4raw']
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows = 4
n_cols = 10
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors = plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5*n_cols, 2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of subFibro(long list)', 
             fontsize=12, fontweight='bold', y=1.02)

# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=0
        )
    
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=8, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度
    ax.set_xticks([])
    
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Signature Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 根据Celltype前缀添加背景色区分
    celltype_prefix = get_celltype_prefix(celltype)
    if celltype_prefix == 1:  # apFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 2:  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 3:  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    elif celltype_prefix == 4:  # qFibro
        ax.set_facecolor('#E6FFE6')  # 浅绿色背景

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
feature_short_names = []
for col in available_columns:
    if 'Myfibro1_' in col:
        feature_short_names.append('Myfibro')
        # feature_short_names.append('Myfibro_l')
    elif 'iFibro1_' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro1_' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro1_' in col:
        feature_short_names.append('qFibro')
    elif 'Myfibro_' in col:
        feature_short_names.append('Myfibro')
    elif 'iFibro_' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro_' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro_' in col:
        feature_short_names.append('qFibro')
    else:
        feature_short_names.append(col.replace('_4raw', ''))

# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]


# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('apFibro', '#FFF5E6'),
    2: ('eFibro', '#E6F7FF'),
    3: ('Myfibro', '#F0E6FF'),
    4: ('qFibro', '#E6FFE6')
}

# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []

for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.75, 0.01),  # 底部右侧
           ncol=4,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.1,left=0.05, right=0.9, hspace=0.4, wspace=0.3)

# 12. 保存图形
output_path = './code/leiden_subtype_Long_scores.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"图形已保存到: {output_path}")

# 13. 显示图形
plt.show()




In [ ]:
Fib_meta_data = pd.read_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_meta.tsv',sep='\t')


In [ ]:

age_gsm_df = Fib_meta_data[["Age_group", "GSM"]].copy()
age_gsm_df.rename(columns={'Age_group': 'age_group'}, inplace=True)

# 统计细胞数和原始GSM数
age_cell_count = age_gsm_df.groupby("age_group").size().reset_index(name="cell_count")
age_gsm_count = age_gsm_df.groupby("age_group")["GSM"].nunique().reset_index(name="gsm_count")

# 合并数据并处理
age_stats = pd.merge(age_cell_count, age_gsm_count, on="age_group", how="outer")
age_stats = age_stats.dropna(subset=["cell_count", "gsm_count"]).reset_index(drop=True)
age_stats["cell_count"] = age_stats["cell_count"].astype(int)
age_stats["gsm_count"] = age_stats["gsm_count"].astype(int)

# 设置age_group顺序（与实际分组一致）
custom_age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']  # H 只有1个样本，143 cells，去掉
age_stats = age_stats[age_stats["age_group"].isin(custom_age_order)].copy()
age_stats["age_group"] = pd.Categorical(age_stats["age_group"], categories=custom_age_order, ordered=True)
age_stats = age_stats.sort_values("age_group").reset_index(drop=True)

# --------------------------
# 核心1：计算放大500倍的GSM负向高度（柱子高度=实际GSM数×500，负向显示）
# --------------------------
zoom_factor = 500  # 固定放大500倍
age_stats["gsm_count_neg_zoom"] = -age_stats["gsm_count"] * zoom_factor  # 放大后的负向高度

# 打印统计结果（确认实际GSM数与放大后高度）
print("实际GSM数与放大后柱子高度：")
print(age_stats[["age_group", "gsm_count", "gsm_count_neg_zoom"]])
print(f"\n放大倍数：{zoom_factor}倍（GSM柱子高度=实际GSM数×{zoom_factor}）")


# --------------------------
# 步骤2：绘制柱状图（核心2：自定义y轴负向刻度标签）
# --------------------------
if not age_stats.empty:
    plt.figure(figsize=(8.8, 5))
    # 定义颜色
    cell_color = '#1F77B4'  # colors = ['#1F77B4', '#FF7F0E']
    gsm_color = '#FF7F0E'
    # 1. 绘制细胞数（向上，原始高度，不变）
    bars_cell = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["cell_count"],
        color=cell_color,
        alpha=0.8,
        label="Cell Number"
    )
    # 2. 绘制GSM数（向下，放大500倍的高度）
    bars_gsm = plt.bar(
        x=age_stats["age_group"],
        height=age_stats["gsm_count_neg_zoom"],  # 用放大后的高度
        color=gsm_color,
        alpha=0.8,
        label="Sample Number"
    )
    ax = plt.gca()  # 获取当前坐标轴对象
    y_ticks = ax.get_yticks()  # 所有y轴刻度值（如[-6500, -3250, 0, 29027, 58055, ...]）
    # 自定义刻度标签：负向刻度÷-500（还原为实际GSM数），正向刻度保持不变
    custom_y_labels = []
    for tick in y_ticks:
        if tick < 0:  # 负向刻度（对应GSM）：刻度值 ÷ (-zoom_factor) = 实际GSM数
            actual_gsm = int(tick / (-zoom_factor))  # 如-6500 ÷ -500 = 13
            custom_y_labels.append(str(actual_gsm))  # 显示实际GSM数（如"13"）
        elif tick == 0:  # 0刻度：保持"0"
            custom_y_labels.append("0")
        else:  # 正向刻度（对应细胞数）：保持原始数值（可加千分位）
            custom_y_labels.append(f"{int(tick):,}")  # 如"29,027"
    # 应用自定义刻度标签（关键！让y轴负向显示实际GSM数）
    ax.set_yticklabels(custom_y_labels, fontsize=9)
    # --------------------------
    # 标注与美化（标注均显示实际数量，无放大）
    # --------------------------
    # 细胞数标注（显示实际数量，千分位）
    for bar, count in zip(bars_cell, age_stats["cell_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + age_stats["cell_count"].max() * 0.01,
            s=f"{count:,}",
            ha="center", va="bottom",
            fontsize=9,
            fontweight="bold"
        )
    # GSM标注（显示实际数量，无需放大）
    for bar, count in zip(bars_gsm, age_stats["gsm_count"]):
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() - (age_stats["gsm_count"].max() * zoom_factor) * 0.02,  # 基于放大后高度的偏移
            s=f"{count}",  # 显示实际GSM数（如13）
            ha="center", va="top",
            fontsize=9,
            fontweight="bold",
            color=gsm_color
        )
    plt.xlabel("Age Group", fontsize=12, labelpad=10)
    plt.ylabel("Count", 
               fontsize=11, labelpad=10)
    plt.title(f"Endo distribution of AgeGroup", 
              fontsize=14, pad=20)
    # 图例和0刻度线（加粗0刻度，区分上下区域）
    plt.legend(loc="upper center", fontsize=10, bbox_to_anchor=(0.5, -0.08), ncol=2)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=2)
    # 调整布局，避免标签截断
    plt.tight_layout
    plt.savefig('external_inputs/Endo_cellsample_agegroup_1.png', dpi=300, bbox_inches='tight')
    plt.savefig('external_inputs/Endo_cellsample_agegroup_1.pdf', format='pdf', bbox_inches='tight')
    plt.show()
else:
    print("无可用数据用于绘图，请检查custom_age_order是否与age_group匹配！")
    

In [ ]:
######################## Next try to benchmark the EN(Gene+CT) model, and do gene & functional analysis 
####### Gene( KEGG +GO), functional (pathway +GWAS)


In [ ]:
FibEndo_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo/EN_ct_gene_FibEndo_with0_benchmarking_df.tsv',sep='\t')
FibEndo_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo/EN_ct_gene_FibEndo_without0_benchmarking_df.tsv',sep='\t')
FibEndoInte_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo_inte/EN_ct_gene_FibEndoInte_without0_benchmarking_df.tsv',sep='\t')
FibEndoInte_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo_inte/EN_ct_gene_FibEndoInte_with0_benchmarking_df.tsv',sep='\t')
FibEndoInte_with0_model_df['Celltype'] = 'FibEndoInte'
FibEndoInte_without0_model_df['Celltype'] = 'FibEndoInte'
Fibro_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/EN_ct_gene_Fibro_with0_benchmarking_df.tsv',sep='\t')
Fibro_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/EN_ct_gene_Fibro_without0_benchmarking_df.tsv',sep='\t')
Endo_with0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Endo/EN_ct_gene_Endo_with0_benchmarking_df.tsv',sep='\t')
Endo_without0_model_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Endo/EN_ct_gene_Endo_without0_benchmarking_df.tsv',sep='\t')
evaluation_df_combined = pd.concat([FibEndo_with0_model_df, FibEndo_without0_model_df,FibEndoInte_with0_model_df,FibEndoInte_without0_model_df, Fibro_with0_model_df, Fibro_without0_model_df, Endo_with0_model_df, Endo_without0_model_df], ignore_index=True)
evaluation_df_combined
evaluation_df_combined['Type'] = evaluation_df_combined['Type'].replace({
    'with0': 'A',
    'without0': 'noA'
})


In [ ]:
evaluation_df_combined


In [ ]:
os.getcwd()

In [ ]:

# 设置绘图风格和字体大小
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12  # 增大基础字体大小

fig, axes = plt.subplots(2, 2, figsize=(30, 14))  # 进一步调整尺寸
fig.suptitle('Model Performance Benchmarking by Celltype and Type', 
             fontsize=20, fontweight='bold', y=0.98)  # 增大标题字体
metrics = ['R2', 'MAE', 'MAD', 'PCC']
metric_titles = ['R² Score','Mean Absolute Error(MAE)', 
                 'Median Absolute Deviation (MAD)', 'Pearson Correlation Coefficient (PCC)']
nFeature_order = [50, 100, 200, 500, 1000, 3000, 5000, 8000, 10000, 15000, 20000]
celltypes = evaluation_df_combined['Celltype'].unique()
type_values = evaluation_df_combined['Type'].unique()
# 使用鲜明的高对比度颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
alphas = {'A': 0.4, 'noA': 1.0}

# 绘制每个指标的子图
for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx//2, idx%2]
    x_positions = np.arange(len(nFeature_order))
    bar_width = 0.8 / (len(celltypes) * len(type_values))
    for i, celltype in enumerate(celltypes):
        for j, type_val in enumerate(type_values):
            # 筛选当前组合的数据
            subset = evaluation_df_combined[
                (evaluation_df_combined['Celltype'] == celltype) & 
                (evaluation_df_combined['Type'] == type_val)
            ]
            subset = subset.set_index('nFeature').reindex(nFeature_order).reset_index()
            pos = x_positions + (i * len(type_values) + j) * bar_width
            color = colors[i % len(colors)]
            alpha = alphas[type_val]
            # 绘制柱状图
            bars = ax.bar(pos, subset[metric], width=bar_width, 
                         color=color, alpha=alpha, edgecolor='black', linewidth=0.8)
            # 在柱子上添加数值标签
            for k, (x, y) in enumerate(zip(pos, subset[metric])):
                if not np.isnan(y):  # 只对非NaN值添加标签
                    # 根据数值大小调整标签位置和颜色
                    if metric in ['R2', 'PCC']:
                        # R2和PCC值较大，在柱子内部顶部显示
                        va = 'top'
                        label_y = y * 0.95
                        color_text = 'white' if y > 0.5 else 'black'
                    else:
                        # MAE和MAD值较小，在柱子外部顶部显示
                        va = 'bottom'
                        label_y = y + 0.02 * np.max(subset[metric])
                        color_text = 'black'
                    ax.text(x, label_y, f'{y:.3f}', 
                           ha='center', va=va, fontsize=9, fontweight='bold',
                           color=color_text, rotation=90 if len(nFeature_order) > 8 else 0)
    
    # 设置子图属性 - 增大所有文字元素
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)  # 增大子图标题
    ax.set_xlabel('nFeature', fontsize=14, fontweight='bold', labelpad=10)  # 增大x轴标签
    ax.set_ylabel(metric, fontsize=14, fontweight='bold', labelpad=10)  # 增大y轴标签
    # 设置x轴刻度和标签 - 增大坐标轴文字
    ax.set_xticks(x_positions + bar_width * (len(celltypes) * len(type_values) - 1) / 2)
    ax.set_xticklabels([f'{x:,}' for x in nFeature_order], 
                      rotation=45, ha='right', fontsize=12)  # 增大x轴刻度标签
    # 设置y轴刻度标签大小
    ax.tick_params(axis='y', labelsize=12)
    # 添加网格
    ax.grid(True, alpha=0.3, axis='y')
    # 根据指标类型调整y轴范围，为数值标签留出空间
    if metric in ['R2', 'PCC']:
        ax.set_ylim(bottom=0, top=1.1)  # 为顶部标签留出空间
    else:
        # 对于MAE和MAD，自动调整上限
        y_max = ax.get_ylim()[1]
        ax.set_ylim(bottom=0, top=y_max * 1.15)  # 为顶部标签留出空间

# 创建图例 - 增大图例文字
celltype_handles = [plt.Rectangle((0,0),1,1, fc=colors[i], alpha=1.0, edgecolor='black') 
                   for i in range(len(celltypes))]
type_handles = [plt.Rectangle((0,0),1,1, fc='gray', alpha=alpha, edgecolor='black') 
               for alpha in alphas.values()]
# 在主图上添加图例 - 增大图例字体
# legend1 = fig.legend(celltype_handles, celltypes, 
#                    title='Celltype', loc='lower left', 
#                    bbox_to_anchor=(0.98, 0.95),
#                    fontsize=13, title_fontsize=14)  # 增大图例字体
# legend2 = fig.legend(type_handles, alphas.keys(), 
#                    title='Type', loc='lower right', 
#                    bbox_to_anchor=(0.98, 0.78),
#                    fontsize=13, title_fontsize=14)  # 增大图例字体

# Subtypes图例（左下区域）
fig.legend(celltype_handles, celltypes, title='Celltype',
           loc='lower center', 
           bbox_to_anchor=(0.5, 0.1),  # 底部中间
           ncol=len(celltype_handles),  # 水平排列
           fontsize=9,
           title_fontsize=13,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(type_handles, alphas.keys(),title='Type',
           loc='lower center',
           bbox_to_anchor=(0.75, 0.1),  # 底部右侧
           ncol=2,  # 水平排列
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.92, right=0.88)  # 为图例留出更多空间
# 保存图像
fig_path ='./data/scalex/fibro/human/age_prediction/EN_Exp_CT/small_file/'
plt.savefig(fig_path+'EN_CT_gene_benchmark.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig_path+'EN_CT_gene_benchmark.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print("可视化完成！图像已保存为 'Benchmark.png' 和 'Benchmark.pdf'")



In [ ]:

################ 下面的code是修改过后不需要 'Type' legend的具体的 Python 代码 
# 设置绘图风格和字体大小
type ='A'
evaluation_df_combined_sub = evaluation_df_combined[evaluation_df_combined['Type']==type]

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12  # 增大基础字体大小
fig, axes = plt.subplots(2, 2, figsize=(30, 14))  # 进一步调整尺寸
fig.suptitle('Elastic net[Gene+CT] benchmarking('+type+')', 
             fontsize=20, fontweight='bold', y=0.98)  # 增大标题字体，更新标题
metrics = ['R2','PCC', 'MAE', 'MAD']
metric_titles = ['R² Score', 'Pearson Correlation Coefficient (PCC)','Mean Absolute Error(MAE)', 
                 'Median Absolute Deviation (MAD)']
nFeature_order = [50, 100, 200, 500, 1000, 3000, 5000, 8000, 10000, 15000, 20000]
celltypes = evaluation_df_combined_sub['Celltype'].unique()
# 使用鲜明的高对比度颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']

# 绘制每个指标的子图
for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx//2, idx%2]
    x_positions = np.arange(len(nFeature_order))
    bar_width = 0.8 / len(celltypes)  # 调整bar宽度，因为现在只有一种Type
    for i, celltype in enumerate(celltypes):
        # 筛选当前celltype的数据
        subset = evaluation_df_combined_sub[
            (evaluation_df_combined_sub['Celltype'] == celltype)
        ]
        subset = subset.set_index('nFeature').reindex(nFeature_order).reset_index()
        pos = x_positions + i * bar_width  # 调整位置计算
        
        # 绘制柱状图，不使用alpha透明度
        bars = ax.bar(pos, subset[metric], width=bar_width, 
                     color=colors[i % len(colors)], alpha=1.0, edgecolor='black', linewidth=0.8)
        # 在柱子上添加数值标签
        for k, (x, y) in enumerate(zip(pos, subset[metric])):
            if not np.isnan(y):  # 只对非NaN值添加标签
                # 根据数值大小调整标签位置和颜色
                if metric in ['R2', 'PCC']:
                    # R2和PCC值较大，在柱子内部顶部显示
                    va = 'top'
                    label_y = y * 0.95
                    color_text = 'white' if y > 0.5 else 'black'
                else:
                    # MAE和MAD值较小，在柱子外部顶部显示
                    va = 'bottom'
                    label_y = y + 0.02 * np.max(subset[metric])
                    color_text = 'black'
                ax.text(x, label_y, f'{y:.3f}', 
                       ha='center', va=va, fontsize=9, fontweight='bold',
                       color=color_text, rotation=90 if len(nFeature_order) > 8 else 0)
    
    # 设置子图属性 - 增大所有文字元素
    ax.set_title(title, fontsize=16, fontweight='bold', pad=16)  # 增大子图标题
    ax.set_xlabel('nFeature', fontsize=15, fontweight='bold', labelpad=10)  # 增大x轴标签
    ax.set_ylabel(metric, fontsize=15, fontweight='bold', labelpad=10)  # 增大y轴标签
    
    # 设置x轴刻度和标签 - 增大坐标轴文字
    ax.set_xticks(x_positions + bar_width * (len(celltypes) - 1) / 2)  # 调整刻度位置
    ax.set_xticklabels([f'{x:,}' for x in nFeature_order], 
                      rotation=45, ha='right', fontsize=14)  # 增大x轴刻度标签
    
    # 设置y轴刻度标签大小
    ax.tick_params(axis='y', labelsize=14)
    
    # 添加网格
    ax.grid(True, alpha=0.3, axis='y')
    
    # 根据指标类型调整y轴范围，为数值标签留出空间
    if metric in ['R2', 'PCC']:
        ax.set_ylim(bottom=0, top=1.1)  # 为顶部标签留出空间
    else:
        # 对于MAE和MAD，自动调整上限
        y_max = ax.get_ylim()[1]
        ax.set_ylim(bottom=0, top=y_max * 1.15)  # 为顶部标签留出空间

# 创建图例 - 只需Celltype图例
celltype_handles = [plt.Rectangle((0,0),1,1, fc=colors[i], alpha=1.0, edgecolor='black') 
                   for i in range(len(celltypes))]

# 在图形底部添加Celltype图例
fig.legend(celltype_handles, celltypes, title='Celltype',
           loc='lower center', 
           bbox_to_anchor=(0.52, 0.01),  # 底部中间
           ncol=len(celltype_handles),  # 水平排列
           fontsize=12,
           title_fontsize=13,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.92)  # 移除右侧留出的空间，因为Type图例已移除

# 保存图像
fig_path ='./data/scalex/fibro/human/age_prediction/EN_Exp_CT/small_file/'
plt.savefig(fig_path+'EN_CT_gene_'+type+'_benchmark.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig_path+'EN_CT_gene_'+type+'_benchmark.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print("可视化完成！图像已保存为 'EN_CT_gene_benchmark.png' 和 'EN_CT_gene_benchmark.pdf'")


In [ ]:
evaluation_df_combined_fib = evaluation_df_combined[(evaluation_df_combined['Celltype']=='Fibro')]
evaluation_df_combined_fib = evaluation_df_combined_fib[evaluation_df_combined_fib['Type']=='noA']


In [ ]:
evaluation_df_combined_fib


In [ ]:
################# Use the Fibro(noA), EN(gene+proportion) model, 8000 个吧, 1w Fibro效果有点点下降 
adata_h = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_healthy.h5ad')


In [ ]:
len(adata_h.obs['sample'].unique())


In [ ]:
target_cell_types = list(set(adata_h.obs['Celltype']))   # use the l3 celltype annotation as CT Proportion label 
k_features_list = [50,100,150,200,500,1000,3000,5000,8000,10000,15000,20000]
base_dir = './data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/with0/'
columns_list = ['Celltype', 'nFeature','model_nF', 'Type', 'R2', 'MAE', 'MAD', 'MSE', 'PCC']
evaluation_rows_data = []
k_feature = 20000


In [ ]:
print(k_feature)

######################################### 1. 构建特征矩阵 
# 步骤 1：计算细胞比例矩阵
cell_counts = adata_h.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
df.set_index('sample', inplace=True)
# 过滤：去掉太少细胞类型的个体
cell_prop_data = df.iloc[:, 2:]
zero_counts = (cell_prop_data == 0).sum(axis=1)    #### zero_counts 是 【0, len(target_cellt_types]的一个值
max_zeros = len(target_cell_types)   # 要求至少存在20种细胞类型（总共51列-3）
# df = df[zero_counts < max_zeros]  
# 保留的个体原始名字（去除后缀）
remaining_samples = df.index.str[:-2].unique()
print(f"剩余个体数: {df.shape[0]}")
print(f"剩余细胞总数: {cell_counts.loc[remaining_samples].sum().sum()}")
# 步骤 2：构建基因表达特征
cell_types = adata_h.obs['Celltype'].unique().tolist()
genes = adata_h.var_names.tolist()
# 只保留剩余个体
adata_filtered = adata_h[adata_h.obs['sample'].isin(remaining_samples)]   ## 对样本进行 filtering
all_feature_rows = []
index_names = []
# 遍历每个个体
for donor in tqdm(remaining_samples):
    adata_donor = adata_filtered[adata_filtered.obs['sample'] == donor]
    age = adata_donor.obs['Age'][0]
    total_cells = adata_donor.n_obs
    feature_dict = {'Age': age}
    for ct in cell_types:
        cells = adata_donor[adata_donor.obs['Celltype'] == ct]
        # 比例
        ratio = cells.n_obs / total_cells if total_cells > 0 and cells.n_obs > 0 else 0
        feature_dict[f"{ct}_ratio"] = ratio
        # 表达
        if cells.n_obs == 0:
            gene_values = np.zeros(len(genes))
        else:
            gene_values = cells.X.mean(axis=0).A1 if hasattr(cells.X, "A1") else np.asarray(cells.X.mean(axis=0)).flatten()  # .A1 转换为一维 ndarray，.flatten() 转换为 1D
        gene_names = [f"{ct}_{g}" for g in genes]
        feature_dict.update(dict(zip(gene_names, gene_values)))
    all_feature_rows.append(feature_dict)
    index_names.append(donor)
    

In [ ]:
# 构建最终 DataFrame
df_all_features = pd.DataFrame(all_feature_rows, index=index_names)
df_all_features.index.name = "sample"
######################################### 2.Get the weight matrix 
y = df_all_features['Age'].values
X = df_all_features.drop(columns='Age')


In [ ]:
# 区分细胞比例特征（假设以 "_prop" 结尾）和表达特征（其余）
prop_cols = [col for col in X.columns if col.endswith('_ratio')]
expr_cols = [col for col in X.columns if col not in prop_cols]
X_prop = X[prop_cols].copy()
X_expr = X[expr_cols].copy()
######################################### 3.去除低方差表达特征
vt = VarianceThreshold(threshold=1e-5)
X_expr_vt = vt.fit_transform(X_expr)
# 选择最相关的前 k 个特征（比如 3000 个）
skb = SelectKBest(score_func=f_regression, k=k_feature)    ######## 这里有一步选取相关特征的数目
X_expr_selected = skb.fit_transform(X_expr_vt, y)
# 获取保留下来的列名
selected_expr_cols = X_expr.columns[vt.get_support()][skb.get_support()]
# 合并，标准化
scaler_prop = StandardScaler()
scaler_expr = StandardScaler()
X_prop_scaled = scaler_prop.fit_transform(X_prop)
X_expr_scaled = scaler_expr.fit_transform(X_expr_selected)
# 合并，带列名
all_cols = list(X_prop.columns) + list(selected_expr_cols)
X_final = np.hstack([X_prop_scaled, X_expr_scaled])
X_final_df = pd.DataFrame(X_final, columns=all_cols)
######################################### 4.五倍交叉验证的code
cv = KFold(n_splits=5, shuffle=True, random_state=42)
model = ElasticNetCV(
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
    alphas=None,
    cv=cv,
    max_iter=5000,
    n_jobs=-1
)
scores = cross_val_score(model, X_final, y, cv=cv, scoring='r2')
print(f"交叉验证 R2 得分（平均±标准差）：{scores.mean():.3f} ± {scores.std():.3f}")


In [ ]:
######################################### 5.画 交叉验证的 mean(R2)图  #############
path = base_dir + str(k_feature)
os.makedirs(path, exist_ok=True)
plt.figure(figsize=(6, 3))
plt.boxplot(scores, vert=False, patch_artist=True,
            boxprops=dict(facecolor='skyblue', color='dodgerblue'),
            medianprops=dict(color='red'),
            whiskerprops=dict(color='dodgerblue'),
            capprops=dict(color='dodgerblue'),
            flierprops=dict(markerfacecolor='gray', marker='o', markersize=5, alpha=0.6))
plt.scatter(scores, [1]*len(scores), color='dodgerblue', edgecolor='k', label='Each Fold', zorder=5)
plt.axvline(scores.mean(), color='red', linestyle='--', label=f'Mean R² = {scores.mean():.3f}')
plt.title("Cross-Validation R² Scores")
plt.xlabel("R² Score")
plt.yticks([])
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
# plt.savefig(path+'/CV_R2.pdf', bbox_inches='tight', transparent=True)


In [ ]:

######################################### 6.合并细胞比例和筛选后的表达特征列名
model.fit(X_final, y)
selected_expr_cols = X_expr.columns[vt.get_support()][skb.get_support()]
all_cols = list(X_prop.columns) + list(selected_expr_cols)
# 提取 ElasticNet 模型的非零系数
coef = model.coef_
coef_df = pd.DataFrame({'Feature': all_cols, 'Weight': coef})
coef_df = coef_df[coef_df['Weight'] != 0].copy()   # weight 为 0 的feature没有用
# 添加特征类型列
coef_df['FeatureType'] = coef_df['Feature'].apply(
    lambda x: 'Cell Proportion' if x in X_prop.columns else 'Gene Expression'
)
# 按绝对值排序，获取前 20 个重要特征
coef_df = coef_df.sort_values(by='Weight', key=np.abs, ascending=False)


In [ ]:
coef_df.shape


In [ ]:
top_n = 20     ##  only display top50
top_features = coef_df.head(top_n)
# 可视化
plt.figure(figsize=(8,6 ))
sns.barplot(data=top_features, x='Weight', y='Feature', hue='FeatureType', dodge=False, palette='Set2')
plt.title(f'Top {top_n} Most Important Features from ElasticNet', fontsize=16)
plt.xlabel('ElasticNet Coefficient (Weight)', fontsize=14)
plt.ylabel('Feature Name', fontsize=14)
plt.legend(title='Feature Type', loc='best', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
# plt.savefig(path+'/top50Features.pdf', bbox_inches='tight', transparent=True)
# 输出统计信息
print(f"非零特征数量: {len(coef_df)}")


In [ ]:
y_pred = model.predict(X_final)
plot_en_prediction(y, y_pred)
os.chdir(path)
# joblib.dump({
#     'vt': vt,
#     'skb': skb,
#     'scaler_prop': scaler_prop,
#     'scaler_expr': scaler_expr,
#     'model': model,
#     'prop_cols': prop_cols,
#     'expr_cols': expr_cols,
#     "nonzero_features": coef_df
# }, path+'/Fibro_EnGeneCt_model_A_'+str(k_feature)+'_faetures.pkl')


In [ ]:
######################################### 7.Plot the prediction figure
y_pred = model.predict(X_final)
plot_en_prediction(y, y_pred,path=path)
######################################### 8.保存模型和全部预处理器 + 特征列名（便于验证集处理）
os.chdir(path)
joblib.dump({
    'vt': vt,
    'skb': skb,
    'scaler_prop': scaler_prop,
    'scaler_expr': scaler_expr,
    'model': model,
    'prop_cols': prop_cols,
    'expr_cols': expr_cols,
    "nonzero_features": coef_df
}, path+'/Fibro_EnGeneCt_model_A_'+str(k_feature)+'_faetures.pkl')


In [ ]:

######################################### 9.Get the evaluation metrics
y_true = np.array(y).flatten()
y_pred = np.array(y_pred).flatten()
# 计算评价指标
mae = mean_absolute_error(y_true, y_pred)
mad = np.median(np.abs(y_true - y_pred))
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
pcc, _ = pearsonr(y_true, y_pred)
evaluation_row_data = {
    'Celltype': 'Fibro',
    'nFeature': k_feature,
    'model_nF':coef_df.shape[0],
    'Type': 'without0',
    'R2': r2,
    'MAE': mae,
    'MAD': mad,
    'MSE': mse,
    'PCC': pcc
}
evaluation_rows_data.append(evaluation_row_data)



In [ ]:
########## 开始对 top_features 做functional & gene analysis 
split_cols = coef_df["Feature"].str.rsplit("_", n=1, expand=True)
coef_df['Gene'] =split_cols[1]
coef_df['Celltype'] =split_cols[0]
split_cols = coef_df["Celltype"].str.rsplit("_", n=1, expand=True)
coef_df['Tissue'] =split_cols[1]
coef_df['Celltype_sub'] =split_cols[0]
split_cols = coef_df["Celltype_sub"].str.rsplit("_", n=1, expand=True)
coef_df['subtype'] =split_cols[0]
coef_df


In [ ]:
# coef_df['Gene'].value_counts().head(40)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
######################################################### This script was used for EN(exp+CT proportion) model benchmarking   ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
import joblib
from tqdm import tqdm
import matplotlib
import warnings
matplotlib.font_manager._log.setLevel('ERROR')
warnings.filterwarnings('ignore', message='findfont: Font family.*not found')
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
# ----------------------
# 1. 加载你的真实数据（替换为你的coef_df）
# ----------------------
# coef_df = pd.read_csv("your_coef_df.csv")  # 或从其他地方加载

# ----------------------
# 2. 计算Celltype出现频次
# ----------------------
celltype_counts = coef_df['Celltype'].value_counts().reset_index()
celltype_counts.columns = ['Celltype', 'Frequency']

# ----------------------
# 3. Cancer Cell期刊配色（专业、高区分度）
# ----------------------
cancer_cell_palette = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5'
]

# 确保颜色数量足够
if len(celltype_counts) > len(cancer_cell_palette):
    cancer_cell_palette = cancer_cell_palette * (len(celltype_counts) // len(cancer_cell_palette) + 1)
cell_colors = cancer_cell_palette[:len(celltype_counts)]

# ----------------------
# 4. 绘制Cancer Cell风格柱状图
# ----------------------
plt.rcParams['font.family'] = 'Arial'  # 期刊指定字体
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 0.5

fig, ax = plt.subplots(figsize=(10, 6))

# 绘制柱状图
bars = ax.bar(
    x=celltype_counts['Celltype'],
    height=celltype_counts['Frequency'],
    color=cell_colors,
    edgecolor='white',
    linewidth=0.5
)

# 添加频次标签
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.1,
        f'{int(height)}',
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )

# 设置图表样式
ax.set_title('Feature Importance Matrix: Celltype Frequency', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Celltype', fontsize=12, labelpad=10)
ax.set_ylabel('Frequency', fontsize=12, labelpad=10)
ax.set_xticks(np.arange(len(celltype_counts)))
ax.set_xticklabels(celltype_counts['Celltype'], rotation=45, ha='right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

# 调整布局
plt.tight_layout()

# 保存为PDF（Cancer Cell期刊要求矢量图）
# plt.savefig('celltype_frequency_cancer_cell_style.pdf', dpi=300, bbox_inches='tight', format='pdf')
plt.show()

# 打印频次统计结果
print("Celltype频次统计：")
print(celltype_counts)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
# 2. 计算Celltype出现频次并按频次降序排序（便于可视化）
celltype_counts = coef_df['Celltype'].value_counts().reset_index()
celltype_counts.columns = ['Celltype', 'Frequency']
celltype_counts = celltype_counts.sort_values('Frequency', ascending=True)
# ----------------------
# 3. Cancer Cell期刊配色（专业、高区分度）
# ----------------------
cancer_cell_palette = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5'
]
# 确保颜色数量匹配Celltype数量（不足则循环）
if len(celltype_counts) > len(cancer_cell_palette):
    cancer_cell_palette = cancer_cell_palette * (len(celltype_counts) // len(cancer_cell_palette) + 1)
cell_colors = cancer_cell_palette[:len(celltype_counts)]

# ----------------------
# 4. 绘制Cancer Cell风格水平柱状图（核心修改）
# ----------------------
# 期刊规范：Arial字体、细边框、无 
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 0.5
# 设置画布尺寸（适配水平图）
fig, ax = plt.subplots(figsize=(6, 10))
# 绘制水平柱状图
bars = ax.barh(
    y=celltype_counts['Celltype'],  # Y轴为Celltype
    width=celltype_counts['Frequency'],  # X轴为频次
    color=cell_colors,
    edgecolor='white',  # 白色边框提升区分度
    linewidth=0.5
)
# 在每个柱子右侧添加频次数字标签（期刊风格）
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.1,  # 标签在柱子右侧
        bar.get_y() + bar.get_height() / 2,
        f'{int(width)}',  # 显示整数频次
        ha='left',
        va='center',
        fontsize=9,
        fontweight='bold'
    )
# ----------------------
# 5. 期刊级样式调整（关键）
# ----------------------
# 标题：加粗、适中字号，符合Cancer Cell排版
ax.set_title('Feature count: Celltype Frequency', fontsize=14, fontweight='bold', pad=20)
# 轴标签：清晰、适中间距
ax.set_xlabel('Frequency', fontsize=12, labelpad=10)
ax.set_ylabel('Celltype', fontsize=12, labelpad=10)
# 隐藏冗余边框（仅保留X/Y轴）
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# 浅灰色垂直网格线（提升可读性，不杂乱）
ax.grid(axis='x', linestyle='--', alpha=0.3)
# Y轴标签居中对齐，字号适配
ax.tick_params(axis='y', labelsize=9)
# ----------------------
# 6. 保存（期刊要求矢量图+高分辨率）
plt.tight_layout()  # 自动调整布局，避免标签截断
# plt.savefig(
#     'celltype_frequency_horizontal_cancer_cell.pdf',
#     dpi=300,
#     bbox_inches='tight',  # 完整保存所有元素
#     format='pdf'  # 矢量图，支持期刊缩放
# )
plt.show()


In [ ]:
len(Inter_genes)

In [ ]:
####### 查看哪些基因出现次数最多,出现越多次说明可能越重要，在不同的 Celltype 中比较稳定  
coef_df_gene = coef_df[coef_df['FeatureType']=='Gene Expression']
gene_counts = coef_df_gene['Gene'].value_counts()
gene_list = gene_counts[gene_counts >= 2].index.tolist()


In [ ]:
# coef_df_gene['Weight'].min()
coef_df_gene['Gene'].value_counts().head(20)

In [ ]:
gene_list = list(gene_counts.index)
len(gene_list)


In [ ]:
len(gene_list)


In [ ]:
####### 查看哪些基因出现次数最多,出现越多次说明可能越重要，在不同的 Celltype 中比较稳定  

##### 接下来看下上面的 gene_list(也就是至少在 2 种Celltype中都有Weight 的基因的 权重查看)
from matplotlib.colors import LinearSegmentedColormap
df_expr = coef_df_gene[coef_df_gene['Gene'].isin(gene_list)]
celltype_order = df_expr.groupby("Celltype")["Gene"].nunique().sort_values(ascending=False).index.tolist()
# 统计每个基因出现在多少细胞类型中
gene_order = df_expr.groupby("Gene")["Celltype"].nunique().sort_values(ascending=False).index.tolist()
# 创建 pivot 表格
heatmap_data = df_expr.pivot(index="Gene", columns="Celltype", values="Weight")
# 重新排序行列
heatmap_data = heatmap_data.loc[gene_order, celltype_order]
heatmap_data = heatmap_data.fillna(0)
# 自定义 colormap：蓝->白->红
max_coef = np.max(np.abs(heatmap_data.values))
cmap = LinearSegmentedColormap.from_list("red_blue", ["#183579", "#FFFFFF", "#C0210F"])
# 调整 figsize，使格子更小
plt.figure(figsize=(8, max(4, 0.15*heatmap_data.shape[0])))
sns.heatmap(
    heatmap_data,
    cmap=cmap,
    center=0,
    linewidths=0.3,
    linecolor="gray",
    cbar_kws={"label": "Coefficient"},
    square=False,
    yticklabels=True
)
plt.xlabel("Cell Type", fontsize=10)
plt.ylabel("Gene", fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.title("Gene weight in Celltype", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
gene_counts = coef_df_gene['Gene'].value_counts()
gene_list = gene_counts[gene_counts >= 2].index.tolist()   # list(gene_counts.index)[0:30]  #
sc.pl.heatmap(adata_h, var_names=gene_list,    # ['VCAN','HLA-B','PLEKHM3','SRGN','LXN']
              cmap='RdBu_r', groupby='Age_group')


In [ ]:
# gene_list_clustered


In [ ]:
len(gene_list)


In [ ]:
import scanpy as sc
import numpy as np

# 方法1：设置 swap_axes=True，这样行是基因，列是细胞，然后对基因进行聚类
sc.pl.heatmap(
    adata_h, 
    var_names= gene_list_clustered,  # ['VCAN','HLA-B','PLEKHM3','SRGN','LXN']
    groupby='Age_group',
    cmap='RdBu_r',
    swap_axes=True,  # 交换坐标轴，让基因作为行
    dendrogram=True,  # 显示聚类树
    figsize=(10, 8),
   standard_scale='var',  # 对基因进行标准化
    show=True
)


In [ ]:
import scanpy as sc

# 首先对基因进行聚类分析
sc.tl.dendrogram(
    adata_h, 
    groupby='Age_group',
    var_names=gene_list,  # 指定要聚类的基因
    use_rep='X',  # 使用表达矩阵
    cor_method='pearson',  # 相关性方法
    linkage_method='average'
)

# 然后绘制热图，dendrogram=True会自动使用聚类结果
sc.pl.heatmap(
    adata_h, 
    var_names=gene_list,
    groupby='Age_group',
    cmap='RdBu_r',
    dendrogram=True,  # 这会显示基因和细胞的聚类树
    swap_axes=True,  # 让基因聚类树在左侧
    standard_scale='var',
    figsize=(12, 8),
    show=True
)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# 计算每个年龄组的平均表达
age_groups = sorted(adata_h.obs['Age_group'].unique())
avg_expr_matrix = []
for gene in gene_list:
    gene_avg = []
    for age in age_groups:
        age_cells = adata_h[adata_h.obs['Age_group'] == age]
        expr = age_cells[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray()
        gene_avg.append(np.mean(expr))
    avg_expr_matrix.append(gene_avg)

# 转换为DataFrame
avg_expr_df = pd.DataFrame(
    avg_expr_matrix,
    index=gene_list,
    columns=age_groups
)

# 对基因进行标准化
avg_expr_df_scaled = avg_expr_df.apply(
    lambda x: (x - x.mean()) / x.std(), axis=1
)
# 对基因进行聚类
gene_dist = pdist(avg_expr_df_scaled.values, metric='correlation')
gene_linkage = linkage(gene_dist, method='average')
gene_order = leaves_list(gene_linkage)

# 按聚类顺序重新排列
avg_expr_df_clustered = avg_expr_df_scaled.iloc[gene_order]
# 创建图形
fig, (ax_dendro, ax_heat) = plt.subplots(
    1, 2, 
    figsize=(12, 8),
    gridspec_kw={'width_ratios': [1, 4]}
)
# 绘制基因聚类树
from scipy.cluster.hierarchy import dendrogram
dendrogram(
    gene_linkage,
    ax=ax_dendro,
    orientation='left',
    labels=avg_expr_df_clustered.index.tolist(),
    leaf_font_size=10
)
ax_dendro.set_xticks([])
ax_dendro.set_title('Gene Clustering')
# 绘制热图
im = ax_heat.imshow(
    avg_expr_df_clustered.values,
    cmap='RdBu_r',
    aspect='auto',
    interpolation='nearest'
)
# 设置标签
ax_heat.set_xticks(range(len(age_groups)))
ax_heat.set_xticklabels(age_groups)
ax_heat.set_yticks(range(len(avg_expr_df_clustered.index)))
ax_heat.set_yticklabels(avg_expr_df_clustered.index)
ax_heat.set_xlabel('Age Group')
ax_heat.set_title('Gene Expression Patterns')
# 添加颜色条
plt.colorbar(im, ax=ax_heat, label='Scaled Expression')
plt.suptitle('Gene Clustering Across Age Groups', y=0.98, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
avg_expr_df_clustered


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt


# 创建平均表达矩阵（基因 × 年龄组）
avg_expr = pd.DataFrame(
    index=gene_list,
    columns=sorted(adata_h.obs['Age_group'].unique())
)

for age in avg_expr.columns:
    age_cells = adata_h[adata_h.obs['Age_group'] == age]
    # 提取基因表达并计算平均值
    expr_data = age_cells[:, gene_list].X
    if hasattr(expr_data, 'toarray'):
        expr_data = expr_data.toarray()
    avg_expr[age] = np.mean(expr_data, axis=0)

# 对基因进行层次聚类
# 计算基因之间的相关性距离
gene_dist = pdist(avg_expr.values, metric='correlation')
gene_linkage = linkage(gene_dist, method='average')
gene_order = leaves_list(gene_linkage)

# 按聚类顺序重新排列基因
gene_list_clustered = [gene_list[i] for i in gene_order]

# 使用 matrixplot 绘制带基因聚类的热图
sc.pl.matrixplot(
    adata_h,
    var_names=gene_list_clustered,  # 使用聚类后的基因顺序
    groupby='Age_group',
    cmap='RdBu_r',
    standard_scale='var',  # 对基因进行标准化
    dendrogram=False,  # 不显示细胞聚类
    swap_axes=True,  # 让基因作为行，年龄组作为列
    title='Gene Clustering Across Age Groups',
    figsize=(8, 30)
)
plt.show()


In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

age_groups = sorted(adata_h.obs['Age_group'].unique())
mean_expr_matrix = []

for gene in gene_list:
    gene_expr_by_age = []
    for age in age_groups:
        age_cells = adata_h[adata_h.obs['Age_group'] == age]
        expr = age_cells[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray()
        gene_expr_by_age.append(np.mean(expr))
    mean_expr_matrix.append(gene_expr_by_age)

# 转换为DataFrame
mean_expr_df = pd.DataFrame(
    mean_expr_matrix,
    index=gene_list,
    columns=age_groups
)

# 对基因进行标准化（每行均值为0，标准差为1）
mean_expr_df_scaled = mean_expr_df.apply(
    lambda x: (x - x.mean()) / x.std(), axis=1
)

# 对基因进行层次聚类
gene_linkage = linkage(
    pdist(mean_expr_df_scaled.values, metric='correlation'),
    method='average'
)

# 创建带有基因聚类树的热图
fig = plt.figure(figsize=(10, 8))

# 创建网格布局：[左侧聚类树, 右侧热图]
ax1 = plt.axes([0.05, 0.1, 0.15, 0.8])  # 基因聚类树
ax2 = plt.axes([0.25, 0.1, 0.7, 0.8])   # 热图

# 绘制基因聚类树
with plt.rc_context({'lines.linewidth': 1}):
    dendrogram(
        gene_linkage,
        ax=ax1,
        orientation='left',
        labels=gene_list,
        leaf_font_size=10,
        leaf_rotation=0
    )
ax1.set_xticks([])
ax1.set_title('Gene Clustering')

# 按聚类顺序重新排列基因
from scipy.cluster.hierarchy import leaves_list
gene_order = leaves_list(gene_linkage)
heatmap_data = mean_expr_df_scaled.iloc[gene_order]

# 绘制热图
sns.heatmap(
    heatmap_data,
    ax=ax2,
    cmap='RdBu_r',
    cbar_kws={'label': 'Scaled Expression'},
    annot=False,
    fmt='.2f',
    linewidths=0.5
)
ax2.set_ylabel('')
ax2.set_xlabel('Age Group')
ax2.set_title('Gene Expression Patterns')

plt.suptitle('Gene Clustering Analysis Across Age Groups', y=0.98)
plt.show()

In [ ]:
############################# 测试下 RF( gene+ celltype pro) 的模型
adata_h = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge_healthy.h5ad')


In [ ]:
gene_list


In [ ]:
####### 查看哪些基因出现次数最多,出现越多次说明可能越重要，在不同的 Celltype 中比较稳定  
coef_df_gene = coef_df[coef_df['FeatureType']=='Gene Expression']
gene_counts = coef_df_gene['Gene'].value_counts()
# gene_list = gene_counts[gene_counts >= 2].index.tolist()
gene_list =list(gene_counts.index)[0:30]


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
df_expr = coef_df_gene[coef_df_gene['Gene'].isin(gene_list)]
celltype_order = df_expr.groupby("Celltype")["Gene"].nunique().sort_values(ascending=False).index.tolist()
# 统计每个基因出现在多少细胞类型中
gene_order = df_expr.groupby("Gene")["Celltype"].nunique().sort_values(ascending=False).index.tolist()
# 创建 pivot 表格
heatmap_data = df_expr.pivot(index="Gene", columns="Celltype", values="Weight")
# 重新排序行列
heatmap_data = heatmap_data.loc[gene_order, celltype_order]
heatmap_data = heatmap_data.fillna(0)
# 自定义 colormap：蓝->白->红
max_coef = np.max(np.abs(heatmap_data.values))
cmap = LinearSegmentedColormap.from_list("red_blue", ["#183579", "#FFFFFF", "#C0210F"])
# 调整 figsize，使格子更小
plt.figure(figsize=(8, max(4, 0.15*heatmap_data.shape[0])))
sns.heatmap(
    heatmap_data,
    cmap=cmap,
    center=0,
    linewidths=0.3,
    linecolor="gray",
    cbar_kws={"label": "Coefficient"},
    square=False,
    yticklabels=True
)
plt.xlabel("Cell Type", fontsize=10)
plt.ylabel("Gene", fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.title("Gene weight in Celltype", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 使用rf模型预测
import joblib
# model_package = joblib.load("age_rf_pipeline.pkl")
# rf_pipeline = model_package['pipeline']
# train_features = model_package['feature_cols']

def predict_age_rf(adata, rf_pipeline, individual_col, age_col, celltype_col, min_cells_threshold=28):
    """
    使用训练好的 RF pipeline 对新数据进行预测
    adata: AnnData 对象（外部验证集）
    rf_pipeline: 已训练的 Pipeline（包含 ColumnTransformer + RF）
    individual_col: 样本/个体 ID 列名
    age_col: 年龄列名
    celltype_col: 细胞类型列名
    min_cells_threshold: 每个个体保留的最少细胞类型数量阈值
    """
    import numpy as np
    import pandas as pd
    from tqdm import tqdm

    # 1. 构建细胞类型比例矩阵
    cell_counts = adata.obs.groupby([individual_col, age_col, celltype_col]).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df_valid = cell_props.reset_index()
    df_valid[individual_col] = df_valid[individual_col].astype(str)
    df_valid[individual_col] = df_valid.groupby(individual_col).cumcount().astype(str).radd(df_valid[individual_col] + '_')
    df_valid.set_index(individual_col, inplace=True)
    print(df_valid)
    # 去除细胞类型数量过少的个体
    zero_counts = (df_valid.iloc[:, 2:] == 0).sum(axis=1)
    df_valid = df_valid[zero_counts < min_cells_threshold]
    remaining_ids = df_valid.index.tolist()

    # 2. 构建特征矩阵
    genes = adata.var_names.tolist()
    gene_matrix = adata.X.A if hasattr(adata.X, "A") else adata.X

    data_rows = []
    index_names = []

    for ind in tqdm(remaining_ids):
        donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
        adata_donor = adata[donor_mask]
        total_cells = adata_donor.n_obs

        row = {}
        try:
            row[age_col] = adata_donor.obs[age_col].iloc[0]
            
        except IndexError:
            continue

        for ct in adata_donor.obs[celltype_col].unique():
            ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
            ct_cells = adata[ct_mask]

            ratio = ct_cells.n_obs / total_cells
            row[f"{ct}_ratio"] = ratio

            if ct_cells.n_obs == 0:
                gene_vals = np.zeros(len(genes))
            else:
                sub_expr = gene_matrix[ct_mask]
                gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr.mean(axis=0), "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

            gene_names = [f"{ct}_{g}" for g in genes]
            row.update(dict(zip(gene_names, gene_vals)))

        data_rows.append(row)
        index_names.append(ind)

    df_features = pd.DataFrame(data_rows, index=index_names)
    df_features.index.name = individual_col
    df_features.fillna(0, inplace=True)  # 缺失特征补 0
    
    # 3. 对齐训练特征
    if hasattr(rf_pipeline, 'feature_names_in_'):
        train_features = rf_pipeline.feature_names_in_
    else:
        # 如果你保存了训练特征列表
        train_features = rf_pipeline.named_steps['preprocess'].get_feature_names_out()

    # 缺失列补 0
    missing_cols = {c: 0.0 for c in train_features if c not in df_features.columns}
    if missing_cols:
        df_features = pd.concat([df_features, pd.DataFrame(missing_cols, index=df_features.index)], axis=1)

    # 保留训练列顺序(y不在其中)
    X_features = df_features[train_features]
    # 4. 分离 y 与 X
    y_valid = df_features[age_col] if age_col in df_features.columns else None
    X_valid = X_features.drop(columns=[age_col], errors='ignore')
    
    # 5. 预测
    y_pred = rf_pipeline.predict(X_valid)

    # 6. 输出 DataFrame
    df_results = pd.DataFrame({
        'Actual_Age': y_valid,
        'Predicted_Age': y_pred,
        'Age_Diff': y_pred - y_valid if y_valid is not None else np.nan
    }, index=df_features.index)

    return df_results


In [ ]:
# 可视化预测效果
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

def plot_rf_prediction(y_true, y_pred, extra_points=None, extra_labels=None):
    """
    可视化模型的预测效果
    :param y_true: 实际年龄（Series 或 array）
    :param y_pred: 预测年龄（形状为 (n, 1) 的 array）
    """

    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    # 计算评价指标
    mae = mean_absolute_error(y_true, y_pred)
    mad = np.median(np.abs(y_true - y_pred))
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pcc, _ = pearsonr(y_true, y_pred)


    # 回归线（基于 y_pred 和 y_true 的线性回归）
    reg = LinearRegression().fit(y_pred.reshape(-1, 1), y_true)
    y_fit = reg.predict(y_pred.reshape(-1, 1))

    # 绘图
    plt.figure(figsize=(8, 6))
    x_min, x_max = -10, 130
    plt.xlim(x_min, x_max)
    plt.ylim(x_min, x_max)
    plt.scatter(y_true, y_pred, c='dodgerblue', alpha=0.7, edgecolor='k', label='Samples')
    plt.plot([y_true.min()-5, y_true.max()+35], [y_true.min()-5, y_true.max()+35],
             'r--', lw=2, label='Ideal line (y = x)')
    # 拟合回归线
    slope, intercept = np.polyfit(y_true, y_pred.flatten(), 1)

    # 构造更长的 X 范围
    x_line = np.linspace(min(y_true) - 5, max(y_true) + 35, 100)
    y_line = slope * x_line + intercept

    # 画出加长后的 PLSR 回归线
    plt.plot(x_line, y_line, 'g-', lw=2, label='EN regression line')
    '''for i in range(len(y_true)):
        plt.text(y_true[i], y_pred[i], f"{int(y_true[i])}", fontsize=8, ha='right', va='bottom', color='black') '''
    # 添加文本：R²、MAD 和 PCC
    plt.text(
        0.05, 0.95,
        f"$R^2$: {r2:.2f}\nMAE: {mae:.2f}\nMAD:{mad:.2f}\nPCC: {pcc:.2f}",
        transform=plt.gca().transAxes,
        fontsize=16,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )
    # 添加额外点
    if extra_points:
        for i, (x, y) in enumerate(extra_points):
            label = extra_labels[i] if extra_labels else 'Extra Point'
            plt.scatter(x, y, color='red', edgecolor='black', s=100, marker='o', label=label)
    
    # 去重图例
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='upper right')
    plt.xlabel('True Age', fontsize=18)
    plt.ylabel('Predicted Age', fontsize=18)
    plt.title('RF Model: External healthy validation', fontsize=20)
    plt.legend(loc='lower right', fontsize=16)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

In [ ]:
adata_age = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')
adata_samples = list(adata_h.obs['sample'].unique())  # len(adata_samples)
adata_age = adata_age[~adata_age.obs['sample'].isin(adata_samples)]  # 过滤掉 adata_h 中有的 sample
gse_sample_counts = adata_age.obs.groupby('GSE')['sample'].nunique().sort_values(ascending=False)  # 统计每个GSE中包含的sample数


In [ ]:
top_k = 30  # 可以根据需要调整
top_gses = gse_sample_counts.head(top_k).index.tolist()


In [ ]:
top_gses

In [ ]:
gse_sample_counts.head(30)


In [ ]:
###### Calculate age difference 的函数
def age_diff_gene_prop(adata_valid, pipeline,max_zeros=20):
    # 加载模型和处理器
    vt = pipeline['vt']
    skb = pipeline['skb']
    scaler_prop = pipeline['scaler_prop']
    scaler_expr = pipeline['scaler_expr']
    model = pipeline['model']
    prop_cols = pipeline['prop_cols']
    expr_cols = pipeline['expr_cols']
    # ===== 步骤 1：构建细胞比例矩阵 =====
    cell_counts_valid = adata_valid.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props_valid = cell_counts_valid.div(cell_counts_valid.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df_valid = cell_props_valid.reset_index()
    df_valid['sample'] = df_valid['sample'].astype(str)
    df_valid['sample'] = df_valid.groupby('sample').cumcount().astype(str).radd(df_valid['sample'] + '_')
    df_valid.set_index('sample', inplace=True)
    # 过滤掉细胞类型少于20种的个体
    cell_prop_data_valid = df_valid.iloc[:, 2:]
    zero_counts_valid = (cell_prop_data_valid == 0).sum(axis=1)
    max_zeros = max_zeros  # 51种细胞类型中最多允许31个为0
    df_valid = df_valid[zero_counts_valid < max_zeros]
    remaining_individuals_valid = df_valid.index.str[:-2].unique()
    # ===== 步骤 2：构建验证集特征矩阵 =====
    cell_types = adata_valid.obs['Celltype'].unique().tolist()
    genes = adata_valid.var_names.tolist()
    adata_filtered_valid = adata_valid[adata_valid.obs['sample'].isin(remaining_individuals_valid)]
    all_feature_rows_valid = []
    index_names_valid = []
    for donor in tqdm(remaining_individuals_valid):
        adata_donor = adata_filtered_valid[adata_filtered_valid.obs['sample'] == donor]
        age = adata_donor.obs['Age'][0]
        total_cells = adata_donor.n_obs
        feature_dict = {'Age': age}
        for ct in cell_types:
            cells = adata_donor[adata_donor.obs['Celltype'] == ct]
            ratio = cells.n_obs / total_cells if total_cells > 0 and cells.n_obs > 0 else 0
            feature_dict[f"{ct}_ratio"] = ratio
            if cells.n_obs == 0:
                gene_values = np.zeros(len(genes))
            else:
                gene_values = cells.X.mean(axis=0).A1 if hasattr(cells.X, "A1") else np.asarray(cells.X.mean(axis=0)).flatten()
            gene_names = [f"{ct}_{g}" for g in genes]
            feature_dict.update(dict(zip(gene_names, gene_values)))
        all_feature_rows_valid.append(feature_dict)
        index_names_valid.append(donor)
    df_valid_features = pd.DataFrame(all_feature_rows_valid, index=index_names_valid)
    df_valid_features.index.name = "sample"
    # ===== 步骤 3：特征处理 & 年龄预测 =====
    y_valid = df_valid_features['Age'].values
    X_valid = df_valid_features.drop(columns='Age')
    # 找出缺失的列
    missing_prop_cols = [col for col in prop_cols if col not in X_valid.columns]
    missing_expr_cols = [col for col in expr_cols if col not in X_valid.columns]
    # 为缺失的列创建 DataFrame，值全为 0，行数与 X_valid 一样
    missing_prop_df = pd.DataFrame(0.0, index=X_valid.index, columns=missing_prop_cols)
    missing_expr_df = pd.DataFrame(0.0, index=X_valid.index, columns=missing_expr_cols)
    # 一次性拼接
    X_valid = pd.concat([X_valid, missing_prop_df, missing_expr_df], axis=1)
    # 按照训练集列顺序取出
    X_valid_prop = X_valid[prop_cols].copy()
    X_valid_expr = X_valid[expr_cols].copy()
    # 与训练集相同处理流程
    X_valid_expr_vt = vt.transform(X_valid_expr)
    X_valid_expr_selected = skb.transform(X_valid_expr_vt)
    X_valid_prop_scaled = scaler_prop.transform(X_valid_prop)
    X_valid_expr_scaled = scaler_expr.transform(X_valid_expr_selected)
    X_valid_final = np.hstack([X_valid_prop_scaled, X_valid_expr_scaled])
    # ===== 步骤 4：预测年龄 & 计算差值 =====
    y_pred = model.predict(X_valid_final)
    age_diff = y_pred - y_valid
    df_results = pd.DataFrame({
        'Actual_Age': y_valid,
        'Predicted_Age': y_pred,
        'Age_Diff': age_diff
    }, index=df_valid_features.index)
    return df_results



In [ ]:

adata_1 = adata_age[adata_age.obs['GSE']=='GSE278456'] # top_gses[0]
adata_2 = adata_age[adata_age.obs['GSE']=='GSE164789']
adata_3 = adata_age[adata_age.obs['GSE']=='GSE266616']
adata_4 = adata_age[adata_age.obs['GSE']=='GSE265801']
adata_5 = adata_age[adata_age.obs['GSE']=='GSE290927']
import joblib
pipeline = joblib.load('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/with0/8000/Fibro_EnGeneCt_model_A_8000_faetures.pkl')
GSE1_pred_df = age_diff_gene_prop(adata_1, pipeline)
GSE2_pred_df = age_diff_gene_prop(adata_2, pipeline)
GSE3_pred_df = age_diff_gene_prop(adata_3, pipeline)
GSE4_pred_df = age_diff_gene_prop(adata_4, pipeline)
GSE5_pred_df = age_diff_gene_prop(adata_5, pipeline)



In [ ]:
import joblib
pipeline = joblib.load('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro/with0/8000/Fibro_EnGeneCt_model_A_8000_faetures.pkl')
cont_pred_df = age_diff_gene_prop(adata_h, pipeline)
age_diff_h = y_pred -y    ## healthy 的age difference
GSE1_pred_df = age_diff_gene_prop(adata_1, pipeline)
GSE2_pred_df = age_diff_gene_prop(adata_2, pipeline)
GSE3_pred_df = age_diff_gene_prop(adata_3, pipeline)
GSE4_pred_df = age_diff_gene_prop(adata_4, pipeline)
GSE5_pred_df = age_diff_gene_prop(adata_5, pipeline)
age_diff_dict = {
        'Healthy': cont_pred_df['Age_Diff'],  # age_diff_h,
        'Can|GSE278456': GSE1_pred_df['Age_Diff'],
    'Can|GSE164789': GSE2_pred_df['Age_Diff'],
    'Immune|GSE266616': GSE3_pred_df['Age_Diff'],
         'Can|GSE265801': GSE4_pred_df['Age_Diff'],
     'Rare|GSE290927': GSE5_pred_df['Age_Diff']
    
        
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
age_diff_dict = {
        'Healthy': cont_pred_df['Age_Diff'],  # age_diff_h,
        'Can|GSE278456': GSE1_pred_df['Age_Diff'],
    'Can|GSE164789': GSE2_pred_df['Age_Diff'],
    'Immune|GSE266616': GSE3_pred_df['Age_Diff'],
         'Can|GSE265801': GSE4_pred_df['Age_Diff'],
     'Rare|GSE290927': GSE5_pred_df['Age_Diff']
    
        
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
['GSE235057', 'GSE253198', 'GSE138669', 'GSE269204', 'GSE144239']


In [ ]:

adata_1 = adata_age[adata_age.obs['GSE']=='GSE235057'] # top_gses[0]
adata_2 = adata_age[adata_age.obs['GSE']=='GSE253198']
adata_3 = adata_age[adata_age.obs['GSE']=='GSE138669']
adata_4 = adata_age[adata_age.obs['GSE']=='GSE269204']
adata_5 = adata_age[adata_age.obs['GSE']=='GSE144239']
GSE1_pred_df = age_diff_gene_prop(adata_1, pipeline)
GSE2_pred_df = age_diff_gene_prop(adata_2, pipeline)
GSE3_pred_df = age_diff_gene_prop(adata_3, pipeline)
GSE4_pred_df = age_diff_gene_prop(adata_4, pipeline)
GSE5_pred_df = age_diff_gene_prop(adata_5, pipeline)


In [ ]:
age_diff_dict = {
        'Healthy': cont_pred_df['Age_Diff'],  # age_diff_h,
        'Can|GSE278456': GSE1_pred_df['Age_Diff'],
    'Can|GSE164789': GSE2_pred_df['Age_Diff'],
    'Immune|GSE266616': GSE3_pred_df['Age_Diff'],
         'Can|GSE265801': GSE4_pred_df['Age_Diff'],
     'Rare|GSE290927': GSE5_pred_df['Age_Diff']
    
        
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
GSE5_pred_df



In [ ]:
GSE1_pred_df['Age_Diff'], GSE2_pred_df['Age_Diff'], GSE5_pred_df['Age_Diff']

In [ ]:
top_gses

In [ ]:

coef_df_sub = coef_df[abs(coef_df['Weight']) > 0.1]
coef_df_sub.shape


In [ ]:
(pipeline['nonzero_features']['Weight']<0).value_counts()



In [ ]:
(coef_df['Weight']>0).value_counts()


In [ ]:
adata_age.obs[adata_age.obs['GSE'].isin(top_gses)][['GSE','GSM','Age','Age_group','disease_global','disease_global_check']].drop_duplicates()


In [ ]:
cont_pred_df = age_diff_gene_prop(adata_h, pipeline)
age_diff_h = y_pred -y    ## healthy 的age difference
GSE1_pred_df = age_diff_gene_prop(adata_1, pipeline)
GSE2_pred_df = age_diff_gene_prop(adata_2, pipeline)
GSE3_pred_df = age_diff_gene_prop(adata_3, pipeline)
GSE4_pred_df = age_diff_gene_prop(adata_4, pipeline)
GSE5_pred_df = age_diff_gene_prop(adata_5, pipeline)


In [ ]:
age_diff_dict = {
        'Healthy': cont_pred_df['Age_Diff'],  # age_diff_h,
        'Can|GSE278456': GSE1_pred_df['Age_Diff'],
    'Can|GSE164789': GSE2_pred_df['Age_Diff'],
    'Immune|GSE266616': GSE3_pred_df['Age_Diff'],
         'Can|GSE265801': GSE4_pred_df['Age_Diff'],
     'Rare|GSE290927': GSE5_pred_df['Age_Diff']
    
        
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from tqdm import tqdm


# cont_pred_df = age_diff_gene_prop(adata_healthy, pipeline)
# age_diff_h = cont_pred_df['Age_Diff']

# 批量处理每个GSE数据集
GSEs_pred_dfs = []  # 存储每个GSE的预测结果DataFrame
gse_names = []      # 存储GSE编号
disease_types = []  # 存储疾病类型

# 获取GSE和disease_global_check的对应关系
# 假设您有一个包含所有数据的adata_age对象
gse_disease_map = adata_age.obs[adata_age.obs['GSE'].isin(top_gses)][['GSE', 'disease_global_check']].drop_duplicates()
gse_disease_dict = dict(zip(gse_disease_map['GSE'], gse_disease_map['disease_global_check']))

# 记录有问题的GSE（预测结果异常）
problematic_gses = []
outlier_samples = {}  # 存储每个GSE中的离群样本

print("开始批量处理GSE数据集...")
for gse in top_gses:   # 
    try:
        adata_gse = adata_age[adata_age.obs['GSE']==gse]
        # 计算年龄差异
        pred_df = age_diff_gene_prop(adata_gse, pipeline)
        # 检查是否有异常值（例如年龄差异过大）
        age_diffs = pred_df['Age_Diff'].values
        Q1 = np.percentile(age_diffs, 25)
        Q3 = np.percentile(age_diffs, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 3 * IQR  # 使用更宽松的阈值，3倍IQR
        upper_bound = Q3 + 3 * IQR
        outliers = pred_df[(pred_df['Age_Diff'] < lower_bound) | (pred_df['Age_Diff'] > upper_bound)]
        if len(outliers) > 0:
            outlier_samples[gse] = outliers.index.tolist()
            print(f"警告: {gse} 中有 {len(outliers)} 个离群样本")
            
            # 如果离群点太多，标记为有问题的GSE
            if len(outliers) > len(pred_df) * 0.2:  # 超过20%的样本是离群点
                problematic_gses.append(gse)
                print(f"⚠️ {gse} 离群点比例过高 ({len(outliers)/len(pred_df)*100:.1f}%)，将标记为有问题")
        # 存储结果
        GSEs_pred_dfs.append(pred_df)
        gse_names.append(gse)
        # 获取疾病类型，如果没有映射则使用'Unknown'
        disease_type = gse_disease_dict.get(gse, 'Unknown')
        disease_types.append(disease_type)  
    except Exception as e:
        print(f"处理 {gse} 时出错: {e}")
        problematic_gses.append(gse)
        continue

print(f"\n处理完成！成功处理 {len(GSEs_pred_dfs)} 个GSE，{len(problematic_gses)} 个有问题")

# 构建age_diff_dict
age_diff_dict = {
    'Healthy': age_diff_h  # 健康对照组
}

# 添加每个GSE的结果到字典
for i, (gse, pred_df, disease) in enumerate(zip(gse_names, GSEs_pred_dfs, disease_types)):
    # 可以根据需要自定义标签格式
    # 格式1: 只使用GSE编号
    # label = f"{gse}"
    
    # 格式2: 包含疾病类型
    label = f"{disease}|{gse}"
    
    # 格式3: 如果需要更简洁的标签，可以使用索引
    # label = f"Dataset_{i+1}|{gse}"
    
    age_diff_dict[label] = pred_df['Age_Diff']

# 可选：排除有问题的GSE
exclude_problematic = True  # 设为True排除有问题的GSE
if exclude_problematic and problematic_gses:
    print(f"\n排除有问题的GSE: {problematic_gses}")
    # 创建新的字典，排除有问题的GSE
    filtered_age_diff_dict = {'Healthy': age_diff_h}
    for label in age_diff_dict:
        if label != 'Healthy':
            gse = label.split('|')[-1]  # 根据您的标签格式提取GSE
            if gse not in problematic_gses:
                filtered_age_diff_dict[label] = age_diff_dict[label]
    age_diff_dict = filtered_age_diff_dict

# 绘制箱线图
print("\n绘制年龄差异箱线图...")
plot_age_diff_boxplot(age_diff_dict)

# 输出离群样本信息
if outlier_samples:
    print("\n离群样本信息:")
    outlier_df = pd.DataFrame([
        {'GSE': gse, 'Sample': sample, 'Age_Diff': age_diff_dict[f"{gse_disease_dict.get(gse, 'Unknown')}|{gse}"][sample] 
         if f"{gse_disease_dict.get(gse, 'Unknown')}|{gse}" in age_diff_dict else 'N/A'}
        for gse, samples in outlier_samples.items() 
        for sample in samples
    ])
    print(outlier_df)
    
    # 保存离群样本信息到文件
    # outlier_df.to_csv('outlier_samples.csv', index=False)
    print("离群样本信息已保存到 outlier_samples.csv")

# 保存汇总结果
print("\n保存汇总结果...")
summary_data = []
for label, age_diffs in age_diff_dict.items():
    for age_diff in age_diffs:
        summary_data.append({
            'Group': label,
            'Age_Diff': age_diff
        })

summary_df = pd.DataFrame(summary_data)
# summary_df.to_csv('age_diff_summary.csv', index=False)
print("汇总结果已保存到 age_diff_summary.csv")

# 可选：计算每个GSE的统计信息
stats_data = []
for label, age_diffs in age_diff_dict.items():
    if label != 'Healthy':
        stats_data.append({
            'GSE': label,
            'Mean_Age_Diff': np.mean(age_diffs),
            'Median_Age_Diff': np.median(age_diffs),
            'Std_Age_Diff': np.std(age_diffs),
            'Min_Age_Diff': np.min(age_diffs),
            'Max_Age_Diff': np.max(age_diffs),
            'N_Samples': len(age_diffs)
        })

stats_df = pd.DataFrame(stats_data)
stats_df = stats_df.sort_values('Mean_Age_Diff', ascending=False)
print("\n各GSE统计信息:")
print(stats_df)
# stats_df.to_csv('gse_stats.csv', index=False)

# 如果只想绘制部分GSE（例如前10个），可以使用：
# plot_top_n = 10
# top_gses_to_plot = stats_df.head(plot_top_n)['GSE'].tolist()
# top_age_diff_dict = {'Healthy': age_diff_h}
# for gse in top_gses_to_plot:
#     if gse in age_diff_dict:
#         top_age_diff_dict[gse] = age_diff_dict[gse]
# plot_age_diff_boxplot(top_age_diff_dict)


In [ ]:
gse_results= problematic_gses


In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from tqdm import tqdm
import matplotlib.patches as mpatches


gse_disease_map = adata_age.obs[adata_age.obs['GSE'].isin(top_gses)][['GSE', 'disease_global_check']].drop_duplicates()
gse_disease_dict = dict(zip(gse_disease_map['GSE'], gse_disease_map['disease_global_check']))

# 按疾病类型分组GSE
disease_to_gses = {}
for gse in top_gses:
    disease = gse_disease_dict.get(gse, 'Unknown')
    if disease not in disease_to_gses:
        disease_to_gses[disease] = []
    disease_to_gses[disease].append(gse)

print("按疾病类型分组的GSE:")
for disease, gses in disease_to_gses.items():
    print(f"{disease}: {gses}")

# 批量处理每个GSE数据集
gse_results = {}  # 存储每个GSE的预测结果
problematic_gses = []
outlier_samples = {}

print("\n开始批量处理GSE数据集...")
for gse in top_gses:
    try:
        adata_gse = adata_age[adata_age.obs['GSE']==gse]
        # 计算年龄差异
        pred_df = age_diff_gene_prop(adata_gse, pipeline)
        # 检查异常值
        age_diffs = pred_df['Age_Diff'].values
        Q1 = np.percentile(age_diffs, 25)
        Q3 = np.percentile(age_diffs, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 3 * IQR
        upper_bound = Q3 + 3 * IQR
        
        outliers = pred_df[(pred_df['Age_Diff'] < lower_bound) | (pred_df['Age_Diff'] > upper_bound)]
        if len(outliers) > 0:
            outlier_samples[gse] = outliers.index.tolist()
            print(f"警告: {gse} 中有 {len(outliers)} 个离群样本")
            
            if len(outliers) > len(pred_df) * 0.2:
                problematic_gses.append(gse)
                print(f"⚠️ {gse} 离群点比例过高 ({len(outliers)/len(pred_df)*100:.1f}%)")
        # 存储结果
        gse_results[gse] = pred_df['Age_Diff']     
    except Exception as e:
        print(f"处理 {gse} 时出错: {e}")
        problematic_gses.append(gse)
        continue

print(f"\n处理完成！成功处理 {len(gse_results)} 个GSE，{len(problematic_gses)} 个有问题")

# 排除有问题的GSE
exclude_problematic = True
if exclude_problematic and problematic_gses:
    print(f"\n排除有问题的GSE: {problematic_gses}")
    for gse in problematic_gses:
        if gse in gse_results:
            del gse_results[gse]

# 创建按疾病类型分组的箱线图
def plot_grouped_age_diff_boxplot(age_diff_h, gse_results, disease_to_gses, gse_disease_dict):
    """
    按疾病类型分组绘制年龄差异箱线图
    
    参数:
    - age_diff_h: healthy对照组的年龄差异
    - gse_results: 字典，key为GSE，value为age_diff的Series
    - disease_to_gses: 字典，key为疾病类型，value为该疾病下的GSE列表
    - gse_disease_dict: 字典，key为GSE，value为疾病类型
    """
    
    # 准备数据
    plot_data = []
    
    # 添加Healthy组
    for val in age_diff_h:
        plot_data.append({
            'Age Difference': val,
            'Group': 'Healthy',
            'Disease': 'Healthy',
            'GSE': 'Healthy'
        })
    
    # 按疾病类型顺序添加GSE数据
    all_diseases = sorted([d for d in disease_to_gses.keys() if d != 'Unknown'])
    if 'Unknown' in disease_to_gses:
        all_diseases.append('Unknown')
    
    # 创建颜色映射
    colors = plt.cm.Set3(np.linspace(0, 1, len(all_diseases)))
    disease_color_map = dict(zip(all_diseases, colors))
    
    # 添加每个GSE的数据
    gse_positions = {}  # 记录每个GSE在x轴上的位置
    current_pos = 1  # 从1开始，0留给Healthy
    
    # 先添加Healthy的位置
    gse_positions['Healthy'] = 0
    
    for disease in all_diseases:
        gses_in_disease = disease_to_gses.get(disease, [])
        # 只保留有结果的GSE
        valid_gses = [gse for gse in gses_in_disease if gse in gse_results]
        
        for gse in valid_gses:
            for val in gse_results[gse]:
                plot_data.append({
                    'Age Difference': val,
                    'Group': f"{disease}|{gse}",
                    'Disease': disease,
                    'GSE': gse,
                    'Position': current_pos,
                    'Color': disease_color_map[disease]
                })
            gse_positions[f"{disease}|{gse}"] = current_pos
            current_pos += 1
    
    df_plot = pd.DataFrame(plot_data)
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(1.1 * len(gse_results) + 1, 10))
    # 获取所有组名（按疾病类型排序）
    all_groups = ['Healthy'] + [f"{disease}|{gse}" for disease in all_diseases 
                                for gse in disease_to_gses.get(disease, []) 
                                if gse in gse_results]
    
    # 绘制箱线图
    boxplot = ax.boxplot(
        [df_plot[df_plot['Group'] == group]['Age Difference'].values for group in all_groups],
        positions=range(len(all_groups)),
        widths=0.6,
        patch_artist=True,
        showfliers=False
    )
    
    # 为每个箱子设置颜色
    for i, (box, group) in enumerate(zip(boxplot['boxes'], all_groups)):
        if group == 'Healthy':
            box.set_facecolor('lightgray')
        else:
            disease = group.split('|')[0]
            box.set_facecolor(disease_color_map[disease])
        box.set_alpha(0.7)
    
    # 添加散点图（jittered points）
    for i, group in enumerate(all_groups):
        group_data = df_plot[df_plot['Group'] == group]['Age Difference'].values
        if len(group_data) > 0:
            # 添加随机抖动
            jitter = np.random.normal(0, 0.04, len(group_data))
            ax.scatter([i] * len(group_data) + jitter, group_data, 
                      alpha=0.5, s=20, color='black', zorder=3)
    
    # 设置x轴标签
    x_labels = []
    for group in all_groups:
        if group == 'Healthy':
            x_labels.append('Healthy')
        else:
            gse = group.split('|')[1]
            x_labels.append(gse)
    
    ax.set_xticks(range(len(all_groups)))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    
    # 添加疾病类型的背景色块
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    
    current_pos = 1  # 从1开始（0是Healthy）
    for disease in all_diseases:
        gses_in_disease = [gse for gse in disease_to_gses.get(disease, []) if gse in gse_results]
        n_gses = len(gses_in_disease)
        
        if n_gses > 0:
            # 添加背景色块
            rect = mpatches.Rectangle(
                (current_pos - 0.5, y_min - 0.05 * y_range),
                n_gses,
                0.03 * y_range,
                facecolor=disease_color_map[disease],
                alpha=0.3,
                transform=ax.transData
            )
            ax.add_patch(rect)
            
            # 添加疾病类型标签
            ax.text(
                current_pos + n_gses/2 - 0.5,
                y_min - 0.08 * y_range,
                disease,
                ha='center',
                va='top',
                fontsize=11,
                fontweight='bold',
                color=disease_color_map[disease]
            )
            
            current_pos += n_gses
    
    # 进行显著性检验（每组与Healthy比较）
    healthy_vals = age_diff_h
    
    for i, group in enumerate(all_groups[1:], start=1):  # 从1开始跳过Healthy
        group_vals = df_plot[df_plot['Group'] == group]['Age Difference'].values
        if len(group_vals) > 0:
            try:
                p_value = mannwhitneyu(healthy_vals, group_vals, alternative='two-sided').pvalue
                
                # 添加显著性标记
                if p_value < 0.001:
                    sig = '***'
                elif p_value < 0.01:
                    sig = '**'
                elif p_value < 0.05:
                    sig = '*'
                else:
                    sig = 'ns'
                
                # 在箱子上方添加显著性标记
                ax.text(i, y_max + 0.02 * y_range, sig, 
                       ha='center', va='bottom', fontsize=12, fontweight='bold')
            except:
                pass
    
    # 设置标题和标签
    ax.set_ylabel('Age Difference (Predicted - Actual)', fontsize=12, fontweight='bold')
    ax.set_title('Age Difference by Disease Type and GSE', fontsize=14, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 添加图例
    legend_patches = [mpatches.Patch(color='lightgray', label='Healthy', alpha=0.7)]
    for disease, color in disease_color_map.items():
        if disease != 'Unknown' or any(gse in gse_results for gse in disease_to_gses.get(disease, [])):
            legend_patches.append(mpatches.Patch(color=color, label=disease, alpha=0.7))
    
    ax.legend(handles=legend_patches, loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()
    
    return df_plot, gse_positions



In [ ]:
# 创建按疾病类型分组的箱线图
def plot_grouped_age_diff_boxplot(age_diff_h, gse_results, disease_to_gses, gse_disease_dict):
    """
    按疾病类型分组绘制年龄差异箱线图
    
    参数:
    - age_diff_h: healthy对照组的年龄差异
    - gse_results: 字典，key为GSE，value为age_diff的Series
    - disease_to_gses: 字典，key为疾病类型，value为该疾病下的GSE列表
    - gse_disease_dict: 字典，key为GSE，value为疾病类型
    """
    
    # 准备数据
    plot_data = []
    
    # 添加Healthy组
    for val in age_diff_h:
        plot_data.append({
            'Age Difference': val,
            'Group': 'Healthy',
            'Disease': 'Healthy',
            'GSE': 'Healthy'
        })
    
    # 按疾病类型顺序添加GSE数据
    all_diseases = sorted([d for d in disease_to_gses.keys() if d != 'Unknown'])
    if 'Unknown' in disease_to_gses:
        all_diseases.append('Unknown')
    
    # 创建颜色映射
    colors = plt.cm.Set3(np.linspace(0, 1, len(all_diseases)))
    disease_color_map = dict(zip(all_diseases, colors))
    
    # 添加每个GSE的数据
    gse_positions = {}  # 记录每个GSE在x轴上的位置
    current_pos = 1  # 从1开始，0留给Healthy
    
    # 先添加Healthy的位置
    gse_positions['Healthy'] = 0
    
    for disease in all_diseases:
        gses_in_disease = disease_to_gses.get(disease, [])
        # 只保留有结果的GSE
        valid_gses = [gse for gse in gses_in_disease if gse in gse_results]
        
        for gse in valid_gses:
            for val in gse_results[gse]:
                plot_data.append({
                    'Age Difference': val,
                    'Group': f"{disease}|{gse}",
                    'Disease': disease,
                    'GSE': gse,
                    'Position': current_pos,
                    'Color': disease_color_map[disease]
                })
            gse_positions[f"{disease}|{gse}"] = current_pos
            current_pos += 1
    
    df_plot = pd.DataFrame(plot_data)
    
    # 创建图形 - 调整尺寸使柱子更细长
    # 宽度变小 (0.8 * GSE数量 + 1)，高度变大 (12)
    fig, ax = plt.subplots(figsize=(0.6 * len(gse_results) + 0.5, 8))
    
    # 获取所有组名（按疾病类型排序）
    all_groups = ['Healthy'] + [f"{disease}|{gse}" for disease in all_diseases 
                                for gse in disease_to_gses.get(disease, []) 
                                if gse in gse_results]
    
    # 绘制箱线图 - 减小箱子宽度
    boxplot = ax.boxplot(
        [df_plot[df_plot['Group'] == group]['Age Difference'].values for group in all_groups],
        positions=range(len(all_groups)),
        widths=0.4,  # 减小箱子宽度，从0.6改为0.4
        patch_artist=True,
        showfliers=False
    )
    
    # 为每个箱子设置颜色
    for i, (box, group) in enumerate(zip(boxplot['boxes'], all_groups)):
        if group == 'Healthy':
            box.set_facecolor('lightgray')
        else:
            disease = group.split('|')[0]
            box.set_facecolor(disease_color_map[disease])
        box.set_alpha(0.7)
        # 可选：减小箱线图的线宽
        box.set_linewidth(0.8)
    
    # 调整须线的宽度
    for whisker in boxplot['whiskers']:
        whisker.set_linewidth(0.8)
    for cap in boxplot['caps']:
        cap.set_linewidth(0.8)
    for median in boxplot['medians']:
        median.set_linewidth(1)
    
    # 添加散点图（jittered points）- 调整点的大小
    for i, group in enumerate(all_groups):
        group_data = df_plot[df_plot['Group'] == group]['Age Difference'].values
        if len(group_data) > 0:
            # 添加随机抖动
            jitter = np.random.normal(0, 0.02, len(group_data))  # 减小抖动幅度
            ax.scatter([i] * len(group_data) + jitter, group_data, 
                      alpha=0.4, s=15, color='black', zorder=3)  # 减小点的大小
    
    # 设置x轴标签
    x_labels = []
    for group in all_groups:
        if group == 'Healthy':
            x_labels.append('Healthy')
        else:
            gse = group.split('|')[1]
            x_labels.append(gse)
    
    ax.set_xticks(range(len(all_groups)))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)  # 减小字体
    
    # 添加疾病类型的背景色块
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    
    current_pos = 1  # 从1开始（0是Healthy）
    for disease in all_diseases:
        gses_in_disease = [gse for gse in disease_to_gses.get(disease, []) if gse in gse_results]
        n_gses = len(gses_in_disease)
        
        if n_gses > 0:
            # 添加背景色块 - 调整位置和大小
            rect = mpatches.Rectangle(
                (current_pos - 0.5, y_min - 0.03 * y_range),
                n_gses,
                0.02 * y_range,
                facecolor=disease_color_map[disease],
                alpha=0.3,
                transform=ax.transData
            )
            ax.add_patch(rect)
            
            # 添加疾病类型标签 - 调整位置
            ax.text(
                current_pos + n_gses/2 - 0.5,
                y_min - 0.05 * y_range,
                disease,
                ha='center',
                va='top',
                fontsize=10,  # 减小字体
                fontweight='bold',
                color=disease_color_map[disease]
            )
            
            current_pos += n_gses
    
    # 进行显著性检验（每组与Healthy比较）
    healthy_vals = age_diff_h
    
    # 获取更新后的y轴范围
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    
    for i, group in enumerate(all_groups[1:], start=1):  # 从1开始跳过Healthy
        group_vals = df_plot[df_plot['Group'] == group]['Age Difference'].values
        if len(group_vals) > 0:
            try:
                p_value = mannwhitneyu(healthy_vals, group_vals, alternative='two-sided').pvalue
                
                # 添加显著性标记
                if p_value < 0.001:
                    sig = '***'
                elif p_value < 0.01:
                    sig = '**'
                elif p_value < 0.05:
                    sig = '*'
                else:
                    sig = 'ns'
                
                # 在箱子上方添加显著性标记 - 调整位置
                ax.text(i, y_max + 0.01 * y_range, sig, 
                       ha='center', va='bottom', fontsize=11, fontweight='bold')
            except:
                pass
    
    # 设置标题和标签
    ax.set_ylabel('Age Difference (Predicted - Actual)', fontsize=11, fontweight='bold')
    ax.set_title('Age Difference by Disease Type and GSE', fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.grid(True, alpha=0.2, axis='y')  # 降低网格线透明度
    
    # 添加图例 - 调整位置
    legend_patches = [mpatches.Patch(color='lightgray', label='Healthy', alpha=0.7)]
    for disease, color in disease_color_map.items():
        if disease != 'Unknown' or any(gse in gse_results for gse in disease_to_gses.get(disease, [])):
            legend_patches.append(mpatches.Patch(color=color, label=disease, alpha=0.7))
    
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9, framealpha=0.8)
    
    # 调整布局
    plt.subplots_adjust(bottom=0.15, top=0.95)  # 为底部标签留出更多空间
    plt.tight_layout()
    plt.show()
    
    return df_plot, gse_positions
    

In [ ]:

# 调用绘图函数
print("\n绘制按疾病类型分组的箱线图...")
df_plot, gse_positions = plot_grouped_age_diff_boxplot(
    age_diff_h, 
    gse_results, 
    disease_to_gses, 
    gse_disease_dict
)

# 输出统计信息
print("\n各疾病类型的统计信息:")
for disease in disease_to_gses:
    valid_gses = [gse for gse in disease_to_gses[disease] if gse in gse_results]
    if valid_gses:
        all_diffs = []
        for gse in valid_gses:
            all_diffs.extend(gse_results[gse].values)
        
        print(f"\n{disease}:")
        print(f"  GSE数量: {len(valid_gses)}")
        print(f"  样本总数: {len(all_diffs)}")
        print(f"  平均年龄差异: {np.mean(all_diffs):.2f}")
        print(f"  中位年龄差异: {np.median(all_diffs):.2f}")
        print(f"  标准差: {np.std(all_diffs):.2f}")

# 保存结果
summary_data = []
for group in df_plot['Group'].unique():
    group_data = df_plot[df_plot['Group'] == group]
    for _, row in group_data.iterrows():
        summary_data.append({
            'Group': row['Group'],
            'Disease': row['Disease'],
            'GSE': row['GSE'],
            'Age_Diff': row['Age Difference']
        })

summary_df = pd.DataFrame(summary_data)
#summary_df.to_csv('age_diff_grouped_summary.csv', index=False)
print("\n汇总结果已保存到 age_diff_grouped_summary.csv")

# 保存离群样本信息
if outlier_samples:
    outlier_df = pd.DataFrame([
        {'GSE': gse, 'Sample': sample}
        for gse, samples in outlier_samples.items() 
        for sample in samples
    ])
    # outlier_df.to_csv('outlier_samples.csv', index=False)
    print("离群样本信息已保存到 outlier_samples.csv")
    

In [ ]:
################# 统计下不同的 thresh 筛选到的 细胞数





In [ ]:
import os
from pathlib import Path
import pandas as pd
dir ='external_inputs/pickle'
path = Path(dir)

tsv_files = list(path.rglob('*.tsv'))
tsv_files


In [ ]:
# 使用列表推导式一次性读取所有文件
meta_df_new = pd.concat([
    pd.read_csv(file, sep='\t')
      .assign(file=str(file))
      .assign(thresh=file.stem.split('_')[-1])
      .assign(batch='3th' if 'Third' in str(file) else '1st')
    for file in tsv_files
], ignore_index=True)


In [ ]:
# meta_df_new['file'].to_list()[0:4]


In [ ]:
# meta_df_summary['sample'].value_counts().head(40)


In [ ]:
meta_df_new = meta_df_new[['gse', 'gsm', 'sample', 'species', 'lineage', 'type', 'qc_cn',
       'cell_num', 'clus_num', 'file', 'thresh', 'batch']]


In [ ]:
Endo_agingGenes_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/small_file/AgeAnno_DEGs_endo.tsv',sep='\t')
Strom_agingGenes_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/small_file/AgeAnno_DEGs_stromal.tsv',sep='\t')
Fib_agingGenes_df = pd.read_csv('./data/scalex/fibro/human/age_prediction/small_file/AgeAnno_DEGs_fibro.tsv',sep='\t')
Endo_agingGenes = list(set(Endo_agingGenes_df['gene'].to_list()))
Fib_agingGenes = list(set(Fib_agingGenes_df['gene'].to_list()))
Strom_agingGenes = list(set(Strom_agingGenes_df['gene'].to_list()))



In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

dir ='./data/scalex/fibro/human/age_prediction/EN_Exp_CT/'
path = Path(dir)
model_files = list(path.rglob('*.pkl'))

for file in model_files:
    pipeline = joblib.load(file)
    non0_fea = pipeline['nonzero_features']
    non0_gene_fea = non0_fea[non0_fea['FeatureType']=='Gene Expression']
    important_fea = non0_fea[np.abs(non0_fea['Weight']) >= 0.5]




rows = []
for file in model_files:
    pipeline = joblib.load(file)
    non0_fea = pipeline['nonzero_features']
    gene_fea = non0_fea[non0_fea['FeatureType'] == 'Gene Expression']
    important_fea = gene_fea[np.abs(gene_fea['Weight']) >= 0.5]
    
    path_str = str(file)
    parts = path_str.split('/')
    lineage = parts[-4]
    type_ = parts[-3]
    nFea = parts[-2]
    
    genes_not0 = gene_fea['Feature'].apply(lambda x: x.split('_')[-1]).tolist()
    genes_i = important_fea['Feature'].apply(lambda x: x.split('_')[-1]).tolist()
    
    row = {
        'path': path_str,
        'lineage': lineage,
        'type': type_,
        'nFea': nFea,
        'nFea_not0': len(gene_fea),
        'nFea_i': len(important_fea),
        'genes_not0': ','.join(genes_not0),
        'genes_i': ','.join(genes_i)
    }
    rows.append(row)

model_df = pd.DataFrame(rows)
# 新增列：基因列表和长度
model_df['geneN_not0'] = model_df['genes_not0'].apply(lambda x: x.split(',') if x else [])
model_df['geneN_i'] = model_df['genes_i'].apply(lambda x: x.split(',') if x else [])
model_df['geneN_not0_len'] = model_df['geneN_not0'].apply(len)
model_df['geneN_i_len'] = model_df['geneN_i'].apply(len)

# 查看结果
print(model_df[['nFea', 'nFea_not0', 'nFea_i', 'geneN_not0_len', 'geneN_i_len']])
# model_df.to_csv('./data/scalex/fibro/human/age_prediction/EN_Exp_CT/data/EN_Gene_CT_benchmark_gene_table.tsv',sep='\t',index=False)

##### read in the gag gene file 
gag = pd.read_csv('external_inputs/global_aging_genes.tsv',sep='\t')  # 把 mouse基因转成 human 基因
gag['human_genes'] = gag['global_aging_genes'].str.upper()
gag_genes = gag['human_genes'].to_list()


In [ ]:
model_df['lineage'].unique()


In [ ]:
lineages = model_df['lineage'].unique()
lineages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
import math


# gag_set = set(gag_genes)
gag_set = set(Fib_agingGenes)
# 获取所有唯一的 lineage
lineages = model_df['lineage'].unique()

# 为每个 lineage 创建一张大图
for lineage in ['Fibro']:   
    # 筛选当前 lineage 的数据
    lineage_df = model_df[model_df['lineage'] == lineage].copy()
    
    # 获取该 lineage 中所有唯一的 nFea 值
    nFea_values = sorted(lineage_df['nFea'].unique(), key=lambda x: int(x) if x.isdigit() else x)
    
    # 创建图形 - 4行6列，共24个子图
    fig, axes = plt.subplots(4, 6, figsize=(30, 20))
    fig.suptitle(f'Lineage: {lineage} - GeneSet Inter with Fibro aging genes', fontsize=24, y=0.98)
    
    # 展平 axes 以便于索引
    axes_flat = axes.flatten()
    
    # 为每个子图编号
    subplot_idx = 0
    
    # 遍历所有 type 和 nFea 的组合
    for type_name in ['with0', 'without0']:  # 假设 type 列中有 'with0' 和 'without0'
        for nFea in nFea_values:
            # 获取当前组合的行
            row = lineage_df[(lineage_df['type'] == type_name) & 
                            (lineage_df['nFea'] == nFea)]
            if len(row) == 0:
                # 如果没有数据，创建一个空的子图
                axes_flat[subplot_idx].text(0.5, 0.5, f'No data\n{type_name}\nnFea={nFea}', 
                                           ha='center', va='center', fontsize=12)
                axes_flat[subplot_idx].set_xlim(0, 1)
                axes_flat[subplot_idx].set_ylim(0, 1)
                axes_flat[subplot_idx].axis('off')
                subplot_idx += 1
                continue
            row = row.iloc[0]  # 获取第一行（应该只有一行）
            # 获取基因列表
            not0_genes = set(row['geneN_not0']) if isinstance(row['geneN_not0'], list) else set()
            i_genes = set(row['geneN_i']) if isinstance(row['geneN_i'], list) else set()
            # 计算交集
            not0_gag_intersection = not0_genes & gag_set
            i_gag_intersection = i_genes & gag_set
            # 准备韦恩图数据
            # 集合1: not0_genes
            # 集合2: i_genes
            # 集合3: gag_set
            # 计算各个区域的数值
            only_not0 = len(not0_genes - i_genes - gag_set)
            only_i = len(i_genes - not0_genes - gag_set)
            only_gag = len(gag_set - not0_genes - i_genes)
            not0_i_only = len((not0_genes & i_genes) - gag_set)
            not0_gag_only = len((not0_genes & gag_set) - i_genes)
            i_gag_only = len((i_genes & gag_set) - not0_genes)
            all_three = len(not0_genes & i_genes & gag_set)
            
            # 创建韦恩图
            venn_labels = {
                '100': only_not0,
                '010': only_i,
                '001': only_gag,
                '110': not0_i_only,
                '101': not0_gag_only,
                '011': i_gag_only,
                '111': all_three
            }
            # 绘制韦恩图
            v = venn3(subsets=venn_labels, 
                     set_labels=(f'not0\n({len(not0_genes)})', 
                                f'i\n({len(i_genes)})', 
                                f'GAG\n({len(gag_set)})'),
                     ax=axes_flat[subplot_idx])
            # 设置标题
            axes_flat[subplot_idx].set_title(f'{type_name} | nFea={nFea}\n'
                                            f'not0∩GAG: {len(not0_gag_intersection)} | '
                                            f'i∩GAG: {len(i_gag_intersection)}',
                                            fontsize=10)
            subplot_idx += 1
    
    # 隐藏多余的子图（如果有）
    for idx in range(subplot_idx, len(axes_flat)):
        axes_flat[idx].axis('off')
    
    # 调整布局
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    
    # 保存图片
    # plt.savefig(f'venn_diagram_{lineage}.png', dpi=300, bbox_inches='tight')
    plt.show()



In [ ]:
aging_genes = pd.read_csv('external_inputs/Aging_genes_combine.csv',sep='\t')


In [ ]:
gag_genes = set(aging_genes['gene'].to_list()+ gag_genes)


In [ ]:
len(Endo_agingGenes), len(Fib_agingGenes),len(Strom_agingGenes)

In [ ]:
nFea_values


In [ ]:
# 简化版本 - 只显示 not0 和 i 与 GAG 的交集
for lineage in ['Fibro']:  # lineages
    lineage_df = model_df[model_df['lineage'] == lineage]
    nFea_values = sorted(lineage_df['nFea'].unique(), key=lambda x: int(x) if x.isdigit() else x)
    
    fig, axes = plt.subplots(2, 6, figsize=(30, 10))
    fig.suptitle(f'Lineage: {lineage} - GAG Gene Intersections', fontsize=24, y=0.98)
    axes_flat = axes.flatten()
    
    subplot_idx = 0
    for type_name in ['with0']:
        for nFea in nFea_values:
            row = lineage_df[(lineage_df['type'] == type_name) & 
                           (lineage_df['nFea'] == nFea)]
            
            if len(row) == 0:
                axes_flat[subplot_idx].text(0.5, 0.5, f'No data\n{type_name}\nnFea={nFea}', 
                                           ha='center', va='center')
                axes_flat[subplot_idx].axis('off')
                subplot_idx += 1
                continue
            
            row = row.iloc[0]
            not0_genes = set(row['geneN_not0']) if isinstance(row['geneN_not0'], list) else set()
            i_genes = set(row['geneN_i']) if isinstance(row['geneN_i'], list) else set()
            
            # 简化版韦恩图 - 只显示两个集合
            from matplotlib_venn import venn2
            
            # not0 vs GAG
            venn2([not0_genes, gag_set],
                  set_labels=(f'not0\n({len(not0_genes)})', f'GAG\n({len(gag_set)})'),
                  ax=axes_flat[subplot_idx])
            axes_flat[subplot_idx].set_title(f'{type_name} | nFea={nFea}\n'
                                            f'not0∩GAG: {len(not0_genes & gag_set)}')
            
            subplot_idx += 1
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    # plt.savefig(f'venn_simple_{lineage}.png', dpi=300, bbox_inches='tight')
    plt.show()
    

In [ ]:
Inter_genes = set(not0_genes) & set(Fib_agingGenes)


In [ ]:
### 查看weight > 0.05的feature的基因
important_fea = coef_df_gene[np.abs(coef_df_gene['Weight']) >= 0.05]
important_fea     ### Then save as 'age_important_fea.csv'
selected_col = important_fea['Feature']
selected_genes_cell_type =important_fea['Celltype'].to_list()
# important_fea['subtype'].value_counts()
# 按细胞类型拆分
from collections import defaultdict
# 用defaultdict分组
groups = defaultdict(list)
for item in selected_col:
    cell_type = "_".join(item.split('_')[:-1])# 按_分开
    groups[cell_type].append(item)

# 转成普通字典
groups = dict(groups)
# 输出，变成一个dict,key为不同的 Celltype, 每个key对应的 value 为 faeture
for k, v in groups.items():
    print(f"{k}: {v}")
    

In [ ]:
###### 挑选重要的感兴趣的 Celltype的 important 的feature做功能分析
iFibro_IGFBP7_pan = [gene.split('_')[1] for gene in groups['eFibro_CD74_pan']]    ### 先看这个出现频次最多的 Celltype
# 功能富集
import gseapy as gp
# Enrichr富集分析
enr = gp.enrichr(
    gene_list=iFibro_IGFBP7_pan,
    gene_sets=['GO_Biological_Process_2021', 'KEGG_2021_Human'],
    organism='Human',
    outdir='enrichr_results',  # 输出目录
    cutoff=0.1  # 显著性阈值
)
df = enr.results  # 或者 enr.res2d
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# 只保留显著结果
df = df[df['Adjusted P-value'] < 0.05]
# 排序，取前20个
df = df.sort_values('Adjusted P-value').head(10)
df['minus_log10_p'] = -np.log10(df['P-value'])
df['GeneRatio'] = df['Overlap'].apply(lambda x: float(x.split('/')[0]) / float(x.split('/')[1]))
# 画气泡图
plt.figure(figsize=(12, 7))  # 主图框放大，宽度和高度都调大
# 画图
sns.scatterplot(
    data=df,
    x='minus_log10_p',
    y='Term',
    size='GeneRatio',  # 如果你用了基因比例作为大小
    hue='Adjusted P-value',
    palette='viridis_r',
    sizes=(50, 300),
    edgecolor='black'
)
plt.xlabel('-Log10(P-value)', fontsize=12)  # 坐标轴标签字体调小
plt.ylabel('Term', fontsize=12)
# 坐标轴刻度字体调小
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.title('Age-related gene enrichment in eFibro_CD74_pan', fontsize=14)
plt.legend(title='Gene Overlap', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, title_fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
def age_relatied_gene_enrich(cell_type, groups, cutoff):
    """
    cell_type: 富集年龄相关基因所属细胞类型；
    groups: en模型重要特征前50按细胞类型分类构成的字典;
    cutoff: 通路富集阈值
    """
    # 挑选某cell_type的基因
    gene_list = [gene.split('_')[-1] for gene in groups[cell_type]]
    print(gene_list)

    # Enrichr富集分析
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=['GO_Biological_Process_2021', 'KEGG_2021_Human'],
        organism='Human',
        outdir='enrichr_results',  # 输出目录
        cutoff=cutoff  # 显著性阈值
    )
    df = enr.results
    df = df[df['Adjusted P-value'] < 0.05]
    print(df)
    # 排序，取前10个
    df = df.sort_values('Adjusted P-value').head(10)
    df['minus_log10_p'] = -np.log10(df['P-value'])
    df['GeneRatio'] = df['Overlap'].apply(lambda x: float(x.split('/')[0]) / float(x.split('/')[1]))
    plt.figure(figsize=(12, 7))  # 主图框放大，宽度和高度都调大
    sns.scatterplot(
        data=df,
        x='minus_log10_p',
        y='Term',
        size='GeneRatio',  # 如果你用了基因比例作为大小
        hue='Adjusted P-value',
        palette='viridis_r',
        sizes=(50, 300),
        edgecolor='black'
    )
    plt.xlabel('-Log10(P-value)', fontsize=12)  # 坐标轴标签字体调小
    plt.ylabel('Term', fontsize=12)
    # 坐标轴刻度字体调小
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=12)
    plt.title(f'Age-associated gene enrichment in {cell_type}', fontsize=14)
    plt.legend(title='Gene Overlap', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, title_fontsize=12)
    plt.tight_layout()
    plt.show()
    

In [ ]:
stable_prop = coef_df[coef_df["FeatureType"] == "Cell Proportion"].copy()
stable_expr = coef_df[~coef_df["Feature"].str.endswith("_ratio")].copy()
print("稳定表达特征数量:", stable_expr.shape[0])
stable_expr = stable_expr.rename(columns={
    "Part1": "CellType",
    "Part2": "Gene"
})
stable_expr.head()

######### 也就是查看下每个基因在多少个 Celltype中出现了
gene_cell_counts = stable_expr.groupby("Gene")["Celltype"].nunique().sort_values(ascending=False)
# 按基因统计显著的细胞类型数量和具体细胞类型
gene_cell_summary = stable_expr.groupby("Gene")["Celltype"].agg(
    n_celltypes="nunique",          # 统计数量
    celltypes=lambda x: list(x)     # 统计具体细胞类型
).reset_index()
gene_cell_summary = gene_cell_summary.sort_values(by="n_celltypes", ascending=False)

############ 根据上面的 gene_cell_summary画每个基因在多少个Celltype 中有 weight 的 0-1 矩阵( weight权重变成 0-1 化)
import seaborn as sns
import matplotlib.pyplot as plt
# 构建基因 × 细胞类型的二值矩阵
gene_cell_matrix = stable_expr.drop_duplicates(subset=["Gene","Celltype"])
gene_cell_matrix['Stable'] = 1
gene_cell_matrix = gene_cell_matrix.pivot(index='Gene', columns='Celltype', values='Stable').fillna(0)
# 对行排序：按基因稳定的细胞类型数量
gene_cell_matrix['n_celltypes'] = gene_cell_matrix.sum(axis=1)
gene_cell_matrix = gene_cell_matrix.sort_values('n_celltypes', ascending=False)
gene_cell_matrix = gene_cell_matrix.drop(columns='n_celltypes')  # 排序完可以删掉辅助列
# 对列排序：按细胞类型稳定基因数量
col_order = gene_cell_matrix.sum(axis=0).sort_values(ascending=False).index
gene_cell_matrix = gene_cell_matrix[col_order]
# 选前20个基因可视化
top_genes = gene_cell_matrix.head(20).index
matrix_plot = gene_cell_matrix.loc[top_genes]
plt.figure(figsize=(10, 6))
sns.heatmap(matrix_plot, cmap="Blues", cbar_kws={'label':'Stable'}, linewidths=.5, linecolor='gray')
plt.xlabel("Celltype")
plt.ylabel("Gene")
plt.title("Stable Gene Across Celltype(Sorted)")
plt.tight_layout()
plt.show()


In [ ]:
########### 挑选重要的基因做 KEGG 通路富集, 可视化方式有 barplot 或者网络图
# -----------------------
# KEGG 富集分析 + 柱状图 + 网络图 + 输出通路-基因表格
# -----------------------
###### GQA( Global query attention)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from gseapy import enrichr
from matplotlib.colors import Normalize

# -----------------------
# 1️⃣ 使用 stable_expr 数据
# -----------------------
all_genes = stable_expr['Gene'].unique().tolist()

# -----------------------
# 2️⃣ KEGG 富集分析
# -----------------------
enr_kegg = enrichr(
    gene_list=all_genes,
    gene_sets=['KEGG_2021_Human'],  # KEGG库
    organism='Human',
    cutoff=1.0,
    outdir=None
)

# 提取 top 10 富集结果
top_kegg = enr_kegg.results.sort_values('Adjusted P-value').head(10).copy()
top_kegg['neg_log10FDR'] = -np.log10(top_kegg['Adjusted P-value'] + 1e-12)  # 防止0
top_kegg['GeneCount'] = top_kegg['Genes'].apply(lambda x: len(x.split(';')))

# -----------------------
# 3️⃣ 输出通路-基因表格（每条通路一行，Genes 保留分号分隔）
# -----------------------
kegg_pathway_gene_df = top_kegg[['Term', 'Genes']].copy()
kegg_pathway_gene_df.rename(columns={'Term':'Pathway'}, inplace=True)
# 保存为 CSV
# kegg_pathway_gene_df.to_csv("top_kegg_pathway_gene_single_row.csv", index=False)
# print("KEGG Pathway-Gene table (single row per pathway):")
# print(kegg_pathway_gene_df)
# -----------------------
# 4️⃣ 设置颜色映射范围（浅橙梯度）
# -----------------------
min_val = top_kegg['neg_log10FDR'].min()
max_val = top_kegg['neg_log10FDR'].max()
vmax = max_val * 0.8  # 压缩上限
norm = Normalize(vmin=min_val, vmax=vmax)
def map_color_orange(val):
    """映射 -log10(FDR) 到浅橙梯度"""
    c = min(val / vmax, 1.0)
    return plt.cm.Oranges(0.4 + 0.4 * c)

colors_bar = [map_color_orange(v) for v in top_kegg['neg_log10FDR']]
# -----------------------
# 5️⃣ 柱状图
# -----------------------
plt.figure(figsize=(10,6))
sns.barplot(
    x='Combined Score',
    y='Term',
    data=top_kegg,
    dodge=False,
    palette=colors_bar
)
plt.xlabel("Combined Score")
plt.ylabel("KEGG Term")
plt.title("Top Enriched KEGG Terms (All Stable Genes)")
sm = plt.cm.ScalarMappable(cmap="Oranges", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm)
cbar.set_label('-log10(FDR)', rotation=270, labelpad=15)
plt.tight_layout()
plt.show()


In [ ]:
stable_prop = coef_df[coef_df["FeatureType"] == "Cell Proportion"].copy()
stable_expr = coef_df[~coef_df["Feature"].str.endswith("_ratio")].copy()
print("稳定表达特征数量:", stable_expr.shape[0])
stable_expr = stable_expr.rename(columns={
    "Part1": "CellType",
    "Part2": "Gene"
})
stable_expr.head()

######### 也就是查看下每个基因在多少个 Celltype中出现了
gene_cell_counts = stable_expr.groupby("Gene")["Celltype"].nunique().sort_values(ascending=False)
# 按基因统计显著的细胞类型数量和具体细胞类型
gene_cell_summary = stable_expr.groupby("Gene")["Celltype"].agg(
    n_celltypes="nunique",          # 统计数量
    celltypes=lambda x: list(x)     # 统计具体细胞类型
).reset_index()
gene_cell_summary = gene_cell_summary.sort_values(by="n_celltypes", ascending=False)

############ 根据上面的 gene_cell_summary画每个基因在多少个Celltype 中有 weight 的 0-1 矩阵( weight权重变成 0-1 化)
import seaborn as sns
import matplotlib.pyplot as plt
# 构建基因 × 细胞类型的二值矩阵
gene_cell_matrix = stable_expr.drop_duplicates(subset=["Gene","Celltype"])
gene_cell_matrix['Stable'] = 1
gene_cell_matrix = gene_cell_matrix.pivot(index='Gene', columns='Celltype', values='Stable').fillna(0)
# 对行排序：按基因稳定的细胞类型数量
gene_cell_matrix['n_celltypes'] = gene_cell_matrix.sum(axis=1)
gene_cell_matrix = gene_cell_matrix.sort_values('n_celltypes', ascending=False)
gene_cell_matrix = gene_cell_matrix.drop(columns='n_celltypes')  # 排序完可以删掉辅助列
# 对列排序：按细胞类型稳定基因数量
col_order = gene_cell_matrix.sum(axis=0).sort_values(ascending=False).index
gene_cell_matrix = gene_cell_matrix[col_order]
# 选前20个基因可视化
top_genes = Inter_genes   # gene_cell_matrix.head(20).index
matrix_plot = gene_cell_matrix.loc[top_genes]
plt.figure(figsize=(10, 10))
sns.heatmap(matrix_plot, cmap="Blues", cbar_kws={'label':'Stable'}, linewidths=.5, linecolor='gray')
plt.xlabel("Celltype")
plt.ylabel("Gene")
plt.title("InteGenes(Fibro) Celltype(Sorted)")
plt.tight_layout()
plt.show()


In [ ]:
Inter

In [ ]:
############ 只关注那 91个 intersection的基因做 KEGG分析
########### 挑选重要的基因做 KEGG 通路富集, 可视化方式有 barplot 或者网络图
# -----------------------
# KEGG 富集分析 + 柱状图 + 网络图 + 输出通路-基因表格
# -----------------------
###### GQA( Global query attention)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from gseapy import enrichr
from matplotlib.colors import Normalize

# -----------------------
# 1️⃣ 使用 stable_expr 数据
# -----------------------
all_genes = list(Inter_genes)

# -----------------------
# 2️⃣ KEGG 富集分析
# -----------------------
enr_kegg = enrichr(
    gene_list=all_genes,
    gene_sets=['KEGG_2021_Human'],  # KEGG库
    organism='Human',
    cutoff=1.0,
    outdir=None
)

# 提取 top 10 富集结果
top_kegg = enr_kegg.results.sort_values('Adjusted P-value').head(10).copy()
top_kegg['neg_log10FDR'] = -np.log10(top_kegg['Adjusted P-value'] + 1e-12)  # 防止0
top_kegg['GeneCount'] = top_kegg['Genes'].apply(lambda x: len(x.split(';')))

# -----------------------
# 3️⃣ 输出通路-基因表格（每条通路一行，Genes 保留分号分隔）
# -----------------------
kegg_pathway_gene_df = top_kegg[['Term', 'Genes']].copy()
kegg_pathway_gene_df.rename(columns={'Term':'Pathway'}, inplace=True)
# 保存为 CSV
# kegg_pathway_gene_df.to_csv("top_kegg_pathway_gene_single_row.csv", index=False)
# print("KEGG Pathway-Gene table (single row per pathway):")
# print(kegg_pathway_gene_df)
# -----------------------
# 4️⃣ 设置颜色映射范围（浅橙梯度）
# -----------------------
min_val = top_kegg['neg_log10FDR'].min()
max_val = top_kegg['neg_log10FDR'].max()
vmax = max_val * 0.8  # 压缩上限
norm = Normalize(vmin=min_val, vmax=vmax)
def map_color_orange(val):
    """映射 -log10(FDR) 到浅橙梯度"""
    c = min(val / vmax, 1.0)
    return plt.cm.Oranges(0.4 + 0.4 * c)

colors_bar = [map_color_orange(v) for v in top_kegg['neg_log10FDR']]
# -----------------------
# 5️⃣ 柱状图
# -----------------------
plt.figure(figsize=(10,6))
sns.barplot(
    x='Combined Score',
    y='Term',
    data=top_kegg,
    dodge=False,
    palette=colors_bar
)
plt.xlabel("Combined Score")
plt.ylabel("KEGG Term")
plt.title("Top Enriched KEGG Terms (All Stable Genes)")
sm = plt.cm.ScalarMappable(cmap="Oranges", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm)
cbar.set_label('-log10(FDR)', rotation=270, labelpad=15)
plt.tight_layout()
plt.show()


In [ ]:
for gene in all_genes:
    print(gene)
    

In [ ]:
# -----------------------
# 6️⃣ 网络图（优化布局）
# -----------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from gseapy import enrichr
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import networkx as nx
# 先关闭所有现有图形
plt.close('all')
top_kegg['Term'].to_list()
G = nx.Graph()
for _, row in top_kegg.iterrows():
    pathway = row['Term']
    genes_in_pathway = [g.strip() for g in row['Genes'].split(';') if g.strip()]
    neg_log10FDR = row['neg_log10FDR']
    gene_count = len(genes_in_pathway)
    # KEGG pathway 节点大小 = 富集基因数 * 150
    G.add_node(pathway, type='pathway', size=gene_count*150, color=neg_log10FDR)
    # 基因节点固定大小和颜色
    for gene in genes_in_pathway:
        if gene not in G:
            G.add_node(gene, type='gene', size=200, color=0.3)
        G.add_edge(gene, pathway)

# plt.figure(figsize=(14,10))
# spring_layout 调整参数
pos = nx.spring_layout(G, k=0.6, iterations=200, seed=42)
node_sizes = [G.nodes[n]['size'] for n in G.nodes()]
node_colors = []
for n in G.nodes():
    node_attr = G.nodes[n]
    if node_attr['type'] == 'gene':
        node_colors.append('#C1A3DA')  # 基因节点浅紫
    else:
        node_colors.append(map_color_orange(node_attr['color']))  # pathway节点浅橙梯度

# 绘制网络图
# 创建新的图形和坐标轴
fig, ax = plt.subplots(figsize=(14, 10))
# 使用 ax 参数
nx.draw(G, pos, with_labels=True, node_color=node_colors, 
        node_size=node_sizes, font_size=8, edge_color='gray',ax =ax)
plt.title("Gene-KEGG Pathway Network (Top KEGG Terms, color = -log10(FDR))")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 假设已有 meta_df_new 数据框
# meta_df_new = pd.read_csv('your_file.csv')  # 如果需要从文件读取

# 定义要过滤的细胞数阈值
cell_thresholds = [50, 100, 150, 200]

# 创建一个字典来存储不同阈值下的结果
summary_dfs = {}

for threshold in cell_thresholds:
    # 过滤掉细胞数少于阈值的行
    filtered_df = meta_df_new_dedup[meta_df_new_dedup['cell_num'] >= threshold].copy()
    # 按 lineage 和 thresh 分组计算细胞总数
    summary_df = filtered_df.groupby(['lineage', 'thresh'])['cell_num'].sum().reset_index()
    summary_df.columns = ['lineage', 'thresh', f'total_cell_num_th{threshold}']
    # 存储结果
    summary_dfs[threshold] = summary_df

# 合并所有阈值的结果
final_summary = None
for threshold, df in summary_dfs.items():
    if final_summary is None:
        final_summary = df
    else:
        final_summary = pd.merge(final_summary, df, on=['lineage', 'thresh'], how='outer')

# 填充缺失值为0
final_summary = final_summary.fillna(0)

# 确保 thresh 列是数值类型并排序
final_summary['thresh'] = pd.to_numeric(final_summary['thresh'])
final_summary = final_summary.sort_values(['lineage', 'thresh'])

print("处理后的数据框：")
print(final_summary.head(20))

# 保存处理后的数据
# final_summary.to_csv('cell_num_summary_by_threshold.csv', index=False)
print("\n数据已保存到 'cell_num_summary_by_threshold.csv'")


In [ ]:
final_summary.sort_values(['thresh', 'lineage'])


In [ ]:
# 设置绘图风格
plt.style.use('default')
sns.set_palette("husl")

# 获取唯一的 lineage
lineages = final_summary['lineage'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(lineages)))
color_dict = dict(zip(lineages, colors))

# 创建4个子图（每个阈值一个）
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, threshold in enumerate(cell_thresholds):
    ax = axes[idx]
    
    # 获取当前阈值的数据
    col_name = f'total_cell_num_th{threshold}'
    
    for lineage in lineages:
        lineage_data = final_summary[final_summary['lineage'] == lineage]
        if not lineage_data.empty:
            ax.plot(lineage_data['thresh'], lineage_data[col_name], 
                   marker='o', linewidth=2, markersize=6,
                   label=lineage, color=color_dict[lineage])
    
    ax.set_xlabel('Threshold', fontsize=12)
    ax.set_ylabel('Total Cell Number', fontsize=12)
    ax.set_title(f'Cell Number by Lineage (cell_num ≥ {threshold})', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')  # 如果 thresh 跨度大，使用对数坐标

plt.tight_layout()
# plt.savefig('cell_number_lineplots.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 准备数据用于柱状图
plot_data = []
for threshold in cell_thresholds:
    col_name = f'total_cell_num_th{threshold}'
    temp_df = final_summary[['lineage', 'thresh', col_name]].copy()
    temp_df['threshold'] = f'≥{threshold}'
    temp_df = temp_df.rename(columns={col_name: 'total_cell_num'})
    plot_data.append(temp_df)

plot_df = pd.concat(plot_data, ignore_index=True)

# 创建柱状图
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (threshold, ax) in enumerate(zip(cell_thresholds, axes)):
    # 筛选当前阈值的数据
    subset = plot_df[plot_df['threshold'] == f'≥{threshold}']
    
    # 创建透视表
    pivot_data = subset.pivot(index='thresh', columns='lineage', values='total_cell_num')
    
    # 绘制柱状图
    pivot_data.plot(kind='bar', ax=ax, width=0.8)
    
    ax.set_xlabel('Threshold', fontsize=12)
    ax.set_ylabel('Total Cell Number', fontsize=12)
    ax.set_title(f'Cell Number by Lineage (cell_num ≥ {threshold})', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 旋转x轴标签
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
# plt.savefig('cell_number_barplots.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
################ 现在统计下 LLM中的数据   ############ 
import os
import re
import pandas as pd
import numpy as np
import sys
import scanpy as sc
#import scatlasvae  ## 
import scipy.sparse as sp
import pickle

def list_files(directory):
    return [os.path.join(dp, f) for dp, dn, fn in os.walk(os.path.expanduser(directory)) for f in fn]

def list_h5_files(directory):
    return [f for f in list_files(directory) if f.endswith('.h5ad')] 


##################  新的最终一次性结合GSM unique过的meta table, 和 sample unique过的 expr table 来筛选数据(只遍历能筛到数据符合预期的 h5ad path)
### meta 条件: human, 10X Genomics, with tissue(del blood, none, organoid,以及一些有分号的 tissue), 不管有无age
### expr 条件: 根据 Fibro的 10 个marker, Endo的 8 个 marker; aveExpr的 threshold 为 0.2, 是 human的 path 才遍历(对于那些同一个 sample 文件但对应不同的 GSE 的情况我们进行了去重)
meta_table = pd.read_csv('external_inputs/TabulaStromal_reserve_sample_unique_meta_all.tsv',sep='\t')
expr_table = pd.read_csv('external_inputs/TabulaStromal_reserve_sample_unique_exprMat_all.tsv',sep='\t')
expr_table = expr_table.rename(columns= {'GSE':'gse','GSM':'gsm'})
merged_df = pd.merge(meta_table, expr_table, on=['gse', 'gsm'])   ## 有几个差的 GSM, 是meta 那边爬错了，都按 express 这边的来
gsms_diff = list(set(expr_table['gsm'].unique())-set(merged_df['gsm'].unique()))
expr_table_a = expr_table[expr_table['gsm'].isin(gsms_diff)]
gsm_to_gse = dict(zip(expr_table_a['gsm'], expr_table_a['gse']))

# Update GSE values for GSM entries that are in gems_diff list
for gsm_id, new_gse in gsm_to_gse.items():
    print(gsm_id)
    if gsm_id in gsms_diff:
        meta_table.loc[meta_table['gsm'] == gsm_id, 'gse'] = new_gse

########## 然后再 merge 上面的 expr_table 和 meta_table, 这样不会缺样本了 
merged_df = pd.merge(meta_table, expr_table, on=['gse', 'gsm'])


In [ ]:
# merged_df.shape
valid_df = pd.read_csv('external_inputs/TabulaStromal_Fibro_filter4valid_healthy_89Fib_and37Endo_129Sample.tsv',sep='\t')
valid_sample = valid_df['sample'].to_list()
import gc
import copy
adata_list_merge_train = []
adata_list_merge_valid = []
valid_sample_set = set(valid_sample)


In [ ]:
Endo_meta_table = pd.read_csv('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_0.2.tsv',sep='\t')


In [ ]:
Endo_meta_table = Endo_meta_table[['gse', 'gsm', 'sample', 'species', 'lineage',
       'type', 'qc_cn', 'cell_num', 'clus_num']]
Endo_meta_table


In [ ]:
import pickle
pickle_file_valid ='external_inputs/Fibro_human_FibDEG_count_notAge_0.2_valid_healthy.pickle'
with open(pickle_file_valid, 'rb') as f:
    adata_list_merge_valid = pickle.load(f)   #  CustomUnpickler(f).load() 


In [ ]:
Fib_meta_table_valid = pd.DataFrame(columns=['gse','gsm','sample','species','lineage','type','qc_cn','cell_num','clus_num'])
for adata in adata_list_merge_valid:
    print(adata.shape)
    gse = adata.obs['GSE'].unique()[0]
    gsm = adata.obs['GSM'].unique()[0]
    sample =adata.obs['sample'].unique()[0]
    species ='human'
    cell_lineage ='Fibro'
    type='deg'
    adata_raw = sc.read_h5ad(expr_table[expr_table['sample']==sample]['path'].to_list()[0])
    qc_cn = adata_raw.shape[0]
    cell_num= adata.shape[0]
    cluster_num = len(adata.obs['leiden'].unique())
    data = {'gse': gse, 'gsm': gsm, 'sample': sample, 'species': species, 
                           'lineage': cell_lineage, 'type': type, 'qc_cn': qc_cn, 
                           'cell_num': cell_num, 'clus_num': cluster_num}
    df_tmp = pd.DataFrame([data])
    Fib_meta_table_valid = Fib_meta_table_valid._append(df_tmp)
    

In [ ]:

Fib_meta_table_valid['sample'].value_counts()



In [ ]:
5327 - 96


In [ ]:
Fib_meta_table_valid.shape[0] + Fib_meta_table_train.shape[0]


In [ ]:
# adata_list_merge_valid[0:4]
Endo_meta_table_valid = Endo_meta_table[Endo_meta_table['sample'].isin(list(valid_sample_set))]
Endo_meta_table_valid


In [ ]:
Fib_meta_table_valid


In [ ]:
a.shape


In [ ]:
Fib_meta_table_train[Fib_meta_table_train['sample']=='GSM3704358_1']


In [ ]:
len(list(valid_sample_set & set(Fib_meta_table_train['sample'].to_list())))


In [ ]:
Fib_meta_table_train.columns



In [ ]:
meta_table_train = pd.concat([Endo_meta_table_train[['gse', 'gsm', 'sample', 'species', 'lineage', 'type', 'qc_cn',
       'cell_num', 'clus_num']], Fib_meta_table_train],ignore_index=True)

meta_table_train 


In [ ]:
Endo_meta_table_train.to_csv('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_0.2.tsv',sep='\t',index=False)


In [ ]:
Endo_meta_table_train['cell_num'].sum()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 定义细胞数区间
bins = [50, 100, 200, 500, 1000, 5000, 10000, float('inf')]
labels = ['[50-100]', '[100-200]', '[200-500]', '[500-1k]', 
          '[1k-5k]', '[5k-1w]', '>1w']

meta_table_train['cell_range'] = pd.cut(meta_table_train['cell_num'], 
                                         bins=bins, 
                                         labels=labels, 
                                         right=False)

# 按 lineage 和 cell_range 分组统计
grouped = meta_table_train.groupby(['lineage', 'cell_range']).size().unstack(fill_value=0)

# 获取 lineage 列表和区间列表
lineages = grouped.index.tolist()
x = np.arange(len(labels))
width = 0.8 / len(lineages)

# 创建图表
fig, ax = plt.subplots(figsize=(14, 8))

# 颜色列表
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']

# 绘制分组柱状图并添加数值标签
for i, lineage in enumerate(lineages):
    offset = (i - len(lineages)/2 + 0.5) * width
    bars = ax.bar(x + offset, grouped.loc[lineage], width, label=lineage, 
                  color=colors[i % len(colors)], edgecolor='black', linewidth=0.5)
    
    # 在每个柱子上方添加数值标签
    for bar in bars:
        height = bar.get_height()
        if height > 0:  # 只为高度大于0的柱子添加标签
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{int(height)}', ha='center', va='bottom', fontsize=8, rotation=0)

ax.set_xlabel('Cell Number Range', fontsize=12)
ax.set_ylabel('Sample Count', fontsize=12)
ax.set_title('Distribution of Sample Counts Across Cell Number Ranges by Lineage', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend(title='Lineage', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 打印统计结果
print("\n统计结果:")
print(grouped)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 假设已有 meta_df_new 数据框
# 1. 数据处理
cell_thresholds = [50, 100, 150, 200]
summary_dfs = {}

for threshold in cell_thresholds:
    filtered_df = meta_df_new[meta_df_new['cell_num'] >= threshold]
    summary_df = filtered_df.groupby(['lineage', 'thresh'])['cell_num'].sum().reset_index()
    summary_df.columns = ['lineage', 'thresh', f'total_cell_num_th{threshold}']
    summary_dfs[threshold] = summary_df

# 合并所有结果
final_summary = None
for threshold, df in summary_dfs.items():
    if final_summary is None:
        final_summary = df
    else:
        final_summary = pd.merge(final_summary, df, on=['lineage', 'thresh'], how='outer')

final_summary = final_summary.fillna(0)
final_summary['thresh'] = pd.to_numeric(final_summary['thresh'])
final_summary = final_summary.sort_values(['lineage', 'thresh'])

# 2. 可视化设置
plt.style.use('default')
lineages = final_summary['lineage'].unique()

# 创建8种不同的颜色（4个阈值 × 2个lineage？或者为每条线分配唯一颜色）
# 使用tab20调色板获取8种不同颜色
colors = plt.cm.tab20(np.linspace(0, 1, 8))

# 为每条线创建颜色映射
line_colors = {}
color_idx = 0
for lineage in lineages:
    for threshold in cell_thresholds:
        line_colors[(lineage, threshold)] = colors[color_idx % len(colors)]
        color_idx += 1

# 创建图形
plt.figure(figsize=(18, 10))

# 存储每条线的数据点位置，用于智能放置标签
label_positions = []

for i, lineage in enumerate(lineages):
    lineage_data = final_summary[final_summary['lineage'] == lineage]
    
    if not lineage_data.empty:
        for j, threshold in enumerate(cell_thresholds):
            col_name = f'total_cell_num_th{threshold}'
            values = lineage_data[col_name].values
            x_vals = lineage_data['thresh'].values
            
            # 获取这条线的颜色
            line_color = line_colors[(lineage, threshold)]
            
            # 绘制折线
            line = plt.plot(x_vals, values, 
                           marker='o', linewidth=2.5, markersize=8,
                           label=f'{lineage} (≥{threshold})', 
                           color=line_color, alpha=0.9)
            # 为每个数据点添加数字标签，智能放置避免重叠
            for k, (x, y) in enumerate(zip(x_vals, values)):
                if y > 0:
                    # 检查这个位置是否已有标签
                    overlapping = False
                    base_offset_x = 20
                    base_offset_y = 20
                    
                    # 根据点的位置调整标签偏移
                    # 奇数点向左偏移，偶数点向右偏移
                    if k % 2 == 0:
                        offset_x = base_offset_x
                    else:
                        offset_x = -base_offset_x
                    # 根据阈值大小调整垂直偏移
                    if threshold >= 100:
                        offset_y = base_offset_y + 5
                    else:
                        offset_y = base_offset_y - 5
                    # 如果是第一个点，稍微调整位置
                    if k == 0:
                        offset_x = 15
                        offset_y = 10
                    
                    # 如果是最后一个点，调整位置
                    if k == len(x_vals) - 1:
                        offset_x = -15
                    
                    # 添加标签
                    plt.annotate(f'{int(y):,}',
                                xy=(x, y),
                                xytext=(offset_x, offset_y),
                                textcoords='offset points',
                                ha='center' if offset_x > 0 else 'center',
                                va='bottom' if offset_y > 0 else 'top',
                                fontsize=9, fontweight='bold',
                                color=line_color,
                                bbox=dict(boxstyle='round,pad=0.3', 
                                        facecolor='white', alpha=0.8,
                                        edgecolor=line_color, linewidth=1.5))
                    
                    # 存储标签位置用于后续检查
                    label_positions.append((x, y, offset_x, offset_y))

# 设置坐标轴标签和标题
plt.xlabel('Threshold', fontsize=16, fontweight='bold')
plt.ylabel('Total Cell Number', fontsize=16, fontweight='bold')
plt.title('Cell Number by Lineage and Threshold', fontsize=18, fontweight='bold', pad=20)

# 设置图例，放在图外
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10, frameon=True, 
          fancybox=True, shadow=True, ncol=1)

# 设置网格
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# 设置x轴为对数刻度
plt.xscale('log')

# 设置y轴范围，给标签留出空间
y_max = final_summary[[f'total_cell_num_th{t}' for t in cell_thresholds]].max().max()
plt.ylim(0, y_max * 1.2)

# 设置x轴刻度
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# 添加背景色区分不同的lineage（可选）
# 为每个lineage添加背景色块
for i, lineage in enumerate(lineages):
    lineage_data = final_summary[final_summary['lineage'] == lineage]
    if not lineage_data.empty:
        x_min = lineage_data['thresh'].min()
        x_max = lineage_data['thresh'].max()
        # 添加半透明背景
        plt.axvspan(x_min, x_max, alpha=0.05, color=colors[i*4] if i*4 < len(colors) else 'gray')

# 调整布局
plt.tight_layout()

# 保存图片
# plt.savefig('cell_number_with_smart_labels.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
from functools import reduce

# 1. 按 lineage 分组分析
lineage_list = EN_Gene_bench['lineage'].unique()
results = {}

for lineage in lineage_list:
    # 筛选当前lineage的数据
    lineage_data = EN_Gene_bench[EN_Gene_bench['lineage'] == lineage]
    
    # 获取当前lineage中所有nFea值
    nfea_values = lineage_data['nFea'].unique()
    
    for nfea in nfea_values:
        # 筛选当前lineage和nFea的数据
        subset = lineage_data[lineage_data['nFea'] == nfea]
        
        # 计算geneN_not0列中所有行共有的基因
        if len(subset) > 0:
            not0_genes = [set(eval(genes)) for genes in subset['geneN_not0']]
            common_not0 = reduce(set.intersection, not0_genes)
            
            # 计算geneN_i列中所有行共有的基因
            i_genes = [set(eval(genes)) for genes in subset['geneN_i']]
            common_i = reduce(set.intersection, i_genes)
            
            # 存储结果
            key = f"{lineage}_nFea_{nfea}"
            results[key] = {
                'geneN_not0_common': sorted(list(common_not0)),
                'geneN_i_common': sorted(list(common_i)),
                'n_rows': len(subset)
            }

# 打印按lineage和nFea分组的结果
print("="*80)
print("按 Lineage 和 nFea 分组分析的共同基因：")
print("="*80)
for key, value in results.items():
    print(f"\n{key}:")
    print(f"  行数: {value['n_rows']}")
    print(f"  geneN_not0 共同基因 ({len(value['geneN_not0_common'])}个):")
    print(f"    {value['geneN_not0_common']}")
    print(f"  geneN_i 共同基因 ({len(value['geneN_i_common'])}个):")
    print(f"    {value['geneN_i_common']}")

# 2. 不区分lineage，所有数据一起分析
print("\n" + "="*80)
print("不区分 Lineage 的整体分析：")
print("="*80)

# 计算geneN_not0列中所有行共有的基因
all_not0_genes = [set(eval(genes)) for genes in EN_Gene_bench['geneN_not0']]
all_common_not0 = reduce(set.intersection, all_not0_genes)

# 计算geneN_i列中所有行共有的基因
all_i_genes = [set(eval(genes)) for genes in EN_Gene_bench['geneN_i']]
all_common_i = reduce(set.intersection, all_i_genes)

print(f"\n总行数: {len(EN_Gene_bench)}")
print(f"geneN_not0 共同基因 ({len(all_common_not0)}个):")
print(sorted(list(all_common_not0)))
print(f"\ngeneN_i 共同基因 ({len(all_common_i)}个):")
print(sorted(list(all_common_i)))

# 3. 可选：保存结果到文件
import json

# 创建完整的分析结果字典
full_results = {
    'by_lineage_nfea': results,
    'overall': {
        'geneN_not0_common': sorted(list(all_common_not0)),
        'geneN_i_common': sorted(list(all_common_i)),
        'total_rows': len(EN_Gene_bench)
    }
}

# 保存为JSON文件
with open('gene_common_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)

print("\n" + "="*80)
print("结果已保存到 'gene_common_analysis.json'")
print("="*80)

# 4. 如果需要更详细的统计信息，可以添加如下代码
print("\n" + "="*80)
print("统计信息摘要：")
print("="*80)

# 统计每个lineage的行数
print("\n各Lineage行数统计：")
for lineage in lineage_list:
    count = len(EN_Gene_bench[EN_Gene_bench['Lineage'] == lineage])
    print(f"  {lineage}: {count}行")

# 统计nFea的分布
print("\nnFea值分布：")
for lineage in lineage_list:
    lineage_data = EN_Gene_bench[EN_Gene_bench['Lineage'] == lineage]
    nfea_counts = lineage_data['nFea'].value_counts().sort_index()
    print(f"\n{lineage}:")
    for nfea, count in nfea_counts.items():
        print(f"  nFea={nfea}: {count}行")
        

In [ ]:
#### renew the specific information of the 



In [ ]:
import pandas as pd
from functools import reduce

# 1. 按 lineage 分组分析
lineage_list = EN_Gene_bench['lineage'].unique()
results = {}

for lineage in lineage_list:
    # 筛选当前lineage的数据
    lineage_data = EN_Gene_bench[EN_Gene_bench['lineage'] == lineage]
    
    if len(lineage_data) > 0:
        # 计算geneN_not0列中所有行共有的基因
        not0_genes = [set(eval(genes)) for genes in lineage_data['geneN_not0']]
        common_not0 = reduce(set.intersection, not0_genes)
        
        # 计算geneN_i列中所有行共有的基因
        i_genes = [set(eval(genes)) for genes in lineage_data['geneN_i']]
        common_i = reduce(set.intersection, i_genes)
        
        # 存储结果
        results[lineage] = {
            'geneN_not0_common': sorted(list(common_not0)),
            'geneN_i_common': sorted(list(common_i)),
            'n_rows': len(lineage_data)
        }

# 打印按lineage分组的结果
print("="*80)
print("按 Lineage 分组分析的共同基因：")
print("="*80)
for lineage, value in results.items():
    print(f"\n{lineage}:")
    print(f"  行数: {value['n_rows']}")
    print(f"  geneN_not0 共同基因 ({len(value['geneN_not0_common'])}个):")
    if value['geneN_not0_common']:
        print(f"    {value['geneN_not0_common']}")
    else:
        print(f"    没有共同基因")
    
    print(f"  geneN_i 共同基因 ({len(value['geneN_i_common'])}个):")
    if value['geneN_i_common']:
        print(f"    {value['geneN_i_common']}")
    else:
        print(f"    没有共同基因")

# 2. 不区分lineage，所有数据一起分析
print("\n" + "="*80)
print("不区分 Lineage 的整体分析：")
print("="*80)

# 计算geneN_not0列中所有行共有的基因
all_not0_genes = [set(eval(genes)) for genes in EN_Gene_bench['geneN_not0']]
all_common_not0 = reduce(set.intersection, all_not0_genes)

# 计算geneN_i列中所有行共有的基因
all_i_genes = [set(eval(genes)) for genes in EN_Gene_bench['geneN_i']]
all_common_i = reduce(set.intersection, all_i_genes)

print(f"\n总行数: {len(EN_Gene_bench)}")
print(f"geneN_not0 共同基因 ({len(all_common_not0)}个):")
if all_common_not0:
    print(sorted(list(all_common_not0)))
else:
    print("没有共同基因")

print(f"\ngeneN_i 共同基因 ({len(all_common_i)}个):")
if all_common_i:
    print(sorted(list(all_common_i)))
else:
    print("没有共同基因")

# 3. 保存结果到文件
import json

# 创建完整的分析结果字典
full_results = {
    'by_lineage': results,
    'overall': {
        'geneN_not0_common': sorted(list(all_common_not0)) if all_common_not0 else [],
        'geneN_i_common': sorted(list(all_common_i)) if all_common_i else [],
        'total_rows': len(EN_Gene_bench)
    }
}

# 保存为JSON文件
with open('gene_common_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)

print("\n" + "="*80)
print("结果已保存到 'gene_common_analysis.json'")
print("="*80)
# 4. 统计信息
print("\n" + "="*80)
print("统计信息摘要：")
print("="*80)
# 统计每个lineage的行数
print("\n各Lineage行数统计：")
for lineage in lineage_list:
    count = len(EN_Gene_bench[EN_Gene_bench['Lineage'] == lineage])
    print(f"  {lineage}: {count}行")

# 5. 可选：找出每个lineage中独有的基因（在某个lineage中所有行都有，但在其他lineage中不一定有）
print("\n" + "="*80)
print("各Lineage独有的共同基因（相对于其他lineage）：")
print("="*80)
# 收集每个lineage的共同基因
lineage_common_not0 = {}
lineage_common_i = {}
for lineage in lineage_list:
    if lineage in results:
        lineage_common_not0[lineage] = set(results[lineage]['geneN_not0_common'])
        lineage_common_i[lineage] = set(results[lineage]['geneN_i_common'])

# 找出每个lineage独有的基因
for lineage in lineage_list:
    if lineage in lineage_common_not0:
        # 获取其他所有lineage的共同基因集合
        other_lineages = [l for l in lineage_list if l != lineage and l in lineage_common_not0]
        
        if other_lineages:
            # 其他lineage的共同基因的并集
            others_not0 = set().union(*[lineage_common_not0[l] for l in other_lineages])
            others_i = set().union(*[lineage_common_i[l] for l in other_lineages])
            
            # 当前lineage独有的基因
            unique_not0 = lineage_common_not0[lineage] - others_not0
            unique_i = lineage_common_i[lineage] - others_i
            
            print(f"\n{lineage}:")
            print(f"  geneN_not0 独有的共同基因 ({len(unique_not0)}个):")
            if unique_not0:
                print(f"    {sorted(list(unique_not0))}")
            else:
                print(f"    没有独有的共同基因")
            
            print(f"  geneN_i 独有的共同基因 ({len(unique_i)}个):")
            if unique_i:
                print(f"    {sorted(list(unique_i))}")
            else:
                print(f"    没有独有的共同基因")
                

In [ ]:
### 我今天要把这个 GWAS的东西弄完